# Clasificación y Corrección de Anomalías

## Crear un ambiente virtual y actulizar pip

python3 -m venv .venv

source .venv/bin/activate

pip install -U pip

### Instalación de Librerías Necesarias

pip install pandas numpy matplotlib seaborn scikit-learn tensorflow

### Lo siguiente va sólo para usar la GPU de mi MAC

pip install torch torchvision torchaudio

## Definición de los tipos de anomalías para clasificación

1. Datos Faltantes:

- Descripción: Se refiere a la ausencia de un valor de un sensor donde se esperaba uno. Esto indica que no se recibió o no se registró ninguna lectura del sensor en un momento dado.
- Identificación: Se detecta directamente cuando el valor del sensor es NaN (Not a Number), nulo, o está ausente en el dataset. No requiere el modelo ML de clasificación para ser identificado como "dato faltante", sino que se maneja como un caso previo.
- Naturaleza: Generalmente es una anomalía "real" del sistema de adquisición o transmisión de datos.

2. Contextual:

- Descripción: Un valor de sensor que, aunque podría parecer inusual o un outlier si se observa de forma aislada para ese sensor, es realmente plausible y explicable cuando se considera el "contexto" completo. Este contexto incluye las lecturas de otros sensores, el estado de los actuadores (ventilación, calefacción, riego, etc.) y/o el momento específico en el tiempo (p.ej., ciclos diurnos/nocturnos, estacionales, o después de una intervención conocida). Es decir, una anomalía contextual no es necesariamente un error, pero podría parecerlo si no se toma en cuenta el contexto en el que ocurre.
- Ejemplo: Un aumento brusco de la temperatura interna que coincide con el encendido del sistema de calefacción. Una humedad muy baja que coincide con una fuerte ventilación y baja humedad externa.
- Naturaleza: El valor del sensor en sí es probablemente correcto, pero representa una condición operativa del invernadero que es infrecuente o se encuentra en los extremos del rango normal de operación bajo ciertas circunstancias.

3. Sensor Atascado:

- Descripción: El sensor reporta el mismo valor (o valores dentro de un rango muy estrecho e invariable) durante un período de tiempo prolongado, a pesar que las condiciones ambientales o las lecturas de otros sensores relacionados indican que el valor medido debería estar cambiando.
- Ejemplo: Un sensor de temperatura que muestra 25.0°C durante varias horas seguidas, mientras la radiación solar varía significativamente o la temperatura exterior cambia.
- Naturaleza: Es una anomalía "real", indicativa de un mal funcionamiento del sensor.

4. Ruido:

- Descripción: El sensor reporta lecturas que muestran fluctuaciones erráticas, picos o caídas repentinas y de corta duración que no se corresponden con cambios reales en la variable medida ni con el comportamiento de otros sensores. Estos valores suelen desviarse significativamente de la tendencia local de la señal.
- Ejemplo: Una serie de lecturas de humedad: 60%, 61%, 95%, 62%, 60%, donde el valor de 95% es un pico aislado e inverosímil que no se sostiene y no tiene explicación contextual.
- Naturaleza: Es una anomalía "real", a menudo debida a interferencias electromagnéticas, problemas de conexión o fallos intermitentes del sensor.

5. Valores Fuera de Rango:

- Descripción: El sensor reporta un valor que cae fuera de los límites físicos, lógicos o esperados para la variable que está midiendo. Estos límites pueden estar definidos por las especificaciones del sensor, el conocimiento del dominio (p.ej., la temperatura en un invernadero no puede ser de -100°C) o rangos operativos históricamente observados.
- Ejemplo: Un sensor de pH del suelo que reporta un valor de 17 (cuando el rango normal es 0-14). Un sensor de CO2 que reporta un valor negativo.
- Naturaleza: Es una anomalía "real", indicando un error grave del sensor o un problema en la transmisión/procesamiento de datos.

6. Desviación Inexplicable Respecto a Pares:

- Descripción: Esta anomalía ocurre cuando la relación esperada entre un sensor y otros sensores diferentes pero altamente correlacionados se rompe de manera inexplicable. Aunque no se disponga de múltiples sensores midiendo exactamente la misma variable (pares directos), se pueden identificar pares de sensores que normalmente exhiben un comportamiento fuertemente interdependiente (p.ej., radiación exterior y radiación interior; temperatura del aire y temperatura del suelo). Una desviación significativa de esta relación característica, que no puede explicarse por otros factores contextuales, sugiere un problema en al menos uno de los sensores involucrados.
- Ejemplo: Si la radiación exterior aumenta considerablemente, pero la radiación global interior permanece inesperadamente baja (o viceversa) sin una razón contextual válida, esto indicaría una posible anomalía en el sensor de radiación interior o exterior. De manera similar, si la temperatura del aire interior cambia drásticamente pero la temperatura del suelo no muestra una respuesta coherente (considerando su inercia) a lo largo del tiempo.
- Naturaleza: Es una anomalía "real", que apunta a un posible fallo, descalibración o problema en uno de los sensores que forman parte de esa relación correlacionada, o a un evento no medido que rompe dicha correlación. El desafío aquí es determinar cuál de los sensores es el problemático o si la relación modelada es la que falla.

## Preparación de datos para la detección de anomalías

### Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import joblib
import pickle
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pandas import Timedelta
from collections import defaultdict


# Configuraciones adicionales para mejorar la visualización (opcional)
plt.style.use('seaborn-v0_8-whitegrid') # Estilo de gráficos
pd.set_option('display.max_columns', None) # Para mostrar todas las columnas de un DataFrame
pd.set_option('display.float_format', lambda x: '%.3f' % x) # Formato para números flotantes

### Cargar el Dataset

In [ ]:
# Cargar el dataset
file_path = 'Homologados/AgroConnect_OPC_20250301_20250430.csv'
df_original = pd.read_csv(file_path)
print("Dataset cargado exitosamente.")

# Mostrar las primeras 5 filas para una inspección inicial
print("\nPrimeras 5 filas del dataset:")
print(df_original.head())

# Mostrar información general del DataFrame (tipos de datos, valores no nulos)
print("\nInformación general del DataFrame:")
df_original.info()

# Mostrar estadísticas descriptivas para las columnas numéricas
print("\nEstadísticas descriptivas:")
print(df_original.describe())

### Verificar Datos Faltantes de Forma Explícita

In [ ]:
if 'df_original' in locals():
    print("\nConteo de datos faltantes (NaN) por columna:")
    missing_values = df_original.isnull().sum()
    print(missing_values[missing_values > 0]) # Muestra solo columnas con datos faltantes
    if missing_values.sum() == 0:
        print("No se encontraron datos faltantes (NaN) explícitos en el dataset.")

### Pasos Preliminares Adicionales

In [ ]:
# Convertir la columna de fecha a datetime
if df_original['Fecha'].dtype == 'object':
    df_original['Fecha'] = pd.to_datetime(df_original['Fecha'], format='%d/%m/%Y %H:%M:%S')

# Extraer características temporales que podrían ser útiles
df_original['Hora'] = df_original['Fecha'].dt.hour
df_original['DiaSemana'] = df_original['Fecha'].dt.dayofweek
df_original['Mes'] = df_original['Fecha'].dt.month

In [ ]:
if 'df_original' in locals():
    print("\nNombres de columnas originales:")
    print(df_original.columns.tolist())

In [ ]:
if 'df_original' in locals():
    num_duplicates = df_original.duplicated().sum()
    if num_duplicates > 0:
        print(f"\nSe encontraron {num_duplicates} filas duplicadas completas.")
    else:
        print("\nNo se encontraron filas duplicadas completas en el dataset.")

In [ ]:
# Crear una copia del DataFrame original para trabajar
df_trabajo = df_original.copy()
print("Copia del DataFrame creada como 'df_trabajo'.")
# Inicializar la columna para la etiqueta de detección de anomalías
# Inicialmente, todas las filas se consideran "normales"
df_trabajo['etiqueta_deteccion'] = 'normal'
print("Columna 'etiqueta_deteccion' inicializada a 'normal'.")
# Inicializar la columna para el tipo específico de anomalía
# Inicialmente, no hay tipo de anomalía asignado (NaN o None)
df_trabajo['etiqueta_tipo_anomalia'] = pd.NA # Usar pd.NA para datos faltantes de pandas

print("Columna 'etiqueta_tipo_anomalia' inicializada a NA.")
# Verificar las nuevas columnas
print("\nPrimeras filas de 'df_trabajo' con las nuevas columnas de etiquetas:")
print(df_trabajo.head())
print("\nConteo de valores en 'etiqueta_deteccion':")
print(df_trabajo['etiqueta_deteccion'].value_counts())
print("\nConteo de valores en 'etiqueta_tipo_anomalia' (deberían ser todos NA/NaN por ahora):")
print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)) # dropna=False para contar los NA


### Inyección de Anomalías

1. Datos Faltantes

In [ ]:
columnas_sensores_y_actuadores = [
    'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
    'XCO2I', 'XHINV', 'XTINV', 'XTS',
    'UVENT_cen', 'UVENT_lN'
]

# Filtrar la lista para asegurar que solo contenga columnas existentes en el DataFrame df_trabajo
columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]

print(f"Se utilizarán las siguientes columnas para potencialmente inyectar NaNs: \n{columnas_objetivo_existentes}")

In [ ]:
def inyectar_datos_faltantes(df, porcentaje_filas_afectadas, columnas_objetivo, num_valores_nan_por_fila=1):
    """
    Inyecta NaNs en un porcentaje de filas y en un número específico de columnas objetivo.

    Args:
        df (pd.DataFrame): DataFrame de trabajo (debe tener 'etiqueta_deteccion' y 'etiqueta_tipo_anomalia').
        porcentaje_filas_afectadas (float): Porcentaje de filas 'normales' a afectar (ej: 0.01 para 1%).
        columnas_objetivo (list): Lista de nombres de columnas donde se pueden inyectar NaNs.
        num_valores_nan_por_fila (int): Cuántas columnas aleatorias por fila afectada se convertirán a NaN.

    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas y etiquetas actualizadas.
    """
    if not columnas_objetivo:
        print("No se proporcionaron columnas objetivo. No se inyectarán datos faltantes.")
        return df

    df_modificado = df.copy() # Trabajar sobre una copia local a la función
    
    indices_normales = df_modificado[df_modificado['etiqueta_deteccion'] == 'normal'].index
    num_filas_a_afectar = int(len(indices_normales) * porcentaje_filas_afectadas)

    if num_filas_a_afectar == 0 and len(indices_normales) > 0: # Evitar mensaje si no hay filas normales
        print(f"Con el {porcentaje_filas_afectadas*100:.2f}% de {len(indices_normales)} filas normales, no se seleccionaron filas para afectar (resultado: 0).")
        print("Prueba un porcentaje mayor o asegúrate de tener filas 'normales' disponibles.")
        return df_modificado
    elif len(indices_normales) == 0:
        print("No hay filas 'normales' disponibles para inyectar anomalías.")
        return df_modificado


    filas_a_afectar_indices = np.random.choice(indices_normales, size=num_filas_a_afectar, replace=False)

    print(f"Inyectando datos faltantes en {len(filas_a_afectar_indices)} filas...")

    nan_count_por_columna = {col: 0 for col in columnas_objetivo} # Para seguimiento

    for idx in filas_a_afectar_indices:
        if len(columnas_objetivo) < num_valores_nan_por_fila:
            columnas_a_modificar_para_esta_fila = columnas_objetivo
        else:
            columnas_a_modificar_para_esta_fila = np.random.choice(columnas_objetivo, size=num_valores_nan_por_fila, replace=False)
        
        for col_name in columnas_a_modificar_para_esta_fila:
            df_modificado.loc[idx, col_name] = np.nan
            nan_count_por_columna[col_name] +=1
        
        df_modificado.loc[idx, 'etiqueta_deteccion'] = 'anomalia'
        df_modificado.loc[idx, 'etiqueta_tipo_anomalia'] = 'Datos Faltantes' # Etiqueta específica
        
    print(f"Inyección de datos faltantes completada. {len(filas_a_afectar_indices)} filas fueron modificadas.")
            
    return df_modificado

In [ ]:
if 'df_trabajo' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    porcentaje_filas_nan = 0.02  # Inyectar NaNs en el 2% de las filas 'normales'
    num_sensores_por_fila_afectada = 1 # En cada fila afectada, poner NaN en 1 valor aleatorio (sensor o actuador)

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    etiquetas_deteccion_antes = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    df_trabajo = inyectar_datos_faltantes(df_trabajo, 
                                            porcentaje_filas_nan, 
                                            columnas_objetivo_existentes, 
                                            num_sensores_por_fila_afectada)

    print("\n--- Resultados de la Inyección de 'Datos Faltantes' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))

    # Verificar cuántos NaNs se crearon en total en las columnas objetivo
    total_nans_inyectados_verificacion = 0
    for col in columnas_objetivo_existentes:
        total_nans_inyectados_verificacion += df_trabajo[col].isnull().sum()

else:
    print("Error: Asegúrate de que 'df_trabajo' esté definido y 'columnas_objetivo_existentes' esté correctamente populada.")

2. Sensor Atascado

Para simular correctamente un sensor atascado durante un período, necesitamos que nuestro DataFrame esté ordenado cronológicamente. Por lo que vamos a usar la columna Fecha, lo ordenaremos y luego resetearemos el índice para tener una secuencia numérica simple que corresponda al orden temporal. Esto facilita la selección de filas consecutivas.

In [ ]:
if 'df_trabajo' in locals():
    # Asegurar que el DataFrame esté ordenado por fecha
    if 'Fecha' in df_trabajo.columns:
        print("Asegurando que el DataFrame esté ordenado por 'Fecha'...")
        df_trabajo = df_trabajo.sort_values(by='Fecha').reset_index(drop=True)
        print("DataFrame ordenado por 'Fecha' y el índice ha sido reseteado.")
    else:
        print("Advertencia: La columna 'Fecha' no se encontró. No se puede asegurar el orden temporal para 'Sensor Atascado'.")

else:
    print("Error: 'df_trabajo' no está definido.")

In [ ]:
# Reutilizamos la lista definida anteriormente
if 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print(f"Se considerarán las siguientes columnas para simular 'Sensor Atascado': \n{columnas_objetivo_existentes}")
else:
    # Esta definición es por si se ejecuta esta celda de forma aislada y la anterior no.
    columnas_sensores_y_actuadores = [ 
        'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
        'XCO2I', 'XHINV', 'XTINV', 'XTS',
        'UVENT_cen', 'UVENT_lN'
    ]
    columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]
    if not columnas_objetivo_existentes:
        print("ADVERTENCIA: 'columnas_objetivo_existentes' está vacía o no definida. Define las columnas sensor/actuador.")
    else:
        print(f"Se utilizarán las siguientes columnas para simular 'Sensor Atascado': \n{columnas_objetivo_existentes}")

In [ ]:
def inyectar_sensor_atascado(df, num_secuencias_stuck, duracion_min_stuck, duracion_max_stuck, columnas_objetivo):
    """
    Inyecta anomalías de "Sensor Atascado" en el DataFrame.

    Args:
        df (pd.DataFrame): DataFrame de trabajo.
        num_secuencias_stuck (int): Número de secuencias de "sensor atascado" a crear.
        duracion_min_stuck (int): Duración mínima (en número de filas/timesteps) del atasco.
        duracion_max_stuck (int): Duración máxima del atasco.
        columnas_objetivo (list): Lista de nombres de columnas donde se puede simular el atasco.

    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas.
    """
    if not columnas_objetivo:
        print("No se proporcionaron columnas objetivo. No se inyectarán anomalías de sensor atascado.")
        return df
    if duracion_min_stuck > duracion_max_stuck:
        print("Error: duracion_min_stuck no puede ser mayor que duracion_max_stuck.")
        return df

    df_modificado = df.copy()
    
    # Intentar seleccionar puntos de inicio solo de filas actualmente 'normales' para evitar la sobreescritura compleja de diferentes tipos de anomalías en la misma secuencia.
    indices_normales_disponibles = list(df_modificado[df_modificado['etiqueta_deteccion'] == 'normal'].index)
    
    if not indices_normales_disponibles:
        print("No hay filas 'normales' disponibles para iniciar una secuencia de sensor atascado.")
        return df_modificado

    secuencias_creadas = 0
    max_intentos_por_secuencia = 10 # Para evitar bucles infinitos si no se encuentran segmentos válidos
    
    print(f"Intentando crear {num_secuencias_stuck} secuencias de 'Sensor Atascado'...")

    for _ in range(num_secuencias_stuck):
        intento_actual = 0
        secuencia_valida_encontrada = False
        while intento_actual < max_intentos_por_secuencia and not secuencia_valida_encontrada:
            if not indices_normales_disponibles: # No quedan más puntos de partida normales
                break 

            # Seleccionar aleatoriamente una columna sensor/actuador
            columna_a_atascar = np.random.choice(columnas_objetivo)
            
            # Seleccionar aleatoriamente una duración para el atasco
            duracion_stuck = np.random.randint(duracion_min_stuck, duracion_max_stuck + 1)
            
            # Seleccionar aleatoriamente un índice de inicio para el atasco
            # Asegurarse de que haya suficientes filas 'normales' consecutivas para la duración
            
            # Filtrar posibles inicios para asegurar que toda la secuencia sea inicialmente normal
            posibles_inicios = []
            for start_candidate_idx in indices_normales_disponibles:
                # Verificar que el segmento completo esté dentro de los límites y sea 'normal' y que no sea demasiado largo como para salir del dataframe
                if start_candidate_idx + duracion_stuck <= len(df_modificado):
                    es_segmento_normal = True
                    for i in range(duracion_stuck):
                        # Como el índice fue reseteado, podemos usar la posición directamente.
                        # Pero es más seguro referenciar por el valor del índice si este no fuera 0..N-1
                        # Aquí asumimos índice 0..N-1 por el reset_index previo.
                        current_row_index = start_candidate_idx + i 
                        if df_modificado.loc[current_row_index, 'etiqueta_deteccion'] != 'normal':
                            es_segmento_normal = False
                            break
                    if es_segmento_normal:
                        posibles_inicios.append(start_candidate_idx)
            
            if not posibles_inicios:
                intento_actual += 1
                continue # No se encontraron inicios válidos para esta duración/columna, reintentar

            start_idx = np.random.choice(posibles_inicios)
            secuencia_valida_encontrada = True

            # Obtener el valor al que se atascará el sensor (valor al inicio del atasco)
            stuck_value = df_modificado.loc[start_idx, columna_a_atascar]

            # Si el valor ya es NaN, no podemos atascarlo a NaN, elegimos el valor anterior no NaN o saltamos.
            if pd.isna(stuck_value):
                # Buscar hacia atrás un valor no NaN en la misma columna
                for k in range(start_idx - 1, -1, -1):
                    if not pd.isna(df_modificado.loc[k, columna_a_atascar]):
                        stuck_value = df_modificado.loc[k, columna_a_atascar]
                        break
                if pd.isna(stuck_value): # Aún NaN, no se pudo encontrar valor previo
                    print(f"  Advertencia: Valor en {columna_a_atascar} en el índice {start_idx} es NaN y no se encontró valor previo. Saltando esta secuencia.")
                    secuencia_valida_encontrada = False # Marcar para reintentar si es posible
                    intento_actual +=1 # contar como intento
                    continue


            # Aplicar el atasco y las etiquetas
            print(f"  Creando atasco en '{columna_a_atascar}' desde índice {start_idx} por {duracion_stuck} periodos. Valor: {stuck_value:.2f}")
            for i in range(duracion_stuck):
                current_row_index = start_idx + i
                df_modificado.loc[current_row_index, columna_a_atascar] = stuck_value
                df_modificado.loc[current_row_index, 'etiqueta_deteccion'] = 'anomalia'
                df_modificado.loc[current_row_index, 'etiqueta_tipo_anomalia'] = 'Sensor Atascado'
            
            secuencias_creadas += 1
            
            # Actualizar la lista de índices normales disponibles eliminando los que se usaron, esto es importante para que los 'posibles_inicios' se recalculen correctamente
            temp_indices_normales = []
            for idx_normal in indices_normales_disponibles:
                # Si el índice normal no está dentro del rango que acabamos de usar
                if not (start_idx <= idx_normal < start_idx + duracion_stuck):
                    temp_indices_normales.append(idx_normal)
            indices_normales_disponibles = temp_indices_normales
            
        if not secuencia_valida_encontrada and intento_actual == max_intentos_por_secuencia :
            print(f"No se pudo crear una secuencia de 'Sensor Atascado' después de {max_intentos_por_secuencia} intentos (posiblemente no hay suficientes segmentos 'normales' consecutivos).")

    print(f"Inyección de 'Sensor Atascado' completada. Se crearon {secuencias_creadas} secuencias.")
    return df_modificado

In [ ]:
if 'df_trabajo' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    # Parámetros para la inyección de "Sensor Atascado"
    num_secuencias = 50  # Número de secuencias de sensor atascado a crear
    duracion_min = 5     # Mínimo 5 periodos de 5 minutos (25 minutos)
    duracion_max = 20    # Máximo 20 periodos de 5 minutos (1h 40m)

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    etiquetas_deteccion_antes_stuck = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_stuck = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)
    
    print(f"\nIniciando inyección de {num_secuencias} secuencias de 'Sensor Atascado'...")
    df_trabajo = inyectar_sensor_atascado(df_trabajo,
                                        num_secuencias,
                                        duracion_min,
                                        duracion_max,
                                        columnas_objetivo_existentes)

    print("\n--- Resultados de la Inyección de 'Sensor Atascado' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_stuck)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_stuck)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    # Verificar una columna para ver si tiene valores repetidos (indicativo de atasco)
    if columnas_objetivo_existentes:
        col_ejemplo = columnas_objetivo_existentes[0] 
        # Contar secuencias de valores repetidos podría ser complejo aquí, pero podemos verificar el número de filas etiquetadas.
        num_sensores_atascados_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Sensor Atascado').sum()
        print(f"\nNúmero total de filas etiquetadas como 'Sensor Atascado': {num_sensores_atascados_etiquetados}")
else:
    print("Error: Asegúrate de que 'df_trabajo' esté definido y 'columnas_objetivo_existentes' esté correctamente populada y que el DataFrame esté ordenado por fecha.")

3. Ruido / Picos Aleatorios

In [ ]:
# Reutilizamos la lista definida anteriormente
if 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print(f"Se considerarán las siguientes columnas para simular 'Ruido / Picos': \n{columnas_objetivo_existentes}")
else:
    # Esta definición es por si se ejecuta esta celda de forma aislada.
    columnas_sensores_y_actuadores = [ 
        'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
        'XCO2I', 'XHINV', 'XTINV', 'XTS',
        'UVENT_cen', 'UVENT_lN'
    ]
    columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]
    if not columnas_objetivo_existentes:
        print("ADVERTENCIA: 'columnas_objetivo_existentes' está vacía o no definida.")
    else:
        print(f"Se utilizarán las siguientes columnas para simular 'Ruido / Picos': \n{columnas_objetivo_existentes}")

# Calcular la desviación estándar de las columnas objetivo a partir del DataFrame ORIGINAL para tener una medida de variabilidad 'limpia'.
std_devs_originales = {}
if 'df_original' in locals() and columnas_objetivo_existentes:
    print("\nCalculando desviaciones estándar de las columnas objetivo (desde df_original):")
    for col in columnas_objetivo_existentes:
        # Asegurarse de que la columna sea numérica antes de calcular std
        if pd.api.types.is_numeric_dtype(df_original[col]):
            std_devs_originales[col] = df_original[col].std()
            # print(f"  Std de {col}: {std_devs_originales[col]:.3f}")
        else:
            print(f"  Advertencia: Columna '{col}' no es numérica en df_original. No se calculará std.")
            std_devs_originales[col] = 0 # O manejar de otra forma, ej: no incluirla para ruido
    if not std_devs_originales:
        print("Advertencia: No se pudieron calcular desviaciones estándar. Verifica 'df_original'.")
elif not columnas_objetivo_existentes:
    print("Advertencia: 'columnas_objetivo_existentes' no está definida. No se pueden calcular std.")
else:
    print("Advertencia: 'df_original' no está definido. No se pueden calcular std de referencia.")

In [ ]:
def inyectar_ruido_picos(df, porcentaje_filas_ruido, columnas_objetivo, std_devs_ref, 
                            factor_ruido_std_min=3, factor_ruido_std_max=5, num_sensores_ruido_por_fila=1):
    """
    Inyecta ruido (picos/caídas) en un porcentaje de filas y columnas de sensores.
    La magnitud del ruido se basa en la desviación estándar de la columna.

    Args:
        df (pd.DataFrame): DataFrame de trabajo.
        porcentaje_filas_ruido (float): Porcentaje de filas 'normales' a afectar.
        columnas_objetivo (list): Lista de nombres de columnas donde inyectar ruido.
        std_devs_ref (dict): Diccionario con {nombre_columna: std_dev} de referencia.
        factor_ruido_std_min (float): Factor mínimo por el cual multiplicar la std_dev para el ruido.
        factor_ruido_std_max (float): Factor máximo por el cual multiplicar la std_dev para el ruido.
        num_sensores_ruido_por_fila (int): Cuántos sensores aleatorios por fila afectada tendrán ruido.

    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas.
    """
    if not columnas_objetivo or not std_devs_ref:
        print("No se proporcionaron columnas objetivo o std_devs_ref. No se inyectará ruido.")
        return df
    if factor_ruido_std_min > factor_ruido_std_max:
        print("Error: factor_ruido_std_min no puede ser mayor que factor_ruido_std_max.")
        return df

    df_modificado = df.copy()
    
    indices_normales = df_modificado[df_modificado['etiqueta_deteccion'] == 'normal'].index
    num_filas_a_afectar = int(len(indices_normales) * porcentaje_filas_ruido)

    if num_filas_a_afectar == 0 and len(indices_normales) > 0:
        print(f"Con el {porcentaje_filas_ruido*100:.2f}% de {len(indices_normales)} filas normales, no se seleccionaron filas para afectar (resultado: 0).")
        return df_modificado
    elif len(indices_normales) == 0:
        print("No hay filas 'normales' disponibles para inyectar ruido.")
        return df_modificado

    filas_a_afectar_indices = np.random.choice(indices_normales, size=num_filas_a_afectar, replace=False)
    
    print(f"Inyectando ruido/picos en {len(filas_a_afectar_indices)} filas...")
    
    ruido_inyectado_count = 0
    for idx in filas_a_afectar_indices:
        # Seleccionar aleatoriamente 'num_sensores_ruido_por_fila' columnas
        columnas_para_ruido_actual = np.random.choice(
            [col for col in columnas_objetivo if std_devs_ref.get(col, 0) > 0], # Solo cols con std > 0
            size=min(num_sensores_ruido_por_fila, len([col for col in columnas_objetivo if std_devs_ref.get(col,0) > 0])), # Evitar error si hay pocas cols válidas
            replace=False
        )
        if not any(columnas_para_ruido_actual): # Si no se seleccionó ninguna columna válida
            continue

        for col_name in columnas_para_ruido_actual:
            current_value = df_modificado.loc[idx, col_name]
            std_dev = std_devs_ref.get(col_name)

            if pd.isna(current_value) or std_dev is None or std_dev == 0:
                # No aplicar ruido si el valor actual es NaN o no hay std_dev válida
                continue 

            # Determinar magnitud y dirección del ruido
            factor_ruido_actual = np.random.uniform(factor_ruido_std_min, factor_ruido_std_max)
            magnitud_ruido = factor_ruido_actual * std_dev
            direccion_ruido = np.random.choice([-1, 1]) # Sumar o restar el ruido
            
            valor_con_ruido = current_value + (direccion_ruido * magnitud_ruido)

            df_modificado.loc[idx, col_name] = valor_con_ruido
            ruido_inyectado_count +=1
            
        # Actualizar etiquetas para la fila (solo una vez por fila, incluso si múltiples sensores tienen ruido)
        df_modificado.loc[idx, 'etiqueta_deteccion'] = 'anomalia'
        df_modificado.loc[idx, 'etiqueta_tipo_anomalia'] = 'Ruido'
            
    print(f"Inyección de ruido/picos completada. Se modificaron {ruido_inyectado_count} puntos de datos en {len(filas_a_afectar_indices)} filas.")
    return df_modificado

In [ ]:
if ('df_trabajo' in locals() and 
    'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes and
    'std_devs_originales' in locals() and std_devs_originales):

    # Parámetros para la inyección de "Ruido / Picos Aleatorios"
    porcentaje_filas_con_ruido = 0.03  # Afectar al 3% de las filas 'normales'
    factor_std_min = 3.0               # Ruido será al menos 3 * std
    factor_std_max = 6.0               # Ruido será como máximo 6 * std
    num_sensores_afectados_por_fila = 1 # Inyectar ruido en 1 sensor por fila afectada

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    etiquetas_deteccion_antes_ruido = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_ruido = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Ruido / Picos Aleatorios'...")
    df_trabajo = inyectar_ruido_picos(df_trabajo,
                                        porcentaje_filas_con_ruido,
                                        columnas_objetivo_existentes,
                                        std_devs_originales,
                                        factor_std_min,
                                        factor_std_max,
                                        num_sensores_afectados_por_fila)

    print("\n--- Resultados de la Inyección de 'Ruido / Picos Aleatorios' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_ruido)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_ruido)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    num_ruido_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Ruido').sum()
    print(f"\nNúmero total de filas etiquetadas como 'Ruido': {num_ruido_etiquetados}")

else:
    print("Error: Asegúrate de que 'df_trabajo', 'columnas_objetivo_existentes' y 'std_devs_originales' estén definidos.")

4. Valores Fuera de Rango

In [ ]:
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print("Calculando estadísticas de rango para las columnas objetivo (usando df_original):\n")
    
    stats_list = []

    for col in columnas_objetivo_existentes:
        if pd.api.types.is_numeric_dtype(df_original[col]):
            # Calcular estadísticas
            min_obs = df_original[col].min()
            max_obs = df_original[col].max()
            mean_obs = df_original[col].mean()
            std_obs = df_original[col].std()
            
            # Calcular percentiles
            p0_1 = df_original[col].quantile(0.001) # 0.1-th percentile
            p1 = df_original[col].quantile(0.01)   # 1st percentile
            p5 = df_original[col].quantile(0.05)   # 5th percentile
            p95 = df_original[col].quantile(0.95)  # 95th percentile
            p99 = df_original[col].quantile(0.99)  # 99th percentile
            p99_9 = df_original[col].quantile(0.999)# 99.9-th percentile
            
            stats_list.append({
                'Columna': col,
                'Min Observado': min_obs,
                'P0.1 (0.1%)': p0_1,
                'P1 (1%)': p1,
                'P5 (5%)': p5,
                'Media': mean_obs,
                'P95 (95%)': p95,
                'P99 (99%)': p99,
                'P99.9 (99.9%)': p99_9,
                'Max Observado': max_obs,
                'Std Dev': std_obs
            })
        else:
            print(f"  Advertencia: Columna '{col}' no es numérica. Se omitirán las estadísticas.")

    if stats_list:
        df_stats = pd.DataFrame(stats_list)
        print(df_stats.to_string()) # .to_string() para mostrar todas las filas y columnas
    else:
        print("No se pudieron calcular estadísticas para ninguna columna.")

else:
    print("Error: 'df_original' o 'columnas_objetivo_existentes' no están definidos o la lista de columnas está vacía.")

In [ ]:
# Verificamos que las variables necesarias existan
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals():
    print("=== CALCULANDO LÍMITES AUTOMÁTICOS (P0.1 y P99.9) ===\n")
    
    stats_list = []
    # Este diccionario guardará los límites para usarlos después en el modelo
    limites_automaticos = {} 

    for col in columnas_objetivo_existentes:
        if pd.api.types.is_numeric_dtype(df_original[col]):
            # 1. Calcular Percentiles Extremos
            p0_1 = df_original[col].quantile(0.001)  # Límite Inferior Automático
            p99_9 = df_original[col].quantile(0.999) # Límite Superior Automático
            
            # 2. Calcular otras estadísticas para el reporte
            min_obs = df_original[col].min()
            max_obs = df_original[col].max()
            mean_obs = df_original[col].mean()
            
            # 3. Guardar en el diccionario de límites
            limites_automaticos[col] = {
                'min': p0_1,
                'max': p99_9
            }
            
            # 4. Agregar a la lista para mostrar la tabla resumen
            stats_list.append({
                'Columna': col,
                'Min Absoluto': min_obs,
                'Límite Inf (P0.1)': p0_1,
                'Media': mean_obs,
                'Límite Sup (P99.9)': p99_9,
                'Max Absoluto': max_obs
            })
        else:
            print(f"⚠️ Advertencia: '{col}' no es numérica. Omitida.")

    # Mostrar la tabla de resultados
    if stats_list:
        df_stats = pd.DataFrame(stats_list)
        print(df_stats.to_string(index=False))
        print("\n✅ Límites definidos exitosamente en la variable 'limites_automaticos'.")
        
        rangos_validos_sensores = limites_automaticos
    else:
        print("❌ No se calcularon estadísticas.")

else:
    print("❌ Error: df_original o columnas_objetivo_existentes no definidos.")

# --- EJEMPLO DE CÓMO SE USARÍAN ESTOS VALORES AUTOMÁTICOS ---
# Si quisieras ver el límite de una variable:
# print(f"El límite superior de XTS es: {limites_automaticos['XTS']['max']}")

In [ ]:
def inyectar_valores_fuera_rango(df, porcentaje_filas_afectadas, rangos_sensores, 
                                    columnas_con_rango, num_sensores_afectados_por_fila=1, 
                                    margen_error_factor=0.1): 
    """
    Inyecta valores fuera del rango definido para un porcentaje de filas y columnas.

    Args:
        df (pd.DataFrame): DataFrame de trabajo.
        porcentaje_filas_afectadas (float): Porcentaje de filas 'normales' a afectar.
        rangos_sensores (dict): Diccionario con los rangos {'min': X, 'max': Y} para cada sensor.
        columnas_con_rango (list): Lista de columnas para las cuales se han definido rangos.
        num_sensores_afectados_por_fila (int): Cuántos sensores aleatorios por fila afectada se modificarán.
        margen_error_factor (float): Factor del span del rango para determinar cuánto se excede el límite.
                                        (ej: 0.1 = 10% del span del rango válido).

    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas.
    """
    if not columnas_con_rango or not rangos_sensores:
        print("No se proporcionaron columnas con rangos definidos o el diccionario de rangos está vacío.")
        return df

    df_modificado = df.copy()
    
    # Asegurarnos de que las columnas de etiquetas existan antes de consultarlas
    if 'etiqueta_deteccion' not in df_modificado.columns:
        df_modificado['etiqueta_deteccion'] = 'normal'
    if 'etiqueta_tipo_anomalia' not in df_modificado.columns:
        df_modificado['etiqueta_tipo_anomalia'] = 'Ninguna'

    indices_normales = df_modificado[df_modificado['etiqueta_deteccion'] == 'normal'].index
    num_filas_a_afectar = int(len(indices_normales) * porcentaje_filas_afectadas)

    if num_filas_a_afectar == 0 and len(indices_normales) > 0:
        print(f"Con el {porcentaje_filas_afectadas*100:.2f}% de {len(indices_normales)} filas normales, no se seleccionaron filas para afectar (resultado: 0).")
        return df_modificado
    elif len(indices_normales) == 0:
        print("No hay filas 'normales' disponibles para inyectar valores fuera de rango.")
        return df_modificado
        
    filas_a_afectar_indices = np.random.choice(indices_normales, size=num_filas_a_afectar, replace=False)
    
    print(f"Inyectando valores fuera de rango en {len(filas_a_afectar_indices)} filas...")
    
    modificaciones_count = 0
    for idx in filas_a_afectar_indices:
        columnas_para_modificar_actual = np.random.choice(
            columnas_con_rango,
            size=min(num_sensores_afectados_por_fila, len(columnas_con_rango)),
            replace=False
        )
        if not any(columnas_para_modificar_actual):
            continue

        for col_name in columnas_para_modificar_actual:
            if col_name not in rangos_sensores:
                continue 

            rango = rangos_sensores[col_name]
            min_val = rango['min']
            max_val = rango['max']
            
            current_value = df_modificado.loc[idx, col_name]
            if pd.isna(current_value): 
                continue

            span_rango = max_val - min_val
            margen_absoluto = span_rango * margen_error_factor
            if margen_absoluto == 0 : 
                margen_absoluto = (abs(max_val) * 0.1) if max_val != 0 else 1.0

            if np.random.rand() < 0.5: 
                valor_anomalo = min_val - margen_absoluto 
            else: 
                valor_anomalo = max_val + margen_absoluto
            
            df_modificado.loc[idx, col_name] = valor_anomalo
            modificaciones_count += 1
            
        df_modificado.loc[idx, 'etiqueta_deteccion'] = 'anomalia'
        df_modificado.loc[idx, 'etiqueta_tipo_anomalia'] = 'Valores Fuera de Rango'
            
    print(f"Inyección de valores fuera de rango completada. Se modificaron {modificaciones_count} puntos de datos en {len(filas_a_afectar_indices)} filas.")
    return df_modificado

In [ ]:
# 1. Recuperamos los límites automáticos
if 'limites_automaticos' in locals():
    rangos_validos_sensores = limites_automaticos

# 2. Generamos la lista de columnas basada en el diccionario
if 'rangos_validos_sensores' in locals():
    columnas_con_rango_definido = list(rangos_validos_sensores.keys())

# 3. Aseguramos que df_trabajo exista
if 'df_trabajo' not in locals() and 'df_original' in locals():
    df_trabajo = df_original.copy()
    print("✅ 'df_trabajo' creado a partir de 'df_original'.")

# --- FIN DE LA ADAPTACIÓN ---


if ('df_trabajo' in locals() and 
    'columnas_con_rango_definido' in locals() and columnas_con_rango_definido and
    'rangos_validos_sensores' in locals() and rangos_validos_sensores):

    # Parámetros para la inyección de "Valores Fuera de Rango"
    porcentaje_filas_oor = 0.02  # Afectar al 2% de las filas 'normales'
    margen_factor = 0.1          # Exceder el límite por un 10% del span del rango válido
    num_sensores_afectados = 1   # Inyectar en 1 sensor por fila afectada

    # Guardar el estado de las etiquetas antes de la inyección para comparar
    # (Añadido manejo de error por si las columnas no existen aún)
    if 'etiqueta_deteccion' not in df_trabajo.columns:
        df_trabajo['etiqueta_deteccion'] = 'normal'
    if 'etiqueta_tipo_anomalia' not in df_trabajo.columns:
        df_trabajo['etiqueta_tipo_anomalia'] = 'Ninguna'

    etiquetas_deteccion_antes_oor = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_oor = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Valores Fuera de Rango'...")
    df_trabajo = inyectar_valores_fuera_rango(df_trabajo,
                                                porcentaje_filas_oor,
                                                rangos_validos_sensores,
                                                columnas_con_rango_definido,
                                                num_sensores_afectados,
                                                margen_factor)

    print("\n--- Resultados de la Inyección de 'Valores Fuera de Rango' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_oor)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_oor)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    num_oor_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Valores Fuera de Rango').sum()
    print(f"\nNúmero total de filas etiquetadas como 'Valores Fuera de Rango': {num_oor_etiquetados}")

else:
    print("Error: Asegúrate de que 'df_trabajo', 'columnas_con_rango_definido' y 'rangos_validos_sensores' estén definidos y no vacíos.")

5. Desviación Inexplicable Respecto a Pares

Calcular y Visualizar la Matriz de Correlación

Primero, vamos a calcular la matriz de correlación para las columnas_objetivo_existentes (sensores y actuadores) utilizando el DataFrame df_original. Esto nos ayudará a identificar qué variables tienden a moverse juntas (correlación positiva alta) o en direcciones opuestas (correlación negativa alta).

In [ ]:
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals() and columnas_objetivo_existentes:
    print("Calculando la matriz de correlación para las columnas objetivo (usando df_original):\n")
    
    # Asegurarse de que solo se usen columnas numéricas para la correlación
    columnas_numericas_para_corr = df_original[columnas_objetivo_existentes].select_dtypes(include=np.number).columns
    
    if len(columnas_numericas_para_corr) < 2:
        print("No hay suficientes columnas numéricas para calcular una matriz de correlación significativa.")
    else:
        correlation_matrix = df_original[columnas_numericas_para_corr].corr()

        print("Matriz de Correlación:")
        print(correlation_matrix)

        # Visualizar la matriz de correlación usando un heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
        plt.title('Matriz de Correlación de Sensores y Actuadores (sobre df_original)')
        plt.show()
else:
    print("Error: 'df_original' o 'columnas_objetivo_existentes' no están definidos o la lista de columnas está vacía.")

In [ ]:
# Columnas para las que podríamos necesitar percentiles (basadas en los pares sugeridos)
columnas_para_percentiles = list(set(['PRAD', 'PRGINT', 'XTINV', 'XHINV', 'UVENT_cen']))
percentiles_calculados = {}

if 'df_original' in locals():
    print("Calculando percentiles para columnas seleccionadas (desde df_original):")
    for col in columnas_para_percentiles:
        if col in df_original.columns and pd.api.types.is_numeric_dtype(df_original[col]):
            p25 = df_original[col].quantile(0.25)
            p50 = df_original[col].quantile(0.50) # Mediana
            p75 = df_original[col].quantile(0.75)
            percentiles_calculados[col] = {'p25': p25, 'p50': p50, 'p75': p75}
            # print(f"  {col}: P25={p25:.2f}, P50={p50:.2f}, P75={p75:.2f}")
        else:
            print(f"  Advertencia: Columna '{col}' no es numérica o no existe. Se omitirán sus percentiles.")
else:
    print("Error: 'df_original' no está definido. No se pueden calcular percentiles.")

In [ ]:
def inyectar_desviacion_correlacion(df, lista_pares, percentiles_ref, porcentaje_filas_afectadas,
                                    filtro_temporal_callable=None):
    """
    Inyecta anomalías de "Desviación de Correlación" para pares de sensores.
    Modifica un sensor del par para que su valor sea incongruente con el otro,
    basado en su correlación general y usando percentiles para definir valores "altos/bajos".

    Args:
        df (pd.DataFrame): DataFrame de trabajo.
        lista_pares (list): Lista de tuplas, cada tupla es un par de nombres de columna.
                            Ej: [('PRAD', 'PRGINT'), ('XTINV', 'XHINV')]
        percentiles_ref (dict): Diccionario con percentiles precalculados para las columnas.
                                Ej: {'PRAD': {'p25': X, 'p75': Y}, ...}
        porcentaje_filas_afectadas (float): Porcentaje de filas 'normales' a afectar.
        filtro_temporal_callable (function, optional): Una función que toma una fila (Series)
                                                        y devuelve True si la anomalía puede
                                                        ser inyectada en esa fila.
                                                        Ej: lambda row: 7 <= row['Hora'] <= 18
                                                        para inyectar solo durante el día.

    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas.
    """
    if not lista_pares or not percentiles_ref:
        print("Lista de pares o percentiles de referencia no proporcionados.")
        return df

    df_modificado = df.copy()
    
    # Considerar solo filas 'normales' y que cumplan el filtro temporal si se proporciona
    indices_candidatos = df_modificado[df_modificado['etiqueta_deteccion'] == 'normal'].index
    if filtro_temporal_callable:
        filas_cumplen_filtro = df_modificado.loc[indices_candidatos].apply(filtro_temporal_callable, axis=1)
        indices_candidatos = indices_candidatos[filas_cumplen_filtro]

    if len(indices_candidatos) == 0:
        print("No hay filas 'normales' disponibles (o que cumplan el filtro temporal) para inyectar anomalías de desviación de correlación.")
        return df_modificado

    num_filas_a_afectar = int(len(indices_candidatos) * porcentaje_filas_afectadas)
    if num_filas_a_afectar == 0:
        print(f"Con el {porcentaje_filas_afectadas*100:.2f}% de {len(indices_candidatos)} filas candidatas, no se seleccionaron filas para afectar.")
        return df_modificado
        
    filas_a_afectar_indices = np.random.choice(indices_candidatos, size=num_filas_a_afectar, replace=False)
    
    print(f"Inyectando desviación de correlación en {len(filas_a_afectar_indices)} filas...")
    
    modificaciones_count = 0
    for idx in filas_a_afectar_indices:
        # Seleccionar aleatoriamente un par de la lista
        sensor_A_col, sensor_B_col = lista_pares[np.random.choice(len(lista_pares))]

        # Asegurarse de que tenemos percentiles para ambos sensores del par
        if not (sensor_A_col in percentiles_ref and sensor_B_col in percentiles_ref):
            # print(f"  Skipping par ({sensor_A_col}, {sensor_B_col}): faltan percentiles.")
            continue

        val_A_original = df_modificado.loc[idx, sensor_A_col]
        val_B_original = df_modificado.loc[idx, sensor_B_col]

        if pd.isna(val_A_original) or pd.isna(val_B_original):
            # print(f"  Skipping fila {idx} para par ({sensor_A_col}, {sensor_B_col}): valor original es NaN.")
            continue
            
        # Decidir aleatoriamente cuál sensor del par modificar (A o B)
        # Por simplicidad, modificaremos siempre el primero (sensor_A) basado en el segundo (sensor_B)
        # Una versión más avanzada podría alternar.
        sensor_a_modificar = sensor_A_col
        sensor_referencia = sensor_B_col
        val_referencia = val_B_original
        
        # Obtener la correlación para saber si es positiva o negativa (desde la matriz global)
        # Esto es una simplificación; la correlación puede ser no lineal o cambiar con el tiempo.
        # Asumimos que la correlación global nos da una idea general.
        corr_val = correlation_matrix.loc[sensor_a_modificar, sensor_referencia] # Usar la matriz global calculada antes

        nuevo_valor_anomalo = val_A_original # Default si no se puede determinar la lógica

        # Lógica para romper la correlación:
        # Si B está alto (ej. > P75), y la corr(A,B) es positiva, A debería estar alto. Hacemos A bajo.
        # Si B está alto (ej. > P75), y la corr(A,B) es negativa, A debería estar bajo. Hacemos A alto.
        if val_referencia > percentiles_ref[sensor_referencia]['p75']: # Sensor de referencia está "alto"
            if corr_val > 0.3: # Correlación positiva fuerte/moderada
                # A debería ser alto, lo hacemos bajo (P25 de A)
                nuevo_valor_anomalo = percentiles_ref[sensor_a_modificar]['p25']
            elif corr_val < -0.3: # Correlación negativa fuerte/moderada
                # A debería ser bajo, lo hacemos alto (P75 de A)
                nuevo_valor_anomalo = percentiles_ref[sensor_a_modificar]['p75']
        elif val_referencia < percentiles_ref[sensor_referencia]['p25']: # Sensor de referencia está "bajo"
            if corr_val > 0.3:
                # A debería ser bajo, lo hacemos alto (P75 de A)
                nuevo_valor_anomalo = percentiles_ref[sensor_a_modificar]['p75']
            elif corr_val < -0.3:
                # A debería ser alto, lo hacemos bajo (P25 de A)
                nuevo_valor_anomalo = percentiles_ref[sensor_a_modificar]['p25']
        else: # Sensor de referencia está en rango medio, no inyectamos o inyectamos un valor "medio" opuesto
            # Por ahora, si B está en rango medio, no forzamos una desviación tan extrema o podríamos elegir aleatoriamente P25 o P75 de A.
            # Para simplificar, si no se cumple una condición clara (B alto/bajo), no modificamos mucho.
            # Vamos a hacer que si B está en rango medio, A tome un valor del extremo opuesto al esperado por la correlación pero menos agresivo, o simplemente no hacer nada para este caso particular.
            # Por ahora, si B no está en los extremos, no se inyecta esta anomalía para esta fila y par.
            continue # Pasar a la siguiente fila si B no está en un extremo claro.

        if nuevo_valor_anomalo == val_A_original and (val_referencia > percentiles_ref[sensor_referencia]['p75'] or val_referencia < percentiles_ref[sensor_referencia]['p25']):
                # Si B estaba en un extremo pero la lógica no cambió A (ej. A ya estaba en el P25/P75)
                # forzamos un cambio más genérico
                if corr_val > 0: nuevo_valor_anomalo = percentiles_ref[sensor_a_modificar]['p25'] if val_A_original > percentiles_ref[sensor_a_modificar]['p50'] else percentiles_ref[sensor_a_modificar]['p75']
                else: nuevo_valor_anomalo = percentiles_ref[sensor_a_modificar]['p75'] if val_A_original < percentiles_ref[sensor_a_modificar]['p50'] else percentiles_ref[sensor_a_modificar]['p25']


        df_modificado.loc[idx, sensor_a_modificar] = nuevo_valor_anomalo
        modificaciones_count += 1
            
        df_modificado.loc[idx, 'etiqueta_deteccion'] = 'anomalia'
        df_modificado.loc[idx, 'etiqueta_tipo_anomalia'] = 'Desviación de Correlación' # Nombre ajustado
            
    print(f"Inyección de desviación de correlación completada. Se modificaron {modificaciones_count} puntos de datos en filas afectadas.")
    return df_modificado

In [ ]:
# Asegurarse de que correlation_matrix esté disponible globalmente o pasarla a la función si es necesario.
# La función actual la usa como variable global.
if ('df_trabajo' in locals() and 
    'percentiles_calculados' in locals() and percentiles_calculados and
    'correlation_matrix' in locals() ):

    # Pares seleccionados (se pueden añadir más o cambiar)
    pares_para_desviacion = [
        ('PRAD', 'PRGINT'),    # Muy fuerte positiva
        ('XTINV', 'XHINV'),   # Fuerte negativa
        ('PRAD', 'UVENT_cen') # Fuerte positiva (operacional)
    ]

    porcentaje_filas_desviacion = 0.03  # Afectar al 3% de las filas 'normales' (y que cumplan filtro)

    # Filtro temporal opcional (ejemplo: solo durante el día para radiación)
    def filtro_dia(row):
        return 7 <= row['Hora'] <= 18 # Asumiendo que 'Hora' existe en df_trabajo

    # Guardar el estado de las etiquetas antes
    etiquetas_deteccion_antes_corr = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_corr = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Desviación de Correlación'...")
    # Para el par ('PRAD', 'PRGINT') y ('PRAD', 'UVENT_cen') podríamos aplicar el filtro_dia
    # Para ('XTINV', 'XHINV') quizás no necesite filtro temporal o uno diferente.
    # Por ahora, aplicaremos el filtro_dia a todas las inyecciones de este tipo para el ejemplo.
    # En una implementación más detallada, podríamoms llamar a la función varias veces con diferentes pares y filtros.
    
    df_trabajo = inyectar_desviacion_correlacion(df_trabajo,
                                                    pares_para_desviacion,
                                                    percentiles_calculados,
                                                    porcentaje_filas_desviacion,
                                                    filtro_temporal_callable=filtro_dia) # O None si no quieremoa filtro

    print("\n--- Resultados de la Inyección de 'Desviación de Correlación' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_corr)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_corr)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
    
    num_desv_corr_etiquetados = (df_trabajo['etiqueta_tipo_anomalia'] == 'Desviación de Correlación').sum()
    print(f"\nNúmero total de filas etiquetadas como 'Desviación de Correlación': {num_desv_corr_etiquetados}")

else:
    print("Error: Asegúrate de que 'df_trabajo', 'percentiles_calculados' y 'correlation_matrix' estén definidos.")

6. Contextual

La identificación y generación sintética de este tipo de anomalías es particularmente desafiante, ya que no se trata de un simple error de medición obvio, sino de una sutileza en el comportamiento del sistema. Requiere un profundo conocimiento del dominio específico del invernadero, incluyendo la interacción entre múltiples variables (temperatura, humedad, CO2, radiación, ventilación, riego, etc.) y los patrones temporales esperados (ciclos diurnos/nocturnos, estacionales, respuestas a intervenciones). Debido a la vasta cantidad de posibles escenarios contextuales, cubrir exhaustivamente todas las variaciones es complejo; por ello, la generación sintética se enfoca en representar algunos ejemplos claros de violaciones a reglas operativas o físicas conocidas del sistema.

Por este motivo generamos las siguientes posibles situaciones:

| **Variable**     | **Descripción**                                                                 |
|------------------|----------------------------------------------------------------------------------|
| Fecha            | Fecha y hora en formato dd/MM/yyyy HH:mm:ss                                     |
| HR_pasillo       | Humedad relativa del aire interior en el pasillo (%)                            |
| PCO2EXT          | Concentración de CO2 en el aire exterior (ppm)                                  |
| PHEXT            | Humedad relativa del aire exterior (%)                                          |
| PotAct           | Potencia eléctrica activa (kW)                                                  |
| PRAD             | Radiación solar global en el exterior (W/m²)                                    |
| PRGINT           | Radiación solar global en el interior (W/m²)                                    |
| PTEXT            | Temperatura del aire en el exterior (°C)                                        |
| PVV              | Velocidad del viento en el exterior (m/s)                                       |
| T_pasillo        | Temperatura del aire interior en el pasillo (°C)                                |
| XCO2I            | Concentración de CO2 en el aire interior (ppm)                                  |
| XHINV            | Humedad relativa del aire interior del invernadero (%)                          |
| XPARI            | Radiación PAR interior (W/m²)                                                   |
| XTINV            | Temperatura del aire interior del invernadero (°C)                              |
| XTS              | Temperatura del suelo del invernadero (°C)                                      |
| UVENT_cen        | Apertura de las ventanas cenitales (%)                                          |
| UVENT_lN         | Apertura de las ventanas laterales (%)                                          |

1. Comportamiento cíclico natural (día/noche)

Regla:
Si PRAD y PRGINT son altos, XTINV, XPARI y PTEXT tienden a subir, y es esperable una baja en XHINV.

Condición contextual: Entre las 08:00 y 18:00 h (luz solar), especialmente en días despejados (PVV < 2 m/s y PHEXT < 70%).

Posible anomalía contextual: Un aumento brusco de temperatura interior XTINV podría parecer una anomalía si se mira solo esa variable, pero es esperable en presencia de alta radiación.

⸻

2. Riego programado

Regla:
Si hay un aumento repentino en XHINV sin una causa atmosférica externa (PHEXT sigue constante), y no hay apertura de ventanas (UVENT_cen y UVENT_lN cercanos a 0), es posible que el sistema de riego esté activo.

Condición contextual: Hora programada de riego o cuando PotAct muestra un pico coincidente (activación de bombas).

Posible anomalía contextual: Lectura alta de XHINV que sería anómala si no se tiene en cuenta el riego.

⸻

3. Presencia humana o intervención manual

Regla:
Un incremento abrupto y localizado en XTINV o T_pasillo, sin cambios similares en otras variables (PRAD, PRGINT, PVV), podría deberse a la cercanía de una persona o maquinaria caliente.

Condición contextual: Sin intervención programada conocida, sin cambios en ventilación o radiación, duración breve (< 5 minutos).

Posible anomalía contextual: Pico de temperatura interior local sin explicación multivariable, pero plausible al considerar intervención humana.

⸻

4. Iluminación artificial nocturna

Regla:
Si se detecta PRAD > 0 durante el periodo nocturno (ej. 22:00–06:00), y PVV y PTEXT indican condiciones estables, puede ser causado por una linterna o luz artificial.

Condición contextual: Noche, sin luna llena aparente, sin actividad solar.

Posible anomalía contextual: Lectura de radiación interior PRGINT o XPARI durante la noche que no es solar pero puede explicarse por acción humana.

⸻

5. Desviación respecto al comportamiento esperado pero correlacionado con actuadores

Regla:
Si XCO2I cae rápidamente y coincide con una gran apertura en UVENT_cen o UVENT_lN, y hay viento (PVV alto), puede parecer una pérdida de CO2, pero es ventilación controlada.

Condición contextual: Apertura de ventanas activa y diferencia alta entre PCO2EXT y XCO2I.

Posible anomalía contextual: Disminución de CO2 interior explicada por sistema de control automático.

⸻

6. Pico térmico tras intervención energética

Regla:
Si PotAct aumenta repentinamente (activación de calefacción) y eso causa un aumento en T_pasillo y XTINV, puede parecer un pico anómalo pero está contextualizado por el sistema.

Condición contextual: Intervención térmica programada o necesaria (temperatura exterior baja, PTEXT < 5 °C).

Posible anomalía contextual: Temperatura elevada en interior, pero consecuencia de calefacción.

⸻

7. Desfase entre sensores por retraso de efecto físico

Regla:
Si hay un desfase entre PRAD exterior y PRGINT, eso puede generar diferencias temporales de temperatura o humedad interior que parecen anomalías si no se considera el desfase.

Condición contextual: Transición rápida entre nublado y soleado, efecto invernadero.

Posible anomalía contextual: Retrasos temporales en respuesta física de sensores internos frente a externos.

Por lo tanto se propone implementar la lógica para dos de las situaciones anteriores que son bastante claras y utilizan bien los datos:

- Variación de la situación 4: "Iluminación Interior Anómala Nocturna"

    - Condición de Contexto Normal: Es de noche (p.ej., Hora entre 23:00 y 04:00) Y la radiación exterior (PRAD) es efectivamente cero (o muy cercana a cero).
    - Comportamiento Esperado: La radiación global interior (PRGINT) también debería ser cero (o muy cercana a cero).
    - Inyección de Anomalía Contextual: En estas condiciones, modificaremos PRGINT para que tome un valor bajo pero claramente no nulo (p.ej., 20-50 W/m²). Este valor de PRGINT no sería "Fuera de Rango" si ocurriera durante el día, pero es anómalo en el contexto nocturno sin radiación exterior.

- Variación de la situación 5: "Falta de Respuesta del CO2 Interior a la Ventilación"

    - Condición de Contexto Normal: Las ventanas de ventilación están significativamente abiertas (p.ej., UVENT_cen > 70% o UVENT_lN > 70%) Y el CO2 exterior (PCO2EXT) es notablemente más bajo que el CO2 interior (XCO2I) (p.ej., PCO2EXT < XCO2I - 50 ppm, indicando un gradiente que favorece la salida de CO2).
    - Comportamiento Esperado: La concentración de CO2 interior (XCO2I) debería disminuir, tendiendo hacia el nivel exterior.
    - Inyección de Anomalía Contextual: En estas condiciones, y durante un breve periodo (p.ej., 2-3 mediciones consecutivas), haremos que XCO2I se mantenga constante o incluso aumente ligeramente. Nuevamente, los valores resultantes de XCO2I podrían ser normales globalmente, pero su comportamiento es anómalo en este contexto de alta ventilación y gradiente favorable.

Estas dos reglas nos permitirán generar anomalías donde el problema no es un valor extremo aislado, sino un comportamiento que no cuadra con la situación.

Iluminación Interior Anómala Nocturna

Esta anomalía simula que el sensor de radiación interior (PRGINT) detecta luz durante la noche profunda, cuando la radiación exterior (PRAD) es nula.

In [ ]:
def inyectar_luz_nocturna_anomala(df, porcentaje_filas_afectadas, 
                                    hora_noche_inicio=23, hora_noche_fin=4, 
                                    prad_umbral_cero=1.0, 
                                    prgint_anomalo_min=20.0, prgint_anomalo_max=50.0):
    """
    Inyecta anomalías de 'luz nocturna anómala' en PRGINT.

    Args:
        df (pd.DataFrame): DataFrame de trabajo.
        porcentaje_filas_afectadas (float): Porcentaje de filas candidatas a afectar.
        hora_noche_inicio (int): Hora de inicio del periodo nocturno (inclusive, 0-23).
        hora_noche_fin (int): Hora de fin del periodo nocturno (inclusive, 0-23).
        prad_umbral_cero (float): Umbral para considerar PRAD como cero.
        prgint_anomalo_min (float): Valor mínimo para el PRGINT anómalo.
        prgint_anomalo_max (float): Valor máximo para el PRGINT anómalo.

    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas.
    """
    df_modificado = df.copy()

    # Definir condición de noche
    if hora_noche_inicio <= hora_noche_fin: # Noche dentro del mismo día (e.g. 0-4)
        es_noche = (df_modificado['Hora'] >= hora_noche_inicio) & (df_modificado['Hora'] <= hora_noche_fin)
    else: # Noche que cruza la medianoche (e.g. 23-4)
        es_noche = (df_modificado['Hora'] >= hora_noche_inicio) | (df_modificado['Hora'] <= hora_noche_fin)

    # Identificar filas candidatas: noche, PRAD es cero, y actualmente 'normal'
    condiciones_contexto = (
        es_noche &
        (df_modificado['PRAD'] < prad_umbral_cero) &
        (df_modificado['etiqueta_deteccion'] == 'normal')
    )
    indices_candidatos = df_modificado[condiciones_contexto].index

    if len(indices_candidatos) == 0:
        print("No se encontraron filas candidatas para 'luz nocturna anómala'.")
        return df_modificado

    num_filas_a_afectar = int(len(indices_candidatos) * porcentaje_filas_afectadas)
    if num_filas_a_afectar == 0:
        print(f"Con el {porcentaje_filas_afectadas*100:.2f}% de {len(indices_candidatos)} filas candidatas, no se seleccionaron filas para 'luz nocturna anómala'.")
        return df_modificado
        
    filas_a_afectar_indices = np.random.choice(indices_candidatos, size=num_filas_a_afectar, replace=False)
    
    print(f"Inyectando 'luz nocturna anómala' en {len(filas_a_afectar_indices)} filas...")

    for idx in filas_a_afectar_indices:
        valor_prgint_anomalo = np.random.uniform(prgint_anomalo_min, prgint_anomalo_max)
        df_modificado.loc[idx, 'PRGINT'] = valor_prgint_anomalo
        df_modificado.loc[idx, 'etiqueta_deteccion'] = 'anomalia'
        df_modificado.loc[idx, 'etiqueta_tipo_anomalia'] = 'Contextual' # Podríamos sub-etiquetar "Contextual_LuzNocturna"
                                                                        # pero por ahora usamos 'Contextual' general.
    
    print(f"Inyección de 'luz nocturna anómala' completada. {len(filas_a_afectar_indices)} filas modificadas.")
    return df_modificado

# --- Uso de la función para Luz Nocturna Anómala ---
if 'df_trabajo' in locals():
    porcentaje_luz_nocturna = 0.01 # Afectar al 1% de las filas candidatas

    # Guardar el estado de las etiquetas antes
    etiquetas_deteccion_antes_ln = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_ln = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'Luz Nocturna Anómala'...")
    df_trabajo = inyectar_luz_nocturna_anomala(df_trabajo, porcentaje_luz_nocturna)

    print("\n--- Resultados de la Inyección de 'Luz Nocturna Anómala' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_ln)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_ln)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print("Error: 'df_trabajo' no está definido.")

Falta de Respuesta del CO2 Interior a la Ventilación

Esta anomalía simula que el CO2 interior (XCO2I) no disminuye como se esperaría cuando la ventilación está alta y hay un gradiente favorable con el CO2 exterior.

In [ ]:
def inyectar_co2_sin_respuesta_ventilacion(df, num_secuencias_afectadas, 
                                            umbral_vent_abierta=70.0, 
                                            umbral_diff_co2=50.0,
                                            duracion_anomalia_min=2, duracion_anomalia_max=4): # Duración en número de timesteps
    """
    Inyecta anomalías de 'CO2 sin respuesta a ventilación' en XCO2I.

    Args:
        df (pd.DataFrame): DataFrame de trabajo (debe estar ordenado por tiempo con índice reseteado).
        num_secuencias_afectadas (int): Número de secuencias anómalas a crear.
        umbral_vent_abierta (float): % mínimo de apertura de UVENT_cen o UVENT_lN.
        umbral_diff_co2 (float): Diferencia mínima (XCO2I - PCO2EXT) para esperar una bajada.
        duracion_anomalia_min (int): Duración mínima de la no-respuesta (en timesteps).
        duracion_anomalia_max (int): Duración máxima de la no-respuesta (en timesteps).


    Returns:
        pd.DataFrame: DataFrame con las anomalías inyectadas.
    """
    df_modificado = df.copy()
    
    secuencias_creadas = 0
    max_intentos_busqueda_total = num_secuencias_afectadas * 20 # Para evitar bucles muy largos

    print(f"Intentando crear {num_secuencias_afectadas} secuencias de 'CO2 sin respuesta a ventilación'...")

    for _ in range(max_intentos_busqueda_total):
        if secuencias_creadas >= num_secuencias_afectadas:
            break

        # Determinar duración aleatoria para esta secuencia anómala
        duracion_actual_anomalia = np.random.randint(duracion_anomalia_min, duracion_anomalia_max + 1)

        # Identificar posibles puntos de INICIO para la secuencia anómala
        # El contexto debe cumplirse en el punto de inicio, y el segmento debe ser 'normal'
        posibles_inicios = []
        
        # Iterar sobre los índices que podrían ser un inicio válido
        # (dejando espacio para la duración de la anomalía)
        for idx_inicio_cand in range(len(df_modificado) - duracion_actual_anomalia + 1):
            # Condición de contexto en el punto de inicio
            cond_vent_cen = df_modificado.loc[idx_inicio_cand, 'UVENT_cen'] > umbral_vent_abierta
            cond_vent_ln = df_modificado.loc[idx_inicio_cand, 'UVENT_lN'] > umbral_vent_abierta
            cond_grad_co2 = df_modificado.loc[idx_inicio_cand, 'PCO2EXT'] < (df_modificado.loc[idx_inicio_cand, 'XCO2I'] - umbral_diff_co2)
            
            if (cond_vent_cen or cond_vent_ln) and cond_grad_co2:
                # Verificar si todo el segmento es 'normal'
                segmento_es_normal = True
                for k in range(duracion_actual_anomalia):
                    if df_modificado.loc[idx_inicio_cand + k, 'etiqueta_deteccion'] != 'normal':
                        segmento_es_normal = False
                        break
                if segmento_es_normal:
                    posibles_inicios.append(idx_inicio_cand)
        
        if not posibles_inicios:
            continue # No se encontraron inicios válidos para esta iteración/duración, intentar de nuevo

        start_idx = np.random.choice(posibles_inicios)
        
        # Guardar el valor inicial de XCO2I para simular que no baja o sube un poco
        xco2i_inicial_contexto = df_modificado.loc[start_idx, 'XCO2I']
        if pd.isna(xco2i_inicial_contexto): continue # Saltar si el valor base es NaN
        
        print(f"  Creando anomalía CO2 en índice {start_idx} por {duracion_actual_anomalia} periodos.")
        for k in range(duracion_actual_anomalia):
            current_idx = start_idx + k
            # Hacer que XCO2I se mantenga o aumente ligeramente
            # df_modificado.loc[current_idx, 'XCO2I'] = xco2i_inicial_contexto # Opción: se mantiene constante
            df_modificado.loc[current_idx, 'XCO2I'] = xco2i_inicial_contexto + np.random.uniform(-2, 5) # Opción: fluctúa sin bajar o sube un poco
            
            df_modificado.loc[current_idx, 'etiqueta_deteccion'] = 'anomalia'
            df_modificado.loc[current_idx, 'etiqueta_tipo_anomalia'] = 'Contextual'
        
        secuencias_creadas += 1
        print(f"  → Secuencia {secuencias_creadas}/{num_secuencias_afectadas} creada.")
    if secuencias_creadas < num_secuencias_afectadas:
        print(f"  Advertencia: Solo se pudieron crear {secuencias_creadas} de las {num_secuencias_afectadas} secuencias de CO2 solicitadas.")
    print(f"Inyección de 'CO2 sin respuesta a ventilación' completada. {secuencias_creadas} secuencias modificadas.")
    return df_modificado

# --- Uso de la función para CO2 sin Respuesta a Ventilación ---
if 'df_trabajo' in locals():
    num_secuencias_co2_contextual = 33 # Intentar crear 100 secuencias de este tipo

    # Guardar el estado de las etiquetas antes
    etiquetas_deteccion_antes_co2 = df_trabajo['etiqueta_deteccion'].value_counts()
    etiquetas_tipo_antes_co2 = df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False)

    print(f"\nIniciando inyección de 'CO2 sin respuesta a ventilación'...")
    df_trabajo = inyectar_co2_sin_respuesta_ventilacion(df_trabajo, num_secuencias_co2_contextual)

    print("\n--- Resultados de la Inyección de 'CO2 sin respuesta a ventilación' ---")
    print("\nConteo de valores en 'etiqueta_deteccion' ANTES:")
    print(etiquetas_deteccion_antes_co2)
    print("\nConteo de valores en 'etiqueta_deteccion' DESPUÉS:")
    print(df_trabajo['etiqueta_deteccion'].value_counts())

    print("\nConteo de valores en 'etiqueta_tipo_anomalia' ANTES:")
    print(etiquetas_tipo_antes_co2)
    print("\nConteo de valores en 'etiqueta_tipo_anomalia' DESPUÉS:")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))
else:
    print("Error: 'df_trabajo' no está definido.")

Revisión Final de Etiquetas de Tipos de Anomalía

In [ ]:
if 'df_trabajo' in locals():
    print("Conteo completo de valores en 'etiqueta_tipo_anomalia':")
    print(df_trabajo['etiqueta_tipo_anomalia'].value_counts(dropna=False))

    print("\nConteo completo de valores en 'etiqueta_deteccion':")
    print(df_trabajo['etiqueta_deteccion'].value_counts(dropna=False))
    
    # Un pequeño resumen del total de filas y filas anómalas
    total_filas = len(df_trabajo)
    filas_anomalas_detectadas = (df_trabajo['etiqueta_deteccion'] == 'anomalia').sum()
    print(f"\nTotal de filas en df_trabajo: {total_filas}")
    print(f"Total de filas marcadas como 'anomalia' en 'etiqueta_deteccion': {filas_anomalas_detectadas}")
    if total_filas > 0:
        print(f"Porcentaje de filas anómalas: {filas_anomalas_detectadas / total_filas * 100:.2f}%")

else:
    print("Error: 'df_trabajo' no está definido.")

In [ ]:
# --- Guardar el estado completo del proceso (variables seleccionadas) ---

# Lista completa de variables a guardar
variable_names_to_save = [
    'df_trabajo',         # DataFrame con anomalías
    'df_original',        # DataFrame original
    'correlation_matrix', # Matriz de correlación
    'df_stats'            # Estadísticas descriptivas
]

data_to_save = {}
all_vars_found_or_critical_met = True 
found_critical_vars = {
    'df_trabajo': False,
    'df_original': False
}
non_critical_missing = []

print("Intentando recopilar variables para guardar...")
for var_name in variable_names_to_save:
    if var_name in locals():
        data_to_save[var_name] = locals()[var_name]
        if var_name in found_critical_vars:
            found_critical_vars[var_name] = True
        print(f"  ✅ Variable '{var_name}' encontrada y añadida para guardar.")
    else:
        if var_name in found_critical_vars:
            all_vars_found_or_critical_met = False
            print(f"  ❌ ERROR CRÍTICO: La variable crítica '{var_name}' no fue encontrada.")
        else:
            non_critical_missing.append(var_name)
            print(f"  ⚠️ ADVERTENCIA: Variable no crítica '{var_name}' no fue encontrada y no será guardada.")

# Verificación de variables críticas
if not (found_critical_vars.get('df_trabajo', True) and found_critical_vars.get('df_original', True)):
    all_vars_found_or_critical_met = False
    print("⛔ No se guardará el estado debido a la falta de variables críticas.")
elif not data_to_save:
    print("❗ Error: No se encontró ninguna variable para guardar.")
else:
    try:
        file_path = "estado_completo_proceso.pkl"
        with open(file_path, "wb") as f:
            pickle.dump(data_to_save, f)
        print(f"✅ Estado del proceso guardado exitosamente en '{file_path}'.")
        print(f"📝 Variables guardadas: {list(data_to_save.keys())}")
        if non_critical_missing:
            print(f"🔍 Variables no críticas no encontradas: {non_critical_missing}")
    except Exception as e:
        print(f"❌ Error al guardar el estado del proceso: {e}")

In [ ]:
# --- Visualización ---
columna = "XTINV" # Variable a visualizar

# Asegurarse de que los dataframes existan y la columna 'Fecha' sea el índice
# (Esta parte podría ser necesaria si los DataFrames en memoria no tienen el índice configurado)
try:
    if 'df_original' in locals():
        if 'Fecha' in df_original.columns:
            df_original['Fecha'] = pd.to_datetime(df_original['Fecha'])
            df_original.set_index('Fecha', inplace=True)
        elif not isinstance(df_original.index, pd.DatetimeIndex):
            print("Advertencia: df_original está en memoria pero no tiene un DatetimeIndex ni columna 'Fecha' adecuada.")
            # Decide cómo manejar esto, quizás mostrar un error o no graficar df_original

    if 'df_trabajo' in locals():
        if 'Fecha' in df_trabajo.columns:
            df_trabajo['Fecha'] = pd.to_datetime(df_trabajo['Fecha'])
            df_trabajo.set_index('Fecha', inplace=True)
        elif not isinstance(df_trabajo.index, pd.DatetimeIndex):
                print("Advertencia: df_trabajo está en memoria pero no tiene un DatetimeIndex ni columna 'Fecha' adecuada.")
            # Decide cómo manejar esto

except Exception as e:
    print(f"Error al configurar índices para DataFrames en memoria: {e}")


if 'df_original' in locals() and 'df_trabajo' in locals():
    
    fig, axes = plt.subplots(2, 1, figsize=(22, 10), sharex=True) # Ejemplo

    # Graficar df_original
    if 'df_original' in locals() and hasattr(df_original, 'index') and isinstance(df_original.index, pd.DatetimeIndex) and columna in df_original.columns:
        axes[0].plot(df_original.index, df_original[columna], label="Original (df_original - en memoria)", alpha=0.7, color="blue")
        axes[0].set_title(f"{columna} - Datos Originales (df_original - en memoria)")
        axes[0].set_ylabel(columna)
        axes[0].legend()
        axes[0].grid(True, linestyle='--', alpha=0.5)
    elif 'df_original' in locals():
            axes[0].set_title(f"{columna} - Datos Originales (df_original en memoria, pero '{columna}' no encontrada o índice incorrecto)")
    else:
        axes[0].set_title(f"df_original no encontrado en memoria")

    # Graficar df_trabajo (similar lógica de verificación)
    if 'df_trabajo' in locals() and hasattr(df_trabajo, 'index') and isinstance(df_trabajo.index, pd.DatetimeIndex) and columna in df_trabajo.columns and 'etiqueta_tipo_anomalia' in df_trabajo.columns:
        axes[1].plot(df_trabajo.index, df_trabajo[columna], label="Con Anomalías (df_trabajo - en memoria)", linestyle="dashed", alpha=0.7, color="orange")
        
        indices_con_anomalias_inyectadas = df_trabajo[df_trabajo['etiqueta_tipo_anomalia'].notna()].index
        if not indices_con_anomalias_inyectadas.empty:
            axes[1].scatter(
                df_trabajo.loc[indices_con_anomalias_inyectadas].index,
                df_trabajo.loc[indices_con_anomalias_inyectadas, columna],
                color='red', label='Anomalías inyectadas', s=50, edgecolors='black', zorder=3
            )
        axes[1].set_title(f"{columna} - Datos con Anomalías (df_trabajo - en memoria)")
        axes[1].set_xlabel("Fecha")
        axes[1].set_ylabel(columna)
        axes[1].legend()
        axes[1].grid(True, linestyle='--', alpha=0.5)
    elif 'df_trabajo' in locals():
        axes[1].set_title(f"{columna} - Datos con Anomalías (df_trabajo en memoria, pero '{columna}' o 'etiqueta_tipo_anomalia' no encontrada o índice incorrecto)")
    else:
        axes[1].set_title(f"df_trabajo no encontrado en memoria")

    plt.tight_layout()
    plt.show()
else:
    print("Los DataFrames df_original y/o df_trabajo no están disponibles en memoria. No se puede generar la visualización.")

## Modelo 1 - Detector de Anomalías: Normal vs. Anomalía

El objetivo de este primer modelo es simple: dada una fila de datos de los sensores (y nuestras características temporales), predecir si es "normal" o una "anomalia".

### Preparación de Datos

Selección de Características (X) y Variable Objetivo (y)

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import joblib
# import pickle
# from sklearn.model_selection import train_test_split
# from sklearn.impute import SimpleImputer
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import LabelEncoder
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer, KNNImputer
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from pandas import Timedelta
# from collections import defaultdict
# # Configuraciones adicionales para mejorar la visualización (opcional)
# plt.style.use('seaborn-v0_8-whitegrid') # Estilo de gráficos
# pd.set_option('display.max_columns', None) # Para mostrar todas las columnas de un DataFrame
# pd.set_option('display.float_format', lambda x: '%.3f' % x) # Formato para números flotantes

In [ ]:
file_path = "estado_completo_proceso.pkl"
try:
    with open(file_path, "rb") as f:
        loaded_data = pickle.load(f)

    # Cargar cada variable guardada en el entorno global actual
    for var_name, var_value in loaded_data.items():
        globals()[var_name] = var_value

    print(f"✅ Estado del proceso cargado exitosamente desde '{file_path}'.")
    print(f"📦 Variables cargadas: {list(loaded_data.keys())}")

    # Verificación de variables cargadas
    expected_vars = ['df_trabajo', 'df_original', 'correlation_matrix', 'df_stats']
    for var in expected_vars:
        if var in globals():
            print(f"  ✅ '{var}' está disponible.")
        else:
            print(f"  ⚠️ '{var}' no se cargó correctamente.")

except FileNotFoundError:
    print(f"❌ Error: Archivo de estado '{file_path}' no encontrado.")
except Exception as e:
    print(f"❌ Error al cargar el estado del proceso: {e}")

In [ ]:
# --- Crear copias de los DataFrames cargados para trabajar con los modelos ---
if 'df_original' in locals() and 'df_trabajo' in locals():
    df_original_modelo = df_original.copy()
    df_trabajo_modelo = df_trabajo.copy() # Esta es la copia que usaremos para el Modelo 1 y Modelo 2
    print("Copias de los DataFrames creadas: df_original_modelo y df_trabajo_modelo.")
    print(f"df_trabajo_modelo.shape: {df_trabajo_modelo.shape}")
else:
    print("Error: df_original y/o df_trabajo no están cargados. No se pueden crear copias para los modelos.")
    # Es importante manejar este caso, por ejemplo, deteniendo el notebook o levantando un error.
    # Por ahora, solo imprimirá un error. Las celdas siguientes podrían fallar.
    df_trabajo_modelo = None # Para que las siguientes celdas no fallen por variable no definida

In [ ]:
if 'df_trabajo' in locals():
    # Columnas a usar como características (features)
    # Excluimos 'Fecha' (ya usamos sus componentes), y las columnas de etiquetas
    
    # Primero, identificamos todas las columnas que NO son etiquetas ni la fecha original
    columnas_potenciales_features = [col for col in df_trabajo.columns if col not in ['Fecha', 'etiqueta_deteccion', 'etiqueta_tipo_anomalia']]
    
    # Nos aseguramos de que las columnas seleccionadas sean numéricas o puedan ser tratadas por el modelo.
    # Random Forest puede manejar NaNs con algunas implementaciones o si se imputan.
    # Por ahora, las incluimos. La imputación podría ser un paso posterior si es necesario.
    X = df_trabajo[columnas_potenciales_features].copy() 
    print("Características (X) seleccionadas:")
    print(X.head())
    print(f"\nDimensiones de X: {X.shape}")

    # Variable Objetivo (y)
    # Convertimos las etiquetas 'normal' y 'anomalia' a valores numéricos (0 y 1)
    y = df_trabajo['etiqueta_deteccion'].map({'normal': 0, 'anomalia': 1})
    print("\nVariable Objetivo (y) (0 para normal, 1 para anomalia):")
    print(y.head())
    print(y.value_counts()) # Para ver el balance de clases
    print(f"\nDimensiones de y: {y.shape}")

    # Verificar si hay NaNs en y (no debería haber si 'etiqueta_deteccion' siempre tuvo valor)
    if y.isnull().sum() > 0:
        print(f"\n¡Advertencia! Se encontraron {y.isnull().sum()} NaNs en la variable objetivo 'y'. Esto debe corregirse.")

else:
    print("Error: 'df_trabajo' no está definido. No se pueden preparar X e y.")

División de Datos: Entrenamiento y Prueba

In [ ]:
if 'X' in locals() and 'y' in locals():
    # Dividir los datos en conjuntos de entrenamiento y prueba
    # test_size=0.3 significa que el 30% de los datos será para prueba, 70% para entrenamiento.
    # random_state es para asegurar la reproducibilidad de la división.
    # stratify=y es importante para clasificación, especialmente si hay desbalance de clases,
    # asegura que la proporción de clases en 'y' se mantenga en los conjuntos de ent. y prueba.
    
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=0.3, 
            random_state=42, 
            stratify=y # Recomendado para mantener proporción de clases
        )

        print("\nDatos divididos en conjuntos de entrenamiento y prueba:")
        print(f"Forma de X_train: {X_train.shape}, Forma de y_train: {y_train.shape}")
        print(f"Forma de X_test: {X_test.shape}, Forma de y_test: {y_test.shape}")

        print("\nDistribución de la variable objetivo en el conjunto de entrenamiento:")
        print(y_train.value_counts(normalize=True))
        print("\nDistribución de la variable objetivo en el conjunto de prueba:")
        print(y_test.value_counts(normalize=True))
        
    except Exception as e:
        print(f"Error durante la división de datos: {e}")
        print("Asegúrate de que X e y tengan el mismo número de muestras y no haya problemas con 'stratify'.")
        if len(y.value_counts()) < 2 :
                print("  Posible causa: La variable objetivo 'y' tiene menos de 2 clases, 'stratify' podría fallar.")
                print("  Intentando dividir sin stratify (menos ideal si hay desbalance):")
                try:
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
                    print("\n  División sin stratify completada.")
                    print(f"  Forma de X_train: {X_train.shape}, Forma de y_train: {y_train.shape}")
                    print(f"  Forma de X_test: {X_test.shape}, Forma de y_test: {y_test.shape}")
                except Exception as e2:
                    print(f"  Error en división sin stratify: {e2}")


else:
    print("Error: 'X' o 'y' no están definidos. Ejecuta el paso anterior.")

### Entrenamiento del Modelo 1: Detector de Anomalías (Random Forest)

Manejo de Valores Faltantes (NaNs) en las Características (X)

In [ ]:
if 'X_train' in locals() and 'X_test' in locals():
    print("Manejo de NaNs antes de la imputación:")
    print(f"NaNs en X_train: {X_train.isnull().sum().sum()}")
    print(f"NaNs en X_test: {X_test.isnull().sum().sum()}")

    # Crear el imputador usando la mediana Y AÑADIENDO COLUMNAS INDICADORAS
    imputer = SimpleImputer(strategy='median', add_indicator=True)

    # Ajustar el imputador SOLO con los datos de entrenamiento
    # Nota: fit no devuelve nada, solo configura el imputer
    imputer.fit(X_train) 
    
    # Aplicar la imputación (transformar) a los datos de entrenamiento y prueba
    # El resultado será un array de NumPy
    X_train_imputed_np = imputer.transform(X_train)
    X_test_imputed_np = imputer.transform(X_test)

    # Obtener los nombres de las nuevas características (originales + indicadoras)
    try:
        # Intenta obtener los nombres de las características, incluyendo las indicadoras
        feature_names_imputed = imputer.get_feature_names_out(input_features=X_train.columns)
    except AttributeError:
        print("Advertencia: imputer.get_feature_names_out no disponible. Usando nombres genéricos para indicadoras si es necesario.") 
        # Si no se puede obtener, usamos un enfoque alternativo
        feature_names_imputed = list(X_train.columns)
        # Ver cuántas columnas indicadoras se añadieron
        num_original_features = X_train.shape[1]
        num_indicator_features = X_train_imputed_np.shape[1] - num_original_features
        if num_indicator_features > 0:
            # Obtener índices de características que tenían NaNs y para las cuales se creó un indicador
            indicator_indices = imputer.indicator_.features_ # Indices de las cols originales que generaron indicador
            indicator_col_names = [f"missing_{X_train.columns[i]}" for i in indicator_indices]
            feature_names_imputed.extend(indicator_col_names)
        if len(feature_names_imputed) != X_train_imputed_np.shape[1]: # Comprobación de seguridad
            print(f"Discrepancia en nombres de columnas: {len(feature_names_imputed)} vs {X_train_imputed_np.shape[1]}. Feature importance podría no tener nombres correctos.")
            feature_names_imputed = None # Forzar a usar array si los nombres no cuadran


    # Convertir de nuevo a DataFrames de Pandas
    if feature_names_imputed is not None:
        X_train_imputed = pd.DataFrame(X_train_imputed_np, columns=feature_names_imputed, index=X_train.index)
        X_test_imputed = pd.DataFrame(X_test_imputed_np, columns=feature_names_imputed, index=X_test.index)
    else: # Si no pudimos obtener los nombres, trabajamos con arrays de NumPy (menos ideal para feature importance explícita)
        X_train_imputed = X_train_imputed_np
        X_test_imputed = X_test_imputed_np
        print("Advertencia: X_train_imputed y X_test_imputed son arrays NumPy. Nombres de características no disponibles para la tabla de importancia.")


    print("\nManejo de NaNs después de la imputación (con indicadores):")
    # Si son DataFrames, podemos hacer isnull().sum().sum(). Si son arrays, no directamente.
    if isinstance(X_train_imputed, pd.DataFrame):
        print(f"NaNs en X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
        print(f"NaNs en X_test_imputed: {X_test_imputed.isnull().sum().sum()}")
    else: # Si son arrays NumPy, SimpleImputer debería haber manejado todos los NaNs
        print(f"NaNs en X_train_imputed (array NumPy): {np.isnan(X_train_imputed).sum()}")
        print(f"NaNs en X_test_imputed (array NumPy): {np.isnan(X_test_imputed).sum()}")

    print(f"Forma de X_train_imputed: {X_train_imputed.shape} (puede tener más columnas debido a los indicadores)")
    print(f"Forma de X_test_imputed: {X_test_imputed.shape}")
else:
    print("Error: X_train o X_test no están definidos. Ejecuta los pasos anteriores.")

Entrenar el Modelo Random Forest

In [ ]:
if 'X_train_imputed' in locals() and 'y_train' in locals():
    # Inicializar el clasificador Random Forest
    # class_weight='balanced' puede ayudar si hay desbalance entre clases 'normal' y 'anomalia'
    # n_estimators: número de árboles en el bosque.
    # random_state: para reproducibilidad.
    rf_detector = RandomForestClassifier(n_estimators=100,
                                            random_state=42,
                                            class_weight='balanced', # Útil si las clases están desbalanceadas
                                            n_jobs=-1) # Usar todos los procesadores disponibles

    print("\nEntrenando el modelo Random Forest Detector...")
    rf_detector.fit(X_train_imputed, y_train)
    print("Entrenamiento completado.")

else:
    print("Error: X_train_imputed o y_train no están definidos.")

### Realizar Predicciones y Evaluar el Modelo

In [ ]:
if 'rf_detector' in locals() and 'X_test_imputed' in locals() and 'y_test' in locals():
    print("\nRealizando predicciones en el conjunto de prueba...")
    y_pred_detector = rf_detector.predict(X_test_imputed)
    y_pred_proba_detector = rf_detector.predict_proba(X_test_imputed)[:, 1] # Probabilidades para la clase 'anomalia'

    print("\n--- Evaluación del Modelo Detector ---")
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred_detector)
    print(f"Accuracy: {accuracy:.4f}")

    # Matriz de Confusión
    print("\nMatriz de Confusión:")
    # [[VN, FP],
    #  [FN, VP]]
    # VN (True Negative): Normales (no inyectados) correctamente identificados como normales por Autoencoder.
    # FP (False Positive): Normales (no inyectados) incorrectamente identificados como anomalías por Autoencoder.
    # FN (False Negative): Anomalías inyectadas incorrectamente identificadas como normales por Autoencoder.
    # VP (True Positive): Anomalías inyectadas correctamente identificadas como anomalías por Autoencoder.
    cm = confusion_matrix(y_test, y_pred_detector)
    print(cm)
    # Visualizar la matriz de confusión
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal (0)', 'Anomalía (1)'], yticklabels=['Normal (0)', 'Anomalía (1)'])
    plt.xlabel('Predicción')
    plt.ylabel('Valor Real')
    plt.title('Matriz de Confusión - Detector de Anomalías')
    plt.show()

    # Desempaquetar la matriz de confusión para un análisis más detallado
    tn, fp, fn, tp = cm.ravel()
    print(f"\nAnálisis Adicional del Detector:")
    print(f"  - Verdaderos Positivos (VP - Anomalías detectadas): {tp}")
    print(f"  - Falsos Positivos (FP - Normales marcados como anomalías): {fp}")
    print(f"  - Falsos Negativos (FN - Anomalías NO detectadas): {fn}")
    print(f"  - Verdaderos Negativos (TN - Normales correctamente identificados): {tn}")

    # Reporte de Clasificación (Precisión, Recall, F1-score)
    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred_detector, target_names=['Normal (0)', 'Anomalía (1)']))

    # ROC AUC Score
    # (Requiere las probabilidades de la clase positiva)
    roc_auc = roc_auc_score(y_test, y_pred_proba_detector)
    print(f"ROC AUC Score: {roc_auc:.4f}")

    # Precision-Recall AUC Score (Average Precision)
    # Especialmente útil para datasets desbalanceados
    pr_auc = average_precision_score(y_test, y_pred_proba_detector)
    print(f"Precision-Recall AUC Score (Average Precision): {pr_auc:.4f}")

else:
    print("Error: El modelo 'rf_detector' no está entrenado o los datos de prueba no están disponibles.")

Importancia de las Características

In [ ]:
if ('rf_detector' in locals() and
    hasattr(rf_detector, 'feature_importances_') and
    'X_train_imputed' in locals()): # Verificar X_train_imputed

    if not isinstance(X_train_imputed, pd.DataFrame):
        print("Error: X_train_imputed no es un DataFrame de Pandas. No se pueden obtener nombres de características.")
        print("Esto puede suceder si la construcción de feature_names_imputed falló en el paso de imputación.")
    else:
        importances = rf_detector.feature_importances_
        feature_names = X_train_imputed.columns
        
        # Verificar la longitud para evitar el error
        if len(feature_names) == len(importances):
            # Crear un DataFrame para visualizar mejor
            feature_importance_df = pd.DataFrame({'Característica': feature_names, 'Importancia': importances})
            feature_importance_df = feature_importance_df.sort_values(by='Importancia', ascending=False)
            
            print("\n--- Importancia de las Características (Detector de Anomalías) ---")
            print(feature_importance_df.to_string())
            
            # Graficar la importancia de las características
            plt.figure(figsize=(12, max(6, len(feature_names) * 0.4))) 
            plt.barh(feature_importance_df['Característica'], feature_importance_df['Importancia'], color='mediumpurple')
            plt.xlabel("Importancia Relativa")
            plt.ylabel("Característica")
            plt.title("Importancia de las Características - Detector de Anomalías (Random Forest)")
            plt.gca().invert_yaxis() 
            plt.tight_layout() 
            plt.show()
        else:
            print(f"Error de coincidencia de longitud: Nombres de características ({len(feature_names)}) vs. Importancias ({len(importances)}).")
            print("Asegúrate de que X_train_imputed tenga los nombres de columna correctos después de la imputación.")

else:
    print("Error: No se pudo calcular la importancia de las características.")
    print("Asegúrate de que 'rf_detector' esté entrenado y 'X_train_imputed' (como DataFrame con columnas) esté disponible.")

Desglose de Detección por Tipo Real de Anomalía (en el Conjunto de Prueba)

In [ ]:
if ('df_trabajo' in locals() and
    'X_test' in locals() and
    'y_test' in locals() and
    'y_pred_detector' in locals()):

    # Obtener los tipos de anomalía reales para el conjunto de prueba
    # X_test.index contiene los índices originales de df_trabajo para las filas de prueba
    true_anomaly_types_in_test_set = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    
    # Crear el DataFrame para análisis
    df_eval_detector = pd.DataFrame({
        'indice_original': X_test.index, # Para referencia si es necesario
        'y_real_binaria': y_test, # 0 para normal, 1 para anomalia
        'y_pred_detector': y_pred_detector, # 0 para normal predicho, 1 para anomalia predicha
        'tipo_anomalia_real': true_anomaly_types_in_test_set
    })

    print("\n--- Desglose de Detección del Modelo 1 por Tipo Real de Anomalía (en Conjunto de Prueba) ---")

    # Filtrar solo las filas que SON REALMENTE ANOMALÍAS (y_real_binaria == 1)
    df_anomalias_reales_en_test = df_eval_detector[df_eval_detector['y_real_binaria'] == 1]

    if df_anomalias_reales_en_test.empty:
        print("No hay anomalías reales en el conjunto de prueba para analizar.")
    else:
        # Agrupar por el tipo real de anomalía y contar cuántas fueron detectadas
        # (y_pred_detector == 1) y cuántas no (y_pred_detector == 0)
        
        detection_summary = df_anomalias_reales_en_test.groupby('tipo_anomalia_real').agg(
            total_instancias_reales_en_test = pd.NamedAgg(column='y_pred_detector', aggfunc='count'),
            correctamente_detectadas_como_anomalia = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 1).sum()),
            no_detectadas_por_el_detector_FN = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 0).sum())
        ).reset_index()

        # Calcular tasa de detección por tipo
        detection_summary['tasa_deteccion_%'] = (
            detection_summary['correctamente_detectadas_como_anomalia'] / 
            detection_summary['total_instancias_reales_en_test'] * 100
        )
        
        # Ordenar por tasa de detección o total de instancias para mejor visualización
        detection_summary = detection_summary.sort_values(by='tasa_deteccion_%', ascending=False)

        print("\nResumen de detección por tipo de anomalía real:")
        print(detection_summary.to_string())
        
        # Graficar la tasa de detección por tipo de anomalía
        if not detection_summary.empty:
            plt.figure(figsize=(12, max(6, len(detection_summary) * 0.5)))
            
            detection_summary.set_index('tipo_anomalia_real')[['correctamente_detectadas_como_anomalia', 'no_detectadas_por_el_detector_FN']].plot(
                kind='barh', 
                stacked=True, 
                figsize=(12, max(6, len(detection_summary) * 0.6)),
                colormap='viridis'
            )
            plt.title('Rendimiento del Detector por Tipo de Anomalía Real (en Conjunto de Prueba)')
            plt.xlabel('Número de Instancias')
            plt.ylabel('Tipo de Anomalía Real')
            plt.legend(title='Resultado Detección', loc='lower right')
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(10, max(5, len(detection_summary) * 0.5) ))
            sns.barplot(x='tasa_deteccion_%', y='tipo_anomalia_real', data=detection_summary, palette='magma', hue='tipo_anomalia_real', dodge=False, legend=False)
            plt.title('Tasa de Detección del Modelo 1 por Tipo de Anomalía Real (%)')
            plt.xlabel('Tasa de Detección (%)')
            plt.ylabel('Tipo de Anomalía Real')
            plt.xlim(0, 105)
            for index, row in detection_summary.iterrows():
                plt.text(row['tasa_deteccion_%'] + 1,
                        index,
                        f"{row['tasa_deteccion_%']:.1f}%", 
                        color='black', 
                        ha="left", 
                        va="center")
            plt.tight_layout()
            plt.show()

else:
    print("Error: No se pueden generar los datos de desglose.")
    print("Asegúrate de que 'df_trabajo', 'X_test', 'y_test', y 'y_pred_detector' estén disponibles.")

## Modelo 2 - Clasificador de tipos de anomalía

Este modelo tomará como entrada las instancias que ya sabemos (o que el Modelo 1 ha detectado) que son anomalías, y su tarea será decirnos a qué tipo específico de anomalía corresponden: "Datos Faltantes", "Sensor Atascado", "Ruido", "Valores Fuera de Rango", "Desviación de Correlación" o "Contextual".

### Preparación de Datos para el Modelo 2 (Clasificador de Tipos de Anomalía)

Selección de Datos de Anomalías para Entrenamiento y Prueba

### Entrenamiento del Modelo 2: Clasificador de Tipos de Anomalía (Random Forest)

Entrenar el Modelo Random Forest para Clasificación de Tipos

In [ ]:
if ('X_train_imputed' in locals() and 'y_train' in locals() and
    'X_test_imputed' in locals() and 'y_test' in locals() and
    'df_trabajo' in locals()):

    # --- Preparar datos de ENTRENAMIENTO para Modelo 2 ---
    # Filtrar X_train_imputed para obtener solo las filas que son verdaderas anomalías
    indices_anomalias_train = X_train_imputed[y_train == 1].index
    X_model2_train = X_train_imputed.loc[indices_anomalias_train]
    
    # Obtener las etiquetas de tipo de anomalía correspondientes del df_trabajo original
    y_model2_train_str = df_trabajo.loc[indices_anomalias_train, 'etiqueta_tipo_anomalia']

    print(f"Forma de X_model2_train: {X_model2_train.shape}")
    if X_model2_train.empty:
        print("Advertencia: No se encontraron anomalías en el conjunto de entrenamiento para Modelo 2.")
    else:
        print("Distribución de tipos de anomalía en y_model2_train_str:")
        print(y_model2_train_str.value_counts())

    # --- Preparar datos de PRUEBA para Modelo 2 ---
    # Filtrar X_test_imputed para obtener solo las filas que son verdaderas anomalías
    indices_anomalias_test = X_test_imputed[y_test == 1].index
    X_model2_test = X_test_imputed.loc[indices_anomalias_test]
    
    # Obtener las etiquetas de tipo de anomalía correspondientes
    y_model2_test_str = df_trabajo.loc[indices_anomalias_test, 'etiqueta_tipo_anomalia']

    print(f"\nForma de X_model2_test: {X_model2_test.shape}")
    if X_model2_test.empty:
        print("Advertencia: No se encontraron anomalías en el conjunto de prueba para Modelo 2.")
    else:
        print("Distribución de tipos de anomalía en y_model2_test_str:")
        print(y_model2_test_str.value_counts())

    # --- Codificación de Etiquetas de Tipo de Anomalía ---
    # Las etiquetas son categóricas (strings), necesitamos convertirlas a números.
    if not X_model2_train.empty and not X_model2_test.empty:
        label_encoder_model2 = LabelEncoder()

        # Ajustar el codificador SOLO con las etiquetas de entrenamiento
        y_model2_train_encoded = label_encoder_model2.fit_transform(y_model2_train_str)
        
        # Transformar las etiquetas de prueba usando el codificador ya ajustado
        y_model2_test_encoded = label_encoder_model2.transform(y_model2_test_str)

        print("\nEtiquetas codificadas para Modelo 2:")
        print(f"Primeras 5 y_model2_train_encoded: {y_model2_train_encoded[:5]}")
        print(f"Primeras 5 y_model2_test_encoded: {y_model2_test_encoded[:5]}")
        
        # Guardar las clases para referencia futura (ej. para la matriz de confusión)
        clases_modelo2 = label_encoder_model2.classes_
        print(f"Clases del Modelo 2 (codificadas de 0 a {len(clases_modelo2)-1}): {clases_modelo2}")
        
        # Verificar que no haya NaNs en las etiquetas codificadas (no debería haber si todas las anomalías tenían tipo)
        if pd.Series(y_model2_train_encoded).isnull().sum() > 0 or pd.Series(y_model2_test_encoded).isnull().sum() > 0:
            print("¡Advertencia! Se encontraron NaNs en las etiquetas codificadas del Modelo 2.")

    else:
        print("\nNo se procederá con la codificación de etiquetas ya que no hay suficientes datos de anomalías.")

else:
    print("Error: Datos necesarios para Modelo 2 no están definidos ('X_train_imputed', 'y_train', etc.).")

In [ ]:
# Verificar que los datos para Modelo 2 existan y no estén vacíos
if ('X_model2_train' in locals() and not X_model2_train.empty and
    'y_model2_train_encoded' in locals() and len(y_model2_train_encoded) > 0):

    # Inicializar el clasificador Random Forest para Modelo 2
    # class_weight='balanced' es especialmente importante aquí, ya que el número de instancias de cada tipo de anomalía puede ser muy diferente.
    rf_clasificador_tipo = RandomForestClassifier(
        n_estimators=100,        # Número de árboles
        random_state=42,         # Para reproducibilidad
        class_weight='balanced', # Pondera las clases inversamente a su frecuencia
        n_jobs=-1                # Usar todos los procesadores
    )

    print("\nEntrenando el Modelo 2 (Clasificador de Tipos de Anomalía)...")
    rf_clasificador_tipo.fit(X_model2_train, y_model2_train_encoded)
    print("Entrenamiento del Modelo 2 completado.")

else:
    print("Error: Datos de entrenamiento para Modelo 2 (X_model2_train, y_model2_train_encoded) no están definidos o están vacíos.")
    # Detener la ejecución de los siguientes bloques si no hay datos para entrenar
    rf_clasificador_tipo = None

### Realizar Predicciones y Evaluar el Modelo 2

In [ ]:
if (rf_clasificador_tipo is not None and
    'X_model2_test' in locals() and not X_model2_test.empty and
    'y_model2_test_encoded' in locals() and len(y_model2_test_encoded) > 0 and
    'clases_modelo2' in locals()):

    print("\nRealizando predicciones con Modelo 2 en el conjunto de prueba...")
    y_pred_model2 = rf_clasificador_tipo.predict(X_model2_test)
    # y_pred_proba_model2 = rf_clasificador_tipo.predict_proba(X_model2_test) # Para métricas basadas en probabilidad si se necesitaran

    print("\n--- Evaluación del Modelo 2 (Clasificador de Tipos de Anomalía) ---")
    
    # Accuracy General
    accuracy_model2 = accuracy_score(y_model2_test_encoded, y_pred_model2)
    print(f"Accuracy General del Modelo 2: {accuracy_model2:.4f}")

    # Matriz de Confusión Multi-clase
    print("\nMatriz de Confusión del Modelo 2:")
    cm_model2 = confusion_matrix(y_model2_test_encoded, y_pred_model2)
    # print(cm_model2) # Imprimir la matriz numérica si se desea

    # Visualizar la matriz de confusión
    plt.figure(figsize=(10, 8)) # Ajustar tamaño según número de clases
    sns.heatmap(cm_model2, annot=True, fmt='d', cmap='Greens',
                xticklabels=clases_modelo2, yticklabels=clases_modelo2)
    plt.xlabel('Predicción del Tipo de Anomalía')
    plt.ylabel('Tipo de Anomalía Real')
    plt.title('Matriz de Confusión - Modelo 2 (Clasificador de Tipos)')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Reporte de Clasificación Detallado (Precisión, Recall, F1-score por clase)
    print("\nReporte de Clasificación del Modelo 2:")
    print(classification_report(y_model2_test_encoded, y_pred_model2, target_names=clases_modelo2))

else:
    print("Error: Modelo 2 no entrenado o datos de prueba para Modelo 2 no disponibles/vacíos.")
    if 'clases_modelo2' not in locals():
        print("  Además, 'clases_modelo2' (nombres de las clases) no está definido.")

In [ ]:
# --- Lista completa de variables a guardar ---
variable_names_to_save = [
    'X', 'X_test', 'X_train', 'X_test_imputed', 'X_train_imputed',
    'X_model2_test', 'X_model2_train',
    'correlation_matrix', 'df_original', 'df_original_modelo',
    'df_stats', 'df_trabajo', 'df_trabajo_modelo',
    'feature_importance_df', 'var_value'
]

# Si tenemos variables críticas que sí o sí deben estar, las definimos aquí:
found_critical_vars = {
    'X': False,
    'X_train': False,
    'X_test': False
}

data_to_save = {}
all_vars_found_or_critical_met = True
non_critical_missing = []

print("📦 Intentando recopilar variables para guardar...")
for var_name in variable_names_to_save:
    if var_name in locals():
        data_to_save[var_name] = locals()[var_name]
        if var_name in found_critical_vars:
            found_critical_vars[var_name] = True
        print(f"  ✅ '{var_name}' encontrada y añadida.")
    else:
        if var_name in found_critical_vars:
            all_vars_found_or_critical_met = False
            print(f"  ❌ ERROR CRÍTICO: '{var_name}' no fue encontrada.")
        else:
            non_critical_missing.append(var_name)
            print(f"  ⚠️ ADVERTENCIA: '{var_name}' no fue encontrada.")

# Verificación de variables críticas
if not all(found_critical_vars.values()):
    all_vars_found_or_critical_met = False
    print("⛔ No se guardará el estado: faltan variables críticas.")
elif not data_to_save:
    print("❗ No se encontró ninguna variable para guardar.")
else:
    try:
        file_path = "estado_completo_proceso_2.pkl"
        with open(file_path, "wb") as f:
            pickle.dump(data_to_save, f)
        print(f"✅ Estado guardado exitosamente en '{file_path}'.")
        print(f"📝 Variables guardadas: {list(data_to_save.keys())}")
        if non_critical_missing:
            print(f"🔍 Variables no críticas no encontradas: {non_critical_missing}")
    except Exception as e:
        print(f"❌ Error al guardar el estado: {e}")

## Corrección de Anomalías

Ahora que tenemos:

1. Un Modelo 1 (Detector) que nos dice si una fila de datos es "normal" o "anomalia".

2. Un Modelo 2 (Clasificador de Tipos) que, para las filas marcadas como "anomalia", nos dice qué tipo específico de anomalía es (Contextual, Datos Faltantes, Desviación de Correlación, Ruido, Sensor Atascado, Valores Fuera de Rango).

El siguiente paso es decidir cómo corregir cada tipo de anomalía detectada y clasificada. El objetivo es mejorar la calidad de los datos para optimizar la toma de decisiones.

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import joblib
# import pickle
# from sklearn.model_selection import train_test_split
# from sklearn.impute import SimpleImputer
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import LabelEncoder
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer, KNNImputer
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from pandas import Timedelta
# from collections import defaultdict
# # Configuraciones adicionales para mejorar la visualización (opcional)
# plt.style.use('seaborn-v0_8-whitegrid') # Estilo de gráficos
# pd.set_option('display.max_columns', None) # Para mostrar todas las columnas de un DataFrame
# pd.set_option('display.float_format', lambda x: '%.3f' % x) # Formato para números flotantes

In [ ]:
# --- Cargar el estado completo del proceso guardado ---
file_path = "estado_completo_proceso_2.pkl"

try:
    with open(file_path, "rb") as f:
        loaded_data = pickle.load(f)

    # Cargar cada variable guardada en el entorno global actual
    for var_name, var_value in loaded_data.items():
        globals()[var_name] = var_value

    print(f"✅ Estado cargado exitosamente desde '{file_path}'.")
    print(f"📦 Variables cargadas: {list(loaded_data.keys())}")

    # Lista de variables clave que deberías tener cargadas
    expected_vars = [
        'X', 'X_test', 'X_train', 'X_test_imputed', 'X_train_imputed',
        'X_model2_test', 'X_model2_train',
        'correlation_matrix', 'df_original', 'df_original_modelo',
        'df_stats', 'df_trabajo', 'df_trabajo_modelo',
        'feature_importance_df', 'var_value'
    ]

    # Verificación de variables importantes
    print("\n🔍 Verificando variables importantes:")
    for var in expected_vars:
        if var in globals():
            print(f"  ✅ '{var}' está disponible.")
        else:
            print(f"  ⚠️ '{var}' NO se encontró en el entorno.")

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{file_path}'.")
except Exception as e:
    print(f"❌ Error al cargar el estado del proceso: {e}")

### 1. Corrección de "Datos Faltantes" con IterativeImputer

Preparación e Instanciación del Imputador

In [ ]:
# Suponiendo que ya tenemos df_trabajo cargado y contiene las columnas necesarias:
columnas_sensores_y_actuadores = [
    'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
    'XCO2I', 'XHINV', 'XTINV', 'XTS',
    'UVENT_cen', 'UVENT_lN'
]

# Esta es la variable que se usará más adelante
columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo.columns]

# Definimos df_para_corregir como una copia del DataFrame base
df_para_corregir = df_trabajo.copy()

print("columnas_objetivo_existentes:", columnas_objetivo_existentes)
print("df_para_corregir.shape:", df_para_corregir.shape)

In [ ]:
if 'df_para_corregir' in locals() and df_para_corregir is not None and 'columnas_objetivo_existentes' in locals():
    
    # Vamos a redefinir columnas_para_imputacion basándonos en df_para_corregir
    # Excluiremos las columnas de etiquetas/predicciones que añadimos
    columnas_a_excluir_de_imputacion = ['gt_estado', 'gt_tipo_anomalia', 'pred_estado', 'pred_tipo_anomalia']
    columnas_potenciales_para_imputacion = [col for col in df_para_corregir.columns if col not in columnas_a_excluir_de_imputacion]
    
    columnas_numericas_para_imputacion = df_para_corregir[columnas_potenciales_para_imputacion].select_dtypes(include=np.number).columns.tolist()

    if not columnas_numericas_para_imputacion:
        print("Advertencia: No se encontraron columnas numéricas adecuadas para la imputación iterativa en 'df_para_corregir'.")
        iterative_imputer_faltantes = None # Usar un nombre específico para este imputador
    else:
        print(f"Columnas que se considerarán para IterativeImputer (de df_para_corregir): {columnas_numericas_para_imputacion}")

        # Crear una copia de df_para_corregir para realizar la corrección
        df_corregido_faltantes = df_para_corregir.copy() # Usar un nombre específico

        iterative_imputer_faltantes = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1, max_depth=10, min_samples_leaf=5),
            max_iter=10,
            random_state=42,
            initial_strategy='median',
            imputation_order='ascending',
            verbose=1
        )
else:
    print("Error: 'df_para_corregir' o 'columnas_objetivo_existentes' no están definidos o df_para_corregir no se cargó.")
    iterative_imputer_faltantes = None
    df_corregido_faltantes = None # Asegurar que no esté definido si hay error
    columnas_numericas_para_imputacion = []

Aplicar IterativeImputer para Corregir los "Datos Faltantes"

In [ ]:
if iterative_imputer_faltantes is not None and df_corregido_faltantes is not None and columnas_numericas_para_imputacion:

    subset_indices_faltantes = df_corregido_faltantes.index # Índice del DataFrame actual

    print(f"\nValores NaN en 'df_corregido_faltantes' ANTES de IterativeImputer (para columnas seleccionadas):")
    nans_antes_faltantes = df_corregido_faltantes[columnas_numericas_para_imputacion].isnull().sum()
    print(nans_antes_faltantes[nans_antes_faltantes > 0])
    total_nans_antes_faltantes = nans_antes_faltantes.sum()

    print("\nEjecutando IterativeImputer para 'Datos Faltantes' (esto puede tardar un poco)...")
    # Aplicar fit_transform al subset de df_corregido_faltantes que tiene las columnas numéricas
    df_subset_imputado_np_faltantes = iterative_imputer_faltantes.fit_transform(df_corregido_faltantes[columnas_numericas_para_imputacion])

    df_subset_imputado_faltantes = pd.DataFrame(df_subset_imputado_np_faltantes, columns=columnas_numericas_para_imputacion, index=subset_indices_faltantes)

    if 'pred_tipo_anomalia' in df_para_corregir.columns:
        mask_pred_faltantes = df_para_corregir['pred_tipo_anomalia'] == 'Datos Faltantes'
    else:
        mask_pred_faltantes = pd.Series(True, index=df_para_corregir.index) # Todo en True para depender solo del NaN

    for col in columnas_numericas_para_imputacion:
        # Crear una máscara de los valores que son NaN originalmente en esta columna
        mask_nan = df_para_corregir[col].isnull()

        # Combinamos ambas máscaras para obtener solo las celdas exactas que debemos reemplazar
        mask_a_corregir = mask_pred_faltantes & mask_nan
        
        # Reemplazamos solo las anomalías clasificadas como "Datos Faltantes"
        df_corregido_faltantes.loc[mask_a_corregir, col] = df_subset_imputado_faltantes.loc[mask_a_corregir, col]

    print("\nIterativeImputer para 'Datos Faltantes' completado.")
    nans_despues_faltantes = df_corregido_faltantes[columnas_numericas_para_imputacion].isnull().sum().sum()
    print(f"Total NaNs antes: {total_nans_antes_faltantes}, Total NaNs después: {nans_despues_faltantes}")
    print(f"Se corrigieron {total_nans_antes_faltantes - nans_despues_faltantes} anomalías de 'Datos Faltantes'.")

    if 'df_original' in locals(): # Necesita el df_original original, limpio.
        print("\n--- Evaluación de la Calidad de Imputación de Datos Faltantes (comparando con df_original) ---")
        all_true_values_faltantes_eval = []
        all_imputed_values_faltantes_eval = []
        column_metrics_faltantes_eval = []

        for col in columnas_numericas_para_imputacion:
            
            # Obtener los índices de df_para_corregir (que es el test set)
            indices_test_set = df_para_corregir.index
            
            # Identificar dónde df_trabajo tenía NaNs dentro de este test set
            nan_mask_en_trabajo_para_test_set = df_trabajo.loc[indices_test_set, col].isnull()
            
            # Asegurarnos de que estamos evaluando únicamente los que el modelo predijo como "Datos Faltantes"
            if 'pred_tipo_anomalia' in df_para_corregir.columns:
                mask_eval_pred = df_para_corregir.loc[indices_test_set, 'pred_tipo_anomalia'] == 'Datos Faltantes'
            else:
                mask_eval_pred = pd.Series(True, index=indices_test_set)
            
            mask_evaluacion_final = nan_mask_en_trabajo_para_test_set & mask_eval_pred
            indices_nan_evaluables = indices_test_set[mask_evaluacion_final]

            if len(indices_nan_evaluables) == 0:
                continue

            true_values_eval = df_original.loc[indices_nan_evaluables, col]
            imputed_values_eval = df_corregido_faltantes.loc[indices_nan_evaluables, col]
            
            valid_comparison_mask_eval = true_values_eval.notna()
            true_values_valid_eval = true_values_eval[valid_comparison_mask_eval]
            imputed_values_valid_eval = imputed_values_eval[valid_comparison_mask_eval]

            if len(true_values_valid_eval) == 0:
                continue
            
            all_true_values_faltantes_eval.extend(true_values_valid_eval.tolist())
            all_imputed_values_faltantes_eval.extend(imputed_values_valid_eval.tolist())
            
            rmse_eval = np.sqrt(mean_squared_error(true_values_valid_eval, imputed_values_valid_eval))
            mae_eval = mean_absolute_error(true_values_valid_eval, imputed_values_valid_eval)
            
            column_metrics_faltantes_eval.append({
                'Columna': col, 'Num_Val_Evaluados': len(true_values_valid_eval),
                'RMSE': rmse_eval, 'MAE': mae_eval,
                'Media_Real': true_values_valid_eval.mean(), 'Media_Imputada': imputed_values_valid_eval.mean()
            })

        if column_metrics_faltantes_eval:
            df_imputation_metrics_faltantes_eval = pd.DataFrame(column_metrics_faltantes_eval)
            print("\nMétricas de Calidad de Imputación (Datos Faltantes en df_resultados_test_con_predicciones):")
            print(df_imputation_metrics_faltantes_eval.to_string())

            if all_true_values_faltantes_eval and all_imputed_values_faltantes_eval:
                overall_rmse_faltantes_eval = np.sqrt(mean_squared_error(all_true_values_faltantes_eval, all_imputed_values_faltantes_eval))
                overall_mae_faltantes_eval = mean_absolute_error(all_true_values_faltantes_eval, all_imputed_values_faltantes_eval)
                print(f"\nRMSE Global de Imputación (Datos Faltantes): {overall_rmse_faltantes_eval:.3f}")
                print(f"MAE Global de Imputación (Datos Faltantes): {overall_mae_faltantes_eval:.3f}")
        else:
            print("\nNo se evaluaron valores imputados para 'Datos Faltantes' (quizás no había NaNs predichos como tal en df_para_corregir o en df_original para comparar).")

else:
    print("No se pudo ejecutar la corrección de 'Datos Faltantes' porque las variables necesarias no están definidas o no hay columnas para imputar.")

In [ ]:
if ('df_original' in locals() and 
    'df_trabajo' in locals() and 
    'df_corregido_faltantes' in locals() and
    'columnas_numericas_para_imputacion' in locals() and
    columnas_numericas_para_imputacion):

    print("--- Evaluación de la Calidad de Imputación de IterativeImputer ---")
    
    all_true_values = []
    all_imputed_values = []
    column_metrics = []

    # Obtener el índice del DataFrame que fue realmente imputado
    # (df_corregido_faltantes, que se basa en df_para_corregir/X_test)
    index_of_imputed_subset = df_corregido_faltantes.index

    for col in columnas_numericas_para_imputacion:
        # 1. Identificar todos los índices en el df_trabajo completo donde 'col' era NaN
        all_nan_indices_in_df_trabajo = df_trabajo[df_trabajo[col].isnull()].index
        
        if all_nan_indices_in_df_trabajo.empty:
            # print(f"No se encontraron NaNs originales en la columna '{col}' de df_trabajo para evaluar.")
            continue

        # 2. Filtrar estos índices: considerar solo aquellos que también existen en el DataFrame imputado (df_corregido_faltantes)
        #    Esto es crucial porque df_corregido_faltantes es un subconjunto (el conjunto de prueba).
        indices_a_evaluar = index_of_imputed_subset.intersection(all_nan_indices_in_df_trabajo)

        # Filtrar adicionalmente para evaluar solo lo que el modelo identificó como "Datos Faltantes"
        if 'pred_tipo_anomalia' in df_para_corregir.columns:
            mask_pred_eval_global = df_para_corregir.loc[indices_a_evaluar, 'pred_tipo_anomalia'] == 'Datos Faltantes'
            indices_a_evaluar = indices_a_evaluar[mask_pred_eval_global]
        
        if indices_a_evaluar.empty:
            # print(f"No hay NaNs de df_trabajo para la columna '{col}' que correspondan a filas en df_corregido_faltantes para evaluar.")
            continue

        # Obtener los valores verdaderos de df_original para estas celdas específicas
        true_values = df_original.loc[indices_a_evaluar, col]
        
        # Obtener los valores imputados de df_corregido_faltantes para esas mismas celdas
        # Esta línea ahora debería funcionar sin KeyError
        imputed_values = df_corregido_faltantes.loc[indices_a_evaluar, col]
        
        # Es importante asegurarse de que estamos comparando solo donde true_values no sea NaN
        valid_comparison_mask = true_values.notna()
        true_values_valid = true_values[valid_comparison_mask]
        imputed_values_valid = imputed_values[valid_comparison_mask]

        if len(true_values_valid) == 0:
            # print(f"No hay valores verdaderos válidos en df_original para la columna '{col}' en los índices de NaN evaluados. No se puede evaluar.")
            continue
            
        all_true_values.extend(true_values_valid.tolist())
        all_imputed_values.extend(imputed_values_valid.tolist())
        
        # Calcular métricas para esta columna
        rmse = np.sqrt(mean_squared_error(true_values_valid, imputed_values_valid))
        mae = mean_absolute_error(true_values_valid, imputed_values_valid)
        
        column_metrics.append({
            'Columna': col,
            'Num_Valores_Imputados_Evaluados': len(true_values_valid),
            'RMSE': rmse,
            'MAE': mae,
            'Media_Real': true_values_valid.mean(),
            'Media_Imputada': imputed_values_valid.mean()
        })
        # print(f"Columna: {col}, RMSE: {rmse:.3f}, MAE: {mae:.3f}, N_eval: {len(true_values_valid)}")

    if column_metrics:
        df_imputation_metrics = pd.DataFrame(column_metrics)
        print("\nMétricas de Calidad de Imputación por Columna (comparando con df_original):")
        print(df_imputation_metrics.to_string())

        # Calcular métricas globales si hay valores para comparar
        if all_true_values and all_imputed_values:
            overall_rmse = np.sqrt(mean_squared_error(all_true_values, all_imputed_values))
            overall_mae = mean_absolute_error(all_true_values, all_imputed_values)
            print(f"\nRMSE Global de Imputación: {overall_rmse:.3f}")
            print(f"MAE Global de Imputación: {overall_mae:.3f}")

            # Gráfico de Dispersión: Valores Reales vs. Imputados
            plt.figure(figsize=(8, 8))
            sample_size = min(len(all_true_values), 2000) # Tomar una muestra
            if sample_size > 0: # Solo tomar muestra si hay puntos
                sample_indices = np.random.choice(len(all_true_values), sample_size, replace=False)
            
                plt.scatter(
                    np.array(all_true_values)[sample_indices], 
                    np.array(all_imputed_values)[sample_indices], 
                    alpha=0.5, s=10, label='Corrected Points', c='blue'
                )
                min_val = min(min(all_true_values), min(all_imputed_values))
                max_val = max(max(all_true_values), max(all_imputed_values))
                plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal')
                plt.xlabel("True Values")
                plt.ylabel("Corrected Values")
                plt.legend()
                plt.grid(True)
                plt.tight_layout()
                plt.show()
            else:
                print("\nNo hay puntos suficientes para el gráfico de dispersión global.")
        else:
            print("\nNo hay suficientes valores válidos comparables para calcular métricas globales o graficar.")
            
    else:
        print("\nNo se pudieron calcular métricas de imputación para ninguna columna (quizás no había NaNs relevantes o valores verdaderos para comparar).")
else:
    print("Error: DataFrames necesarios ('df_original', 'df_trabajo', 'df_corregido_faltantes') o 'columnas_numericas_para_imputacion' no están definidos o la lista de columnas está vacía.")

### 2. Corrección de "Sensor Atascado" con metodo Híbrido (Corrección Estacional Simple - Interpolación Lineal)

In [ ]:
# --- Función para marcar NaNs ---
def marcar_segmentos_stuck_como_nan_condicional(df_a_marcar, columna_nombre,
                                                predicciones_tipo_M2_series,
                                                nombre_clase_sensor_atascado="Sensor Atascado",
                                                min_duracion_atasco=3):
    valores = df_a_marcar[columna_nombre].values
    indices_df = df_a_marcar.index
    n = len(valores)
    segmentos_nanificados_en_col = []
    nan_count_col = 0
    i = 0
    while i < n:
        if pd.isna(valores[i]):
            i += 1
            continue
        j = i
        while j < n and pd.notna(valores[j]) and valores[j] == valores[i]:
            j += 1
        duracion_actual_atasco = j - i
        fue_clasificado_como_atasco = False
        if duracion_actual_atasco >= min_duracion_atasco:
            indices_reales_segmento_df = indices_df[i:j]
            if not indices_reales_segmento_df.empty:
                # Asegurar compatibilidad de índices antes de la intersección
                if not predicciones_tipo_M2_series.empty and isinstance(predicciones_tipo_M2_series.index, type(indices_df)):
                    indices_validos_en_predicciones = predicciones_tipo_M2_series.index.intersection(indices_reales_segmento_df)
                    if not indices_validos_en_predicciones.empty:
                        if (predicciones_tipo_M2_series.loc[indices_validos_en_predicciones] == nombre_clase_sensor_atascado).any():
                            fue_clasificado_como_atasco = True
                elif predicciones_tipo_M2_series.empty: # Si no hay predicciones, no se puede clasificar como atasco
                    pass # fue_clasificado_como_atasco remains False
                else: # Fallback o advertencia si los tipos de índice no coinciden y las predicciones no están vacías
                    print(f"Advertencia (marcar_segmentos): Incompatibilidad de tipos de índice entre df y predicciones para la columna '{columna_nombre}'. Segmento {indices_df[i] if i < len(indices_df) else 'END'} no se considerará atasco.")


        if fue_clasificado_como_atasco:
            # Esta sección se ejecuta solo si fue_clasificado_como_atasco es True
            idx_inicio_segmento_df = indices_reales_segmento_df[0]
            idx_fin_segmento_df = indices_reales_segmento_df[-1]
            if columna_nombre in df_a_marcar.columns:
                df_a_marcar.loc[idx_inicio_segmento_df:idx_fin_segmento_df, columna_nombre] = np.nan
                segmentos_nanificados_en_col.append((idx_inicio_segmento_df, idx_fin_segmento_df))
                nan_count_col += 1
            else:
                print(f"Advertencia (marcar_segmentos): Columna '{columna_nombre}' no encontrada. No se marcará NaN.")
        
        if fue_clasificado_como_atasco and duracion_actual_atasco >= min_duracion_atasco :
            i = j
        elif duracion_actual_atasco > 0 : # Avanzar incluso si no fue clasificado o no cumplió min_duracion_atasco pero hubo segmento
            i = j
        else: # Caso de valor aislado o fin de datos no procesados como segmento
            i += 1
            
    return nan_count_col, segmentos_nanificados_en_col

# --- Función para corregir con interpolación y evaluar---
def corregir_sensor_atascado_condicional_eval(df_a_corregir, columna_nombre,
                                                predicciones_tipo_M2_series,
                                                nombre_clase_sensor_atascado="Sensor Atascado",
                                                min_duracion_atasco=3):
    valores = df_a_corregir[columna_nombre].values
    indices_df = df_a_corregir.index
    n = len(valores)
    segmentos_corregidos_count = 0
    segmentos_marcados_para_interpolacion = []
    i = 0
    while i < n:
        if pd.isna(valores[i]):
            i += 1
            continue
        j = i
        while j < n and pd.notna(valores[j]) and valores[j] == valores[i]:
            j += 1
        duracion_actual_atasco = j - i
        fue_clasificado_como_atasco = False
        if duracion_actual_atasco >= min_duracion_atasco:
            indices_reales_segmento_df = indices_df[i:j]
            if not indices_reales_segmento_df.empty:
                # Asegurar compatibilidad de índices antes de la intersección
                if not predicciones_tipo_M2_series.empty and isinstance(predicciones_tipo_M2_series.index, type(indices_df)):
                    indices_validos_en_predicciones = predicciones_tipo_M2_series.index.intersection(indices_reales_segmento_df)
                    if not indices_validos_en_predicciones.empty:
                        if (predicciones_tipo_M2_series.loc[indices_validos_en_predicciones] == nombre_clase_sensor_atascado).any():
                            fue_clasificado_como_atasco = True
                elif predicciones_tipo_M2_series.empty:
                    pass
                else:
                    print(f"Advertencia (corregir_sensor_atascado): Incompatibilidad de tipos de índice para la columna '{columna_nombre}'. Segmento {indices_df[i] if i < len(indices_df) else 'END'} no se considerará atasco.")


        if fue_clasificado_como_atasco:
            idx_inicio_segmento_df = indices_reales_segmento_df[0]
            idx_fin_segmento_df = indices_reales_segmento_df[-1]
            if columna_nombre in df_a_corregir.columns:
                df_a_corregir.loc[idx_inicio_segmento_df:idx_fin_segmento_df, columna_nombre] = np.nan
                segmentos_marcados_para_interpolacion.append((idx_inicio_segmento_df, idx_fin_segmento_df))
                segmentos_corregidos_count += 1
            else:
                print(f"Advertencia (corregir_sensor_atascado): Columna '{columna_nombre}' no encontrada. No se marcará NaN.")
        
        if fue_clasificado_como_atasco and duracion_actual_atasco >= min_duracion_atasco:
            i = j
        elif duracion_actual_atasco > 0:
            i = j
        else:
            i += 1
            
    if segmentos_corregidos_count > 0:
        if columna_nombre in df_a_corregir.columns:
            df_a_corregir[columna_nombre] = df_a_corregir[columna_nombre].interpolate(method='linear', limit_direction='both')
        else:
            print(f"Advertencia (interpolación): Columna '{columna_nombre}' no encontrada para interpolar.")
    return segmentos_corregidos_count, segmentos_marcados_para_interpolacion

# --- Función de correción dinamicos ---
def corregir_dinamicos_con_estacional_simple(
    df_a_corregir,
    df_historico_referencia,
    columna_a_corregir,
    start_idx_anomalia,
    end_idx_anomalia,
    columna_hora,
    ventana_dias_previos=7,
    metodo_agregacion='median',
    min_valores_historicos=1
):
    if not isinstance(df_a_corregir.index, pd.DatetimeIndex):
        print(f" ALERTA CRÍTICA en 'corregir_dinamicos_con_estacional_simple': df_a_corregir (columna '{columna_a_corregir}') NO tiene DatetimeIndex. Saltando.")
        return
    if not isinstance(df_historico_referencia.index, pd.DatetimeIndex):
        print(f" ALERTA CRÍTICA en 'corregir_dinamicos_con_estacional_simple': df_historico_referencia (para '{columna_a_corregir}') NO tiene DatetimeIndex. Saltando.")
        return
    if columna_hora not in df_a_corregir.columns:
        print(f" Error en 'corregir_dinamicos_con_estacional_simple': Columna '{columna_hora}' no encontrada en df_a_corregir. Saltando.")
        return
    if columna_hora not in df_historico_referencia.columns:
        print(f" Error en 'corregir_dinamicos_con_estacional_simple': Columna '{columna_hora}' no encontrada en df_historico_referencia. Saltando.")
        return
    if columna_a_corregir not in df_historico_referencia.columns:
        print(f" Error en 'corregir_dinamicos_con_estacional_simple': Columna '{columna_a_corregir}' no en df_historico_referencia. Saltando.")
        return

    try:
        segmento_a_corregir_indices = df_a_corregir.loc[start_idx_anomalia:end_idx_anomalia].index
    except KeyError as e:
        print(f" Error de KeyError al intentar localizar segmento {start_idx_anomalia}-{end_idx_anomalia} en df_a_corregir (índice: {df_a_corregir.index.dtype}, tipo inicio: {type(start_idx_anomalia)}, tipo fin: {type(end_idx_anomalia)}). Error: {e}. Saltando.")
        return
    except Exception as e_segment:
        print(f" Error al obtener segmento {start_idx_anomalia}-{end_idx_anomalia} en df_a_corregir: {e_segment}. Saltando.")
        return

    for idx_anomalo in segmento_a_corregir_indices:
        if not isinstance(idx_anomalo, pd.Timestamp):
            continue

        # --- INICIO DE LA MODIFICACIÓN CLAVE PARA HORA_ANOMALA ---
        _hora_anomala_val_potencial_series = df_a_corregir.loc[idx_anomalo, columna_hora]
        if isinstance(_hora_anomala_val_potencial_series, pd.Series):
            if not _hora_anomala_val_potencial_series.empty:
                if _hora_anomala_val_potencial_series.nunique() > 1: # Chequeo opcional por consistencia
                    print(f" Advertencia: Múltiples horas diferentes ({_hora_anomala_val_potencial_series.unique()}) para el mismo timestamp {idx_anomalo} en df_a_corregir. Usando el primero: {_hora_anomala_val_potencial_series.iloc[0]}.")
                hora_anomala = _hora_anomala_val_potencial_series.iloc[0]
            else:
                # print(f"    Advertencia: Serie de hora anómala vacía para {idx_anomalo}. Saltando este punto.")
                continue
        else:
            hora_anomala = _hora_anomala_val_potencial_series

        fecha_anomala_dt_obj = idx_anomalo.date()

        target_past_dates = pd.date_range(end=fecha_anomala_dt_obj - Timedelta(days=1), periods=ventana_dias_previos, freq='D').date

        min_lookup_date = fecha_anomala_dt_obj - Timedelta(days=ventana_dias_previos + 5)
        max_lookup_date = fecha_anomala_dt_obj - Timedelta(days=1)

        df_historico_relevante = df_historico_referencia[
            (df_historico_referencia.index.date >= min_lookup_date) &
            (df_historico_referencia.index.date <= max_lookup_date)
        ]

        if df_historico_relevante.empty:
            continue

        historico_index_dates = pd.Series(df_historico_relevante.index.date, index=df_historico_relevante.index)

        df_filtrado_temporal = df_historico_relevante[
            historico_index_dates.isin(target_past_dates) &
            (df_historico_relevante[columna_hora] == hora_anomala)
        ]

        valores_historicos_para_hora = df_filtrado_temporal[columna_a_corregir].dropna().tolist()

        if len(valores_historicos_para_hora) >= min_valores_historicos:
            if metodo_agregacion == 'mean':
                valor_corregido = np.mean(valores_historicos_para_hora)
            elif metodo_agregacion == 'median':
                valor_corregido = np.median(valores_historicos_para_hora)
            else:
                valor_corregido = np.nan

            if pd.notna(valor_corregido):
                df_a_corregir.loc[idx_anomalo, columna_a_corregir] = valor_corregido

# --- INICIO DEL PROCESO HÍBRIDO ---

df_entrada_para_atascos_nombre = 'df_corregido_faltantes'
var_X_model2_test_nombre = 'X_model2_test'
var_y_pred_model2_nombre = 'y_pred_model2'
var_label_encoder_model2_nombre = 'label_encoder_model2'
var_df_original_nombre = 'df_original'
var_columnas_objetivo_existentes_nombre = 'columnas_objetivo_existentes'
var_columnas_numericas_para_imputacion_nombre = 'columnas_numericas_para_imputacion'
nombre_clase_sensor_atascado_config = "Sensor Atascado"

if (df_entrada_para_atascos_nombre in locals() or df_entrada_para_atascos_nombre in globals()) and \
    (var_X_model2_test_nombre in locals() or var_X_model2_test_nombre in globals()) and \
    (var_y_pred_model2_nombre in locals() or var_y_pred_model2_nombre in globals()) and \
    (var_label_encoder_model2_nombre in locals() or var_label_encoder_model2_nombre in globals()) and \
    (var_df_original_nombre in locals() or var_df_original_nombre in globals()) and \
    (var_columnas_objetivo_existentes_nombre in locals() or var_columnas_objetivo_existentes_nombre in globals()) and \
    (var_columnas_numericas_para_imputacion_nombre in locals() or var_columnas_numericas_para_imputacion_nombre in globals()):

    print("\n===============================================================================")
    print(f"INICIANDO FASE DE CORRECCIÓN HÍBRIDA PARA '{nombre_clase_sensor_atascado_config}'")
    print("===============================================================================")

    df_entrada_correccion_actual_orig = locals().get(df_entrada_para_atascos_nombre, globals().get(df_entrada_para_atascos_nombre)).copy()
    X_model2_test_actual_orig = locals().get(var_X_model2_test_nombre, globals().get(var_X_model2_test_nombre)).copy()
    y_pred_model2_actual = locals().get(var_y_pred_model2_nombre, globals().get(var_y_pred_model2_nombre))
    label_encoder_model2_actual = locals().get(var_label_encoder_model2_nombre, globals().get(var_label_encoder_model2_nombre))
    df_original_eval_actual_orig = locals().get(var_df_original_nombre, globals().get(var_df_original_nombre)).copy()
    columnas_objetivo_actual = locals().get(var_columnas_objetivo_existentes_nombre, globals().get(var_columnas_objetivo_existentes_nombre))

    print("\n--- Fase de Preparación de Índices y Columna 'Hora' ---")

    df_original_eval_actual = None
    df_entrada_correccion_actual = None
    X_model2_test_actual = None

    # 1. df_original_eval_actual
    temp_df_original = df_original_eval_actual_orig.copy()
    if 'Fecha' in temp_df_original.columns:
        print(f"Procesando '{var_df_original_nombre}': Convirtiendo 'Fecha' a DatetimeIndex...")
        try:
            temp_df_original['Fecha'] = pd.to_datetime(temp_df_original['Fecha'])
            temp_df_original.set_index('Fecha', inplace=True)
            print(f"   Índice de '{var_df_original_nombre}' establecido a partir de 'Fecha'.")
        except Exception as e:
            print(f"   Error al convertir 'Fecha' para '{var_df_original_nombre}': {e}")
            
    if isinstance(temp_df_original.index, pd.DatetimeIndex):
        if 'Hora' not in temp_df_original.columns:
            temp_df_original['Hora'] = temp_df_original.index.hour
            print(f"   Columna 'Hora' creada en '{var_df_original_nombre}'.")
        df_original_eval_actual = temp_df_original
        
        if not df_original_eval_actual.index.is_unique:
            dupes_count = df_original_eval_actual.index.duplicated().sum()
            print(f"ADVERTENCIA: El índice de '{var_df_original_nombre}' (df_original_eval_actual) tiene {dupes_count} duplicados. Se eliminarán manteniendo la primera aparición.")
            df_original_eval_actual = df_original_eval_actual[~df_original_eval_actual.index.duplicated(keep='first')]
        if not df_original_eval_actual.index.is_monotonic_increasing: # implies sorted
            print(f"ADVERTENCIA: El índice de '{var_df_original_nombre}' (df_original_eval_actual) no es monotónicamente creciente. Se ordenará.")
            df_original_eval_actual = df_original_eval_actual.sort_index()
            
    else:
        print(f"ALERTA CRÍTICA: '{var_df_original_nombre}' NO tiene DatetimeIndex. La corrección estacional requiere esto.")
        df_original_eval_actual = temp_df_original

    # 2. df_entrada_correccion_actual
    temp_df_input = df_entrada_correccion_actual_orig.copy()
    if 'Fecha' in temp_df_input.columns:
        print(f"Procesando '{df_entrada_para_atascos_nombre}': Convirtiendo 'Fecha' a DatetimeIndex...")
        try:
            temp_df_input['Fecha'] = pd.to_datetime(temp_df_input['Fecha'])
            temp_df_input.set_index('Fecha', inplace=True)
            print(f"   Índice de '{df_entrada_para_atascos_nombre}' establecido a partir de 'Fecha'.")
        except Exception as e:
            print(f"   Error al convertir 'Fecha' para '{df_entrada_para_atascos_nombre}': {e}")

    if isinstance(temp_df_input.index, pd.DatetimeIndex):
        if 'Hora' not in temp_df_input.columns:
            temp_df_input['Hora'] = temp_df_input.index.hour
            print(f"   Columna 'Hora' creada en '{df_entrada_para_atascos_nombre}'.")
        df_entrada_correccion_actual = temp_df_input

        if not df_entrada_correccion_actual.index.is_unique:
            dupes_count_input = df_entrada_correccion_actual.index.duplicated().sum()
            print(f"ADVERTENCIA: El índice de '{df_entrada_para_atascos_nombre}' (df_entrada_correccion_actual) tiene {dupes_count_input} duplicados. Se eliminarán manteniendo la primera aparición.")
            df_entrada_correccion_actual = df_entrada_correccion_actual[~df_entrada_correccion_actual.index.duplicated(keep='first')]
        if not df_entrada_correccion_actual.index.is_monotonic_increasing:
            print(f"ADVERTENCIA: El índice de '{df_entrada_para_atascos_nombre}' (df_entrada_correccion_actual) no es monotónicamente creciente. Se ordenará.")
            df_entrada_correccion_actual = df_entrada_correccion_actual.sort_index()
            
    else:
        print(f"ALERTA CRÍTICA: '{df_entrada_para_atascos_nombre}' NO tiene DatetimeIndex. La corrección estacional requiere esto.")
        df_entrada_correccion_actual = temp_df_input

    # 3. X_model2_test_actual
    X_model2_test_actual = X_model2_test_actual_orig.copy()
    print(f"Procesando '{var_X_model2_test_nombre}':")
    if not isinstance(X_model2_test_actual.index, pd.DatetimeIndex):
        print(f"   Índice de '{var_X_model2_test_nombre}' NO es DatetimeIndex. Intentando mapeo...")
        df_para_mapeo_fechas = df_original_eval_actual_orig.copy()

        if 'Fecha' in df_para_mapeo_fechas.columns:
            df_para_mapeo_fechas['Fecha_dt_temp'] = pd.to_datetime(df_para_mapeo_fechas['Fecha'])

            if X_model2_test_actual_orig.index.isin(df_para_mapeo_fechas.index).all():
                try:
                    fechas_mapeadas = df_para_mapeo_fechas.loc[X_model2_test_actual_orig.index, 'Fecha_dt_temp']
                    if not pd.Series(fechas_mapeadas).isnull().any():
                        X_model2_test_actual.index = pd.DatetimeIndex(fechas_mapeadas)
                        print(f"   Índice de '{var_X_model2_test_nombre}' establecido por mapeo a 'Fecha' de df_original.")
                        if 'Hora' not in X_model2_test_actual.columns and isinstance(X_model2_test_actual.index, pd.DatetimeIndex):
                            X_model2_test_actual['Hora'] = X_model2_test_actual.index.hour
                            print(f"   Columna 'Hora' (re)creada en '{var_X_model2_test_nombre}'.")
                    else:
                        print(f"   FALLO el mapeo para '{var_X_model2_test_nombre}': algunas fechas mapeadas resultaron NaT.")
                except KeyError:
                    print(f"   FALLO el mapeo para '{var_X_model2_test_nombre}': los valores del índice de X_test ({X_model2_test_actual_orig.index.dtype}) no se encontraron como índice en df_original ({df_para_mapeo_fechas.index.dtype}).")
                except Exception as e:
                    print(f"   Error complejo durante el mapeo de índice para '{var_X_model2_test_nombre}': {e}")
            else:
                print(f"   ALERTA: No todos los índices de '{var_X_model2_test_nombre}' se encuentran en el índice de 'df_original_eval_actual_orig'. Mapeo no posible de esta forma.")
                # faltantes_idx = X_model2_test_actual_orig.index[~X_model2_test_actual_orig.index.isin(df_para_mapeo_fechas.index)]
                # print(f"     Ejemplo de índices de X_test no encontrados en df_original_eval_actual_orig: {faltantes_idx[:5].tolist()}...")
        else:
            print(f"   ALERTA: No se puede realizar el mapeo para '{var_X_model2_test_nombre}'. 'df_original_eval_actual_orig' no tiene columna 'Fecha'.")

    if not isinstance(X_model2_test_actual.index, pd.DatetimeIndex):
        print(f"ALERTA FINAL: '{var_X_model2_test_nombre}' AÚN NO tiene DatetimeIndex. La alineación de predicciones y la corrección estacional probablemente fallarán.")

    print("--- Fin de Fase de Preparación de Índices ---")

    if df_entrada_correccion_actual is None:
        print(f"ALERTA CRÍTICA: df_entrada_correccion_actual es None. Usando la copia original df_entrada_correccion_actual_orig.")
        df_entrada_correccion_actual = df_entrada_correccion_actual_orig.copy()

    df_stuck_hybrid_corrected = df_entrada_correccion_actual.copy()

    if 'pred_tipo_anomalia' in df_stuck_hybrid_corrected.columns:
        serie_predicciones_hybrid = df_stuck_hybrid_corrected['pred_tipo_anomalia']
        print("\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como Atascos.")
    else:
        # Intentamos usar las variables de predicción guardadas si existen
        if (y_pred_model2_actual is not None) and (label_encoder_model2_actual is not None) and (X_model2_test_actual is not None) and (X_model2_test_actual.index is not None):
            try:
                serie_pred_temp = pd.Series(
                    label_encoder_model2_actual.inverse_transform(y_pred_model2_actual),
                    index=X_model2_test_actual.index
                )
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_stuck_hybrid_corrected.index)
                common_idx = df_stuck_hybrid_corrected.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print("\n✅ Se usaron las variables 'y_pred_model2_actual' para filtrar Atascos.")
            except Exception as e:
                # Fallback en caso de error
                serie_predicciones_hybrid = pd.Series(nombre_clase_sensor_atascado_config, index=df_stuck_hybrid_corrected.index)
                print(f"\n⚠️ Falló la carga de variables de predicción ({e}). Se usará pura heurística.")
        else:
            # Fallback final: Si no hay forma de saber las predicciones, asumimos todo como Sensor Atascado 
            # (así la función marcar_segmentos_stuck... funcionará usando solo la heurística de repetición)
            serie_predicciones_hybrid = pd.Series(nombre_clase_sensor_atascado_config, index=df_stuck_hybrid_corrected.index)
            print("\n⚠️ No se encontraron predicciones. Se marcarán Atascos usando puramente la heurística de repetición matemática.")

    min_duracion_stuck_hybrid = 5

    columnas_dinamicas_stuck = ['PRAD', 'PRGINT', 'UVENT_cen', 'UVENT_lN']
    columnas_dinamicas_stuck = [
        col for col in columnas_dinamicas_stuck
        if col in df_stuck_hybrid_corrected.columns and col in columnas_objetivo_actual
    ]
    print(f"Columnas dinámicas para corrección de atascos: {columnas_dinamicas_stuck}")

    segmentos_nanificados_dinamicos_eval = []

    print(f"\n--- Marcando como NaN Segmentos Atascados en Columnas Dinámicas (min_duracion={min_duracion_stuck_hybrid}) ---")
    for col_dyn in columnas_dinamicas_stuck:
        if col_dyn not in df_stuck_hybrid_corrected.columns:
            print(f"Advertencia: Columna dinámica '{col_dyn}' no encontrada. Saltando marcado.")
            continue
        if serie_predicciones_hybrid.empty or serie_predicciones_hybrid.isnull().all():
            print(f"Advertencia: 'serie_predicciones_hybrid' está vacía o toda NaN para la columna '{col_dyn}'. Saltando marcado.")
            continue

        num_marcados_col, lista_segmentos_col = marcar_segmentos_stuck_como_nan_condicional(
            df_stuck_hybrid_corrected,
            col_dyn,
            serie_predicciones_hybrid,
            nombre_clase_sensor_atascado=nombre_clase_sensor_atascado_config,
            min_duracion_atasco=min_duracion_stuck_hybrid
        )
        if num_marcados_col > 0:
            print(f"   En columna DINÁMICA '{col_dyn}': {num_marcados_col} segmentos marcados como NaN.")
            for seg_start, seg_end in lista_segmentos_col:
                segmentos_nanificados_dinamicos_eval.append((col_dyn, seg_start, seg_end))

    if segmentos_nanificados_dinamicos_eval:
        print(f"\nValores NaN ANTES de Corrección Estacional para atascos dinámicos (columnas dinámicas objetivo):")
        nan_sum_before_seasonal = df_stuck_hybrid_corrected[columnas_dinamicas_stuck].isnull().sum()
        print(nan_sum_before_seasonal[nan_sum_before_seasonal > 0])

        print("\n--- Corrigiendo Atascos en Columnas Dinámicas con Método Estacional Simple ---")
        ventana_dias_previos_seasonal = 7
        metodo_agregacion_seasonal = 'median'
        min_valores_historicos_seasonal = 1
        columna_hora_nombre = 'Hora'
        
        if df_original_eval_actual is None or not isinstance(df_original_eval_actual.index, pd.DatetimeIndex):
            print("ALERTA CRÍTICA: df_original_eval_actual no es válido para la corrección estacional. Saltando la corrección de dinámicos.")
        else:
            for col_dyn_corregir, seg_start_idx, seg_end_idx in segmentos_nanificados_dinamicos_eval:
                print(f"   Corrigiendo columna dinámica '{col_dyn_corregir}' para el segmento de {seg_start_idx} a {seg_end_idx}...")
                corregir_dinamicos_con_estacional_simple(
                    df_a_corregir=df_stuck_hybrid_corrected,
                    df_historico_referencia=df_original_eval_actual,
                    columna_a_corregir=col_dyn_corregir,
                    start_idx_anomalia=seg_start_idx,
                    end_idx_anomalia=seg_end_idx,
                    columna_hora=columna_hora_nombre,
                    ventana_dias_previos=ventana_dias_previos_seasonal,
                    metodo_agregacion=metodo_agregacion_seasonal,
                    min_valores_historicos=min_valores_historicos_seasonal
                )
        
            print("Corrección estacional simple para columnas dinámicas completada.")
            nan_sum_after_seasonal = df_stuck_hybrid_corrected[columnas_dinamicas_stuck].isnull().sum()
            print(f"NaNs DESPUÉS de corrección estacional (en columnas dinámicas objetivo):")
            print(nan_sum_after_seasonal[nan_sum_after_seasonal > 0])

        # --- Bloque de Evaluación Estacional ---
        all_true_stuck_seasonal_dinamicos_eval = []
        all_imputed_stuck_seasonal_dinamicos_eval = []
        column_stuck_correction_metrics_seasonal_eval = []

        if df_original_eval_actual is None or not isinstance(df_original_eval_actual.index, pd.DatetimeIndex):
            print("ALERTA CRÍTICA: df_original_eval_actual no es válido para la evaluación estacional. Saltando la evaluación.")
        elif segmentos_nanificados_dinamicos_eval:
            print("\n--- Evaluación Detallada de Corrección (Estacional Simple) para Sensores Dinámicos Atascados ---")
            for col_eval, start_idx_eval, end_idx_eval in segmentos_nanificados_dinamicos_eval:
                if not (start_idx_eval in df_original_eval_actual.index and \
                        end_idx_eval in df_original_eval_actual.index and \
                        start_idx_eval in df_stuck_hybrid_corrected.index and \
                        end_idx_eval in df_stuck_hybrid_corrected.index and \
                        col_eval in df_original_eval_actual.columns and \
                        col_eval in df_stuck_hybrid_corrected.columns):
                    # print(f"Advertencia (Evaluación Estacional): Segmento ({col_eval}, {start_idx_eval}-{end_idx_eval}) tiene índices o columna no válidos. Saltando.")
                    continue
                
                # The problematic line:
                true_vals_seg_eval = df_original_eval_actual.loc[start_idx_eval:end_idx_eval, col_eval]
                imputed_vals_seg_eval = df_stuck_hybrid_corrected.loc[start_idx_eval:end_idx_eval, col_eval]
                
                if len(true_vals_seg_eval) == len(imputed_vals_seg_eval) and true_vals_seg_eval.notna().all():
                    valid_imputed_mask = imputed_vals_seg_eval.notna()
                    # if not valid_imputed_mask.all():
                        # print(f"Info (Evaluación Estacional): Valores imputados para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs DESPUÉS de imputación estacional. Excluyendo NaNs de métricas para este segmento.")
                        # pass
                    true_vals_seg_for_metric = true_vals_seg_eval[valid_imputed_mask]
                    imputed_vals_seg_for_metric = imputed_vals_seg_eval[valid_imputed_mask]
                    if imputed_vals_seg_for_metric.empty:
                        # print(f"  Segmento {col_eval} ({start_idx_eval}-{end_idx_eval}) no tiene valores válidos post-imputación para métricas.")
                        continue
                    all_true_stuck_seasonal_dinamicos_eval.extend(true_vals_seg_for_metric.tolist())
                    all_imputed_stuck_seasonal_dinamicos_eval.extend(imputed_vals_seg_for_metric.tolist())
                    rmse_seg_seasonal = np.sqrt(mean_squared_error(true_vals_seg_for_metric, imputed_vals_seg_for_metric))
                    mae_seg_seasonal = mean_absolute_error(true_vals_seg_for_metric, imputed_vals_seg_for_metric)
                    try:
                        start_loc = df_original_eval_actual.index.get_loc(start_idx_eval)
                        end_loc = df_original_eval_actual.index.get_loc(end_idx_eval)
                        if isinstance(start_loc, slice): start_loc = start_loc.start
                        if isinstance(end_loc, slice): end_loc = end_loc.stop -1
                        segmento_filas_str = f"{start_loc}-{end_loc}"
                    except (KeyError, TypeError):
                        segmento_filas_str = f"Idx:{start_idx_eval}-Idx:{end_idx_eval}"
                    column_stuck_correction_metrics_seasonal_eval.append({
                        'Columna': col_eval, 'Segmento_Filas': segmento_filas_str,
                        'Num_Valores_Corregidos': len(true_vals_seg_for_metric),
                        'RMSE_Correccion_Seasonal': rmse_seg_seasonal, 'MAE_Correccion_Seasonal': mae_seg_seasonal,
                        'Media_Real_Segmento': true_vals_seg_for_metric.mean(), 'Media_Imputada_Segmento': imputed_vals_seg_for_metric.mean()
                    })
                elif not true_vals_seg_eval.notna().all():
                    # print(f"Advertencia (Evaluación Estacional): Valores verdaderos para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs en df_original_eval_actual. Saltando este segmento.")
                    pass
            if column_stuck_correction_metrics_seasonal_eval:
                df_stuck_eval_metrics_seasonal = pd.DataFrame(column_stuck_correction_metrics_seasonal_eval)
                resumen_por_columna_seasonal_stuck = df_stuck_eval_metrics_seasonal.groupby('Columna').agg(
                    Total_Segmentos_Corregidos = ('Segmento_Filas', 'count'),
                    Promedio_RMSE_Seasonal = ('RMSE_Correccion_Seasonal', 'mean'),
                    Promedio_MAE_Seasonal = ('MAE_Correccion_Seasonal', 'mean'),
                    Suma_Valores_Corregidos = ('Num_Valores_Corregidos', 'sum')
                ).reset_index()
                print("\nMétricas de Calidad de Corrección (Estacional Simple en Dinámicos) - Resumen por Columna:")
                print(resumen_por_columna_seasonal_stuck.to_string())

                if all_true_stuck_seasonal_dinamicos_eval and all_imputed_stuck_seasonal_dinamicos_eval:
                    valid_true_global = pd.Series(all_true_stuck_seasonal_dinamicos_eval)
                    valid_imputed_global = pd.Series(all_imputed_stuck_seasonal_dinamicos_eval)
                    nan_mask_global = valid_true_global.notna() & valid_imputed_global.notna()
                    valid_true_global_clean = valid_true_global[nan_mask_global]
                    valid_imputed_global_clean = valid_imputed_global[nan_mask_global]
                    if not valid_true_global_clean.empty:
                        global_rmse_seasonal = np.sqrt(mean_squared_error(valid_true_global_clean, valid_imputed_global_clean))
                        global_mae_seasonal = mean_absolute_error(valid_true_global_clean, valid_imputed_global_clean)
                        print(f"\n  RMSE Global para Dinámicos (Corrección Estacional): {global_rmse_seasonal:.3f} (sobre {len(valid_true_global_clean)} puntos)")
                        print(f"  MAE Global para Dinámicos (Corrección Estacional): {global_mae_seasonal:.3f} (sobre {len(valid_true_global_clean)} puntos)")
                        plt.figure(figsize=(8, 8))
                        sample_size_stuck_seasonal = min(len(valid_true_global_clean), 2000)
                        if sample_size_stuck_seasonal > 0:
                            indices_muestreados = np.random.choice(valid_true_global_clean.index, size=sample_size_stuck_seasonal, replace=False)
                            true_sampled_values = valid_true_global_clean.loc[indices_muestreados].values
                            imputed_sampled_values = valid_imputed_global_clean.loc[indices_muestreados].values
                            plt.scatter(
                                true_sampled_values,
                                imputed_sampled_values,
                                alpha=0.5, s=10, label='Corrected Points'
                            )
                        min_val_plot = valid_true_global_clean.min()
                        max_val_plot = valid_true_global_clean.max()
                        if not valid_imputed_global_clean.empty:
                            min_val_plot = min(min_val_plot, valid_imputed_global_clean.min())
                            max_val_plot = max(max_val_plot, valid_imputed_global_clean.max())
                        if pd.notna(min_val_plot) and pd.notna(max_val_plot) and min_val_plot != max_val_plot:
                            plt.plot([min_val_plot, max_val_plot], [min_val_plot, max_val_plot], 'r--', lw=2, label='Ideal')
                        plt.xlabel("True Values")
                        plt.ylabel("Corrected Values")
                        plt.legend()
                        plt.grid(True)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print("\nNo hay suficientes datos válidos globales (después de filtrar NaNs) para calcular RMSE/MAE o graficar para la corrección estacional.")
            else:
                print("\nNo se evaluaron segmentos de 'Sensor Atascado' en dinámicos con Corrección Estacional (lista de métricas vacía).")
        else:
            print("\nNo se encontraron segmentos marcados como NaN para corrección de atasco en dinámicos (lista 'segmentos_nanificados_dinamicos_eval' vacía después del marcado inicial).")
    else:
        print("\nNo se marcaron segmentos en columnas dinámicas para corrección estacional (lista 'segmentos_nanificados_dinamicos_eval' vacía al inicio).")

    # --- Parte 2: Corrección con Interpolación Lineal para Sensores Estables Atascados ---
    columnas_estables_stuck = [
        col for col in columnas_objetivo_actual
        if col not in columnas_dinamicas_stuck and col in df_stuck_hybrid_corrected.columns
    ]
    print(f"\nColumnas estables para corrección de atascos por interpolación: {columnas_estables_stuck}")
    segmentos_interpolados_estables_eval = []

    print(f"\n--- Iniciando Corrección por Interpolación para '{nombre_clase_sensor_atascado_config}' en Sensores Estables (min_duracion={min_duracion_stuck_hybrid}) ---")
    if columnas_estables_stuck:
        for col_stab in columnas_estables_stuck:
            if col_stab not in df_stuck_hybrid_corrected.columns:
                print(f"Advertencia: Columna estable '{col_stab}' no encontrada en df_stuck_hybrid_corrected. Saltando interpolación.")
                continue
            if serie_predicciones_hybrid.empty or serie_predicciones_hybrid.isnull().all():
                print(f"Advertencia: 'serie_predicciones_hybrid' está vacía o toda NaN para columna estable '{col_stab}'. Saltando interpolación.")
                continue

            temp_serie_predicciones_alineada_stable = pd.Series(dtype=object)
            if isinstance(df_stuck_hybrid_corrected.index, pd.DatetimeIndex) and \
                isinstance(serie_predicciones_hybrid.index, pd.DatetimeIndex):
                if df_stuck_hybrid_corrected.index.equals(serie_predicciones_hybrid.index):
                    temp_serie_predicciones_alineada_stable = serie_predicciones_hybrid
                else:
                    common_idx_pred_stable = df_stuck_hybrid_corrected.index.intersection(serie_predicciones_hybrid.index)
                    if not common_idx_pred_stable.empty:
                        temp_serie_predicciones_alineada_stable = serie_predicciones_hybrid.loc[common_idx_pred_stable].reindex(df_stuck_hybrid_corrected.index)
                    else:
                        print(f"Advertencia: No hay índices Datetime comunes para alinear 'serie_predicciones_hybrid' con 'df_stuck_hybrid_corrected' para la columna estable '{col_stab}'. Saltando interpolación.")
                        continue
            else:
                print(f"Advertencia: Índices no son ambos DatetimeIndex para alinear 'serie_predicciones_hybrid' con 'df_stuck_hybrid_corrected' para la columna estable '{col_stab}'. Saltando interpolación.")
                continue

            if temp_serie_predicciones_alineada_stable.isnull().all():
                print(f"Advertencia: 'serie_predicciones_hybrid' (después de intentar alinear) está toda NaN para la columna estable '{col_stab}'. Saltando interpolación.")
                continue

            num_corregidos_col, lista_segmentos_col = corregir_sensor_atascado_condicional_eval(
                df_stuck_hybrid_corrected,
                col_stab,
                temp_serie_predicciones_alineada_stable,
                nombre_clase_sensor_atascado=nombre_clase_sensor_atascado_config,
                min_duracion_atasco=min_duracion_stuck_hybrid
            )
            if num_corregidos_col > 0:
                print(f"   En columna ESTABLE '{col_stab}': {num_corregidos_col} segmentos marcados como NaN e interpolados.")
                for seg_start, seg_end in lista_segmentos_col:
                    segmentos_interpolados_estables_eval.append((col_stab, seg_start, seg_end))

        all_true_stuck_stable_eval = []
        all_interpolated_stuck_stable_eval = []
        column_stuck_correction_metrics_stable_eval = []
        
        if df_original_eval_actual is None or not isinstance(df_original_eval_actual.index, pd.DatetimeIndex):
            print("ALERTA CRÍTICA: df_original_eval_actual no es válido para la evaluación de interpolación. Saltando la evaluación.")
        elif segmentos_interpolados_estables_eval:
            print("\n--- Evaluación Detallada de Corrección (Interpolación Lineal) para Sensores Estables Atascados ---")
            for col_eval, start_idx_eval, end_idx_eval in segmentos_interpolados_estables_eval:
                if not (start_idx_eval in df_original_eval_actual.index and \
                        end_idx_eval in df_original_eval_actual.index and \
                        start_idx_eval in df_stuck_hybrid_corrected.index and \
                        end_idx_eval in df_stuck_hybrid_corrected.index and \
                        col_eval in df_original_eval_actual.columns and \
                        col_eval in df_stuck_hybrid_corrected.columns):
                    print(f"Advertencia (Evaluación Interpolación): Segmento ({col_eval}, {start_idx_eval}-{end_idx_eval}) tiene índices o columna no válidos. Saltando.")
                    continue
                true_vals_seg_eval = df_original_eval_actual.loc[start_idx_eval:end_idx_eval, col_eval]
                interpolated_vals_seg_eval = df_stuck_hybrid_corrected.loc[start_idx_eval:end_idx_eval, col_eval]
                if len(true_vals_seg_eval) == len(interpolated_vals_seg_eval) and true_vals_seg_eval.notna().all():
                    valid_interp_mask = interpolated_vals_seg_eval.notna()
                    # if not valid_interp_mask.all():
                        # print(f"Info (Evaluación Interpolación): Valores para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs DESPUÉS de interpolación. Excluyendo NaNs de métricas.")
                    true_vals_seg_for_metric_stable = true_vals_seg_eval[valid_interp_mask]
                    interpolated_vals_seg_for_metric_stable = interpolated_vals_seg_eval[valid_interp_mask]
                    if interpolated_vals_seg_for_metric_stable.empty:
                        print(f"  Segmento {col_eval} ({start_idx_eval}-{end_idx_eval}) no tiene valores válidos post-interpolación para métricas.")
                        continue
                    all_true_stuck_stable_eval.extend(true_vals_seg_for_metric_stable.tolist())
                    all_interpolated_stuck_stable_eval.extend(interpolated_vals_seg_for_metric_stable.tolist())
                    rmse_seg_stable = np.sqrt(mean_squared_error(true_vals_seg_for_metric_stable, interpolated_vals_seg_for_metric_stable))
                    mae_seg_stable = mean_absolute_error(true_vals_seg_for_metric_stable, interpolated_vals_seg_for_metric_stable)
                    try:
                        start_loc = df_original_eval_actual.index.get_loc(start_idx_eval)
                        end_loc = df_original_eval_actual.index.get_loc(end_idx_eval)
                        if isinstance(start_loc, slice): start_loc = start_loc.start
                        if isinstance(end_loc, slice): end_loc = end_loc.stop -1
                        segmento_filas_str = f"{start_loc}-{end_loc}"
                    except (KeyError, TypeError):
                        segmento_filas_str = f"Idx:{start_idx_eval}-Idx:{end_idx_eval}"
                    column_stuck_correction_metrics_stable_eval.append({
                        'Columna': col_eval,
                        'Segmento_Filas': segmento_filas_str,
                        'Num_Valores_Corregidos': len(true_vals_seg_for_metric_stable),
                        'RMSE_Correccion_Interp': rmse_seg_stable,
                        'MAE_Correccion_Interp': mae_seg_stable,
                        'Media_Real_Segmento': true_vals_seg_for_metric_stable.mean(),
                        'Media_Interpolada_Segmento': interpolated_vals_seg_for_metric_stable.mean()
                    })
                elif not true_vals_seg_eval.notna().all():
                    print(f"Advertencia (Evaluación Interpolación): Valores verdaderos para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs en df_original_eval_actual. Saltando.")
            if column_stuck_correction_metrics_stable_eval:
                df_stuck_eval_metrics_stable = pd.DataFrame(column_stuck_correction_metrics_stable_eval)
                resumen_por_columna_stable_stuck = df_stuck_eval_metrics_stable.groupby('Columna').agg(
                    Total_Segmentos_Corregidos = ('Segmento_Filas', 'count'),
                    Promedio_RMSE_Interp = ('RMSE_Correccion_Interp', 'mean'),
                    Promedio_MAE_Interp = ('MAE_Correccion_Interp', 'mean'),
                    Suma_Valores_Corregidos = ('Num_Valores_Corregidos', 'sum')
                ).reset_index()
                print("\nMétricas de Calidad de Corrección (Interpolación en Estables) - Resumen por Columna:")
                print(resumen_por_columna_stable_stuck.to_string())
                if all_true_stuck_stable_eval and all_interpolated_stuck_stable_eval:
                    valid_true_stable_global = pd.Series(all_true_stuck_stable_eval)
                    valid_interpolated_stable_global = pd.Series(all_interpolated_stuck_stable_eval)
                    nan_mask_stable_global = valid_true_stable_global.notna() & valid_interpolated_stable_global.notna()
                    valid_true_stable_global_clean = valid_true_stable_global[nan_mask_stable_global]
                    valid_interpolated_stable_global_clean = valid_interpolated_stable_global[nan_mask_stable_global]
                    if not valid_true_stable_global_clean.empty:
                        global_rmse_stable = np.sqrt(mean_squared_error(valid_true_stable_global_clean, valid_interpolated_stable_global_clean))
                        global_mae_stable = mean_absolute_error(valid_true_stable_global_clean, valid_interpolated_stable_global_clean)
                        print(f"\n  RMSE Global para Estables (Interpolación): {global_rmse_stable:.3f} (sobre {len(valid_true_stable_global_clean)} puntos)")
                        print(f"  MAE Global para Estables (Interpolación): {global_mae_stable:.3f} (sobre {len(valid_true_stable_global_clean)} puntos)")
                        plt.figure(figsize=(8, 8))
                        sample_size_stuck_stable = min(len(valid_true_stable_global_clean), 2000)
                        if sample_size_stuck_stable > 0:
                            indices_muestreados_stable = np.random.choice(valid_true_stable_global_clean.index, size=sample_size_stuck_stable, replace=False)
                            true_sampled_stable_values = valid_true_stable_global_clean.loc[indices_muestreados_stable].values
                            interpolated_sampled_stable_values = valid_interpolated_stable_global_clean.loc[indices_muestreados_stable].values
                            plt.scatter(
                                true_sampled_stable_values,
                                interpolated_sampled_stable_values,
                                alpha=0.5, s=10, label='Corrected Points', c='green'
                            )
                        min_val_plot_stable = valid_true_stable_global_clean.min()
                        max_val_plot_stable = valid_true_stable_global_clean.max()
                        if not valid_interpolated_stable_global_clean.empty:
                            min_val_plot_stable = min(min_val_plot_stable, valid_interpolated_stable_global_clean.min())
                            max_val_plot_stable = max(max_val_plot_stable, valid_interpolated_stable_global_clean.max())
                        if pd.notna(min_val_plot_stable) and pd.notna(max_val_plot_stable) and min_val_plot_stable != max_val_plot_stable:
                            plt.plot([min_val_plot_stable, max_val_plot_stable], [min_val_plot_stable, max_val_plot_stable], 'r--', lw=2, label='Ideal')
                        plt.xlabel("True Values")
                        plt.ylabel("Corrected Values")
                        plt.legend()
                        plt.grid(True)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print("\nNo hay suficientes datos válidos globales (después de filtrar NaNs) para calcular RMSE/MAE o graficar para la corrección por interpolación.")
            else:
                print("\nNo se evaluaron segmentos de 'Sensor Atascado' en estables con Interpolación (lista de métricas vacía).")
        else:
            print("\nNo se encontraron segmentos marcados e interpolados para evaluación en sensores estables.")
    else:
        print("\nNo hay columnas de sensores estables para corregir atascos con interpolación o no se encontraron atascos que cumplan criterios.")

    if 'temp_M2_pred_type_hybrid' in df_stuck_hybrid_corrected.columns:
        df_stuck_hybrid_corrected.drop(columns=['temp_M2_pred_type_hybrid'], inplace=True)

    print("\n===============================================================================")
    print(f"FIN DE LA FASE DE CORRECCIÓN HÍBRIDA PARA '{nombre_clase_sensor_atascado_config}'")
    print("===============================================================================")

    print("\nEl DataFrame con las correcciones de 'Datos Faltantes' y 'Sensor Atascado' (usando método estacional para dinámicos) se encuentra en: df_stuck_hybrid_corrected")

else:
    print("Error: DataFrames o variables necesarios para la corrección HÍBRIDA de 'Sensor Atascado' no están disponibles en el entorno global o local.")
    print("Por favor, verifica los siguientes NOMBRES DE VARIABLES en tu notebook y asegúrate de que estén definidos ANTES de ejecutar este bloque:")
    print(f"  - DataFrame de entrada (post-imputación faltantes): '{df_entrada_para_atascos_nombre}'")
    print(f"  - X_test de tu modelo de anomalías: '{var_X_model2_test_nombre}'")
    print(f"  - Predicciones de tu modelo de anomalías: '{var_y_pred_model2_nombre}'")
    print(f"  - LabelEncoder de tu modelo de anomalías: '{var_label_encoder_model2_nombre}'")
    print(f"  - DataFrame original para evaluación: '{var_df_original_nombre}'")
    print(f"  - Lista de columnas objetivo: '{var_columnas_objetivo_existentes_nombre}'")
    print(f"  - Lista de columnas para imputación general (referencia): '{var_columnas_numericas_para_imputacion_nombre}'")

# This part might error if df_stuck_hybrid_corrected was not successfully defined due to issues above
if 'df_stuck_hybrid_corrected' in locals() or 'df_stuck_hybrid_corrected' in globals():
    if columnas_objetivo_actual and all(col in df_stuck_hybrid_corrected.columns for col in columnas_objetivo_actual if isinstance(col, str)): # check if list is not empty and cols exist
        print("\nValores NaN DESPUÉS de la corrección híbrida completa:")
        nans_despues_hibrido = df_stuck_hybrid_corrected[columnas_objetivo_actual].isnull().sum()
        print(nans_despues_hibrido[nans_despues_hibrido > 0])
        # columnas_con_nans_restantes = nans_despues_hibrido[nans_despues_hibrido > 0].index.tolist() # Not used later, commented
    else:
        print("\nNo se pueden mostrar los NaNs después de la corrección: 'columnas_objetivo_actual' no definidas o no válidas para 'df_stuck_hybrid_corrected'.")
else:
    print("\n'df_stuck_hybrid_corrected' no fue definido, no se pueden mostrar los NaNs finales.")

In [ ]:
print("\nValores NaN DESPUÉS de la corrección híbrida completa:")
nans_despues_hibrido = df_stuck_hybrid_corrected[columnas_objetivo_existentes].isnull().sum()
print(nans_despues_hibrido[nans_despues_hibrido > 0])
columnas_con_nans_restantes = nans_despues_hibrido[nans_despues_hibrido > 0].index.tolist()

In [ ]:
current_df_to_clean = df_stuck_hybrid_corrected # o df_ruido_corregido si aplica

if 'iterative_imputer_faltantes' in locals() and iterative_imputer_faltantes is not None:
    # --- PASO FINAL DE IMPUTACIÓN Y EVALUACIÓN ---
    print(f"\n===============================================================================")
    print(f"INICIANDO FASE DE IMPUTACIÓN FINAL Y EVALUACIÓN")
    print(f"===============================================================================")

    # Identificar las columnas que realmente tienen NaNs y que el imputador maneja.
    # 'columnas_numericas_para_imputacion' debe ser la lista de columnas con la que 'iterative_imputer_faltantes' fue originalmente AJUSTADO (fit).
    if not ('columnas_numericas_para_imputacion' in locals() and columnas_numericas_para_imputacion):
        print("Error: 'columnas_numericas_para_imputacion' no está definida o está vacía. No se puede continuar.")
    else:
        # Columnas que el imputador espera
        cols_for_imputer_transform = [col for col in columnas_numericas_para_imputacion if col in current_df_to_clean.columns]
        
        if not cols_for_imputer_transform:
            print(f"Advertencia: Ninguna de las 'columnas_numericas_para_imputacion' se encontró en el DataFrame actual.")
        else:
            nan_check_df = current_df_to_clean[cols_for_imputer_transform]
            nans_before_final_imputation_series = nan_check_df.isnull().sum()
            columnas_con_nans_para_este_paso = nans_before_final_imputation_series[nans_before_final_imputation_series > 0].index.tolist()

            if columnas_con_nans_para_este_paso:
                print(f"\nColumnas con NaNs restantes que serán procesadas por IterativeImputer: {columnas_con_nans_para_este_paso}")

                # --- Preparación para la evaluación ---
                nan_indices_por_columna_final_eval = {}
                for col_eval_final in columnas_con_nans_para_este_paso:
                    # Guardar índices donde la columna es NaN ANTES de la imputación de este paso
                    nan_indices_por_columna_final_eval[col_eval_final] = current_df_to_clean[current_df_to_clean[col_eval_final].isnull()].index
                
                # --- Aplicar la imputación ---
                # .transform() espera TODAS las columnas con las que el imputador fue ajustado (fit), en el mismo orden.
                print(f"\nAplicando IterativeImputer.transform() a {len(cols_for_imputer_transform)} columnas.")
                
                # Prevenir el SettingWithCopyWarning obteniendo los valores y reasignando
                imputed_values_array = iterative_imputer_faltantes.transform(current_df_to_clean[cols_for_imputer_transform])
                imputed_subset_df = pd.DataFrame(imputed_values_array, index=current_df_to_clean.index, columns=cols_for_imputer_transform)
                
                for col_update in cols_for_imputer_transform:
                    current_df_to_clean[col_update] = imputed_subset_df[col_update]

                print("IterativeImputer para imputación final completado.")
                nans_final_count = current_df_to_clean[columnas_con_nans_para_este_paso].isnull().sum().sum()
                print(f"Total NaNs en estas columnas después del paso final: {nans_final_count}")

                # --- Evaluación de esta imputación final ---
                df_original_ref_final = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                if df_original_ref_final is None:
                    # Fallback al df_original global si no se encuentra la versión evaluable
                    df_original_ref_final = locals().get('df_original', globals().get('df_original'))

                if df_original_ref_final is not None and nan_indices_por_columna_final_eval:
                    print("\n--- Evaluación Detallada de la Imputación Final ---")
                    all_true_final_eval = []
                    all_imputed_final_eval = []
                    final_imputation_metrics_eval = []
                    
                    for col_eval_final, nan_indices_final in nan_indices_por_columna_final_eval.items():
                        if nan_indices_final.empty:
                            continue
                        
                        # Intersección para asegurar que los índices existen en ambos DataFrames
                        valid_indices_for_eval = df_original_ref_final.index.intersection(nan_indices_final)
                        
                        if not (col_eval_final in df_original_ref_final.columns and col_eval_final in current_df_to_clean.columns):
                            print(f"Advertencia (Eval Final): Columna '{col_eval_final}' no consistente entre DFs. Saltando.")
                            continue
                        if valid_indices_for_eval.empty:
                            # print(f"Info (Eval Final): No hay índices comunes para '{col_eval_final}' para evaluación. Saltando.")
                            continue

                        true_values_final = df_original_ref_final.loc[valid_indices_for_eval, col_eval_final]
                        imputed_values_final = current_df_to_clean.loc[valid_indices_for_eval, col_eval_final] # Valores DESPUÉS de la imputación

                        valid_comparison_mask_final = true_values_final.notna() & imputed_values_final.notna()
                        true_values_valid_final = true_values_final[valid_comparison_mask_final]
                        imputed_values_valid_final = imputed_values_final[valid_comparison_mask_final]

                        if not true_values_valid_final.empty:
                            all_true_final_eval.extend(true_values_valid_final.tolist())
                            all_imputed_final_eval.extend(imputed_values_valid_final.tolist())

                            rmse_final_col = np.sqrt(mean_squared_error(true_values_valid_final, imputed_values_valid_final))
                            mae_final_col = mean_absolute_error(true_values_valid_final, imputed_values_valid_final)

                            final_imputation_metrics_eval.append({
                                'Columna': col_eval_final,
                                'Num_Valores_Imputados_Eval': len(true_values_valid_final),
                                'RMSE': rmse_final_col,
                                'MAE': mae_final_col,
                                'Media_Real_Original': true_values_valid_final.mean(),
                                'Media_Imputada_Final': imputed_values_valid_final.mean()
                            })
                    
                    if final_imputation_metrics_eval:
                        df_final_imputation_metrics = pd.DataFrame(final_imputation_metrics_eval)
                        print("\nMétricas de Calidad de Imputación Final - Resumen por Columna:")
                        print(df_final_imputation_metrics.to_string())

                        if all_true_final_eval and all_imputed_final_eval:
                            global_rmse_final_imputation = np.sqrt(mean_squared_error(all_true_final_eval, all_imputed_final_eval))
                            global_mae_final_imputation = mean_absolute_error(all_true_final_eval, all_imputed_final_eval)
                            print(f"\nRMSE Global (Imputación Final): {global_rmse_final_imputation:.3f} (sobre {len(all_true_final_eval)} puntos)")
                            print(f"MAE Global (Imputación Final): {global_mae_final_imputation:.3f} (sobre {len(all_true_final_eval)} puntos)")
                            
                            # Gráfico de Dispersión
                            plt.figure(figsize=(8, 8))
                            sample_size_final_eval = min(len(all_true_final_eval), 2000)
                            if sample_size_final_eval > 0:
                                indices_muestreados_final = np.random.choice(len(all_true_final_eval), size=sample_size_final_eval, replace=False)
                                true_sampled_final = np.array(all_true_final_eval)[indices_muestreados_final]
                                imputed_sampled_final = np.array(all_imputed_final_eval)[indices_muestreados_final]
                                
                                plt.scatter(true_sampled_final, imputed_sampled_final, alpha=0.5, s=10, label='Corrected Points', c='purple')
                                min_val_plot_final = np.nanmin(np.concatenate([true_sampled_final, imputed_sampled_final])) # Usar nanmin/nanmax
                                max_val_plot_final = np.nanmax(np.concatenate([true_sampled_final, imputed_sampled_final]))
                                if pd.notna(min_val_plot_final) and pd.notna(max_val_plot_final) and min_val_plot_final < max_val_plot_final:
                                    plt.plot([min_val_plot_final, max_val_plot_final], [min_val_plot_final, max_val_plot_final], 'r--', lw=2, label='Ideal')
                                plt.xlabel("True Values")
                                plt.ylabel("Corrected Values")
                                plt.legend()
                                plt.grid(True)
                                plt.tight_layout()
                                plt.show()
                            else:
                                print("\nNo hay suficientes datos válidos para graficar la imputación final.")
                        else:
                            print("\nNo hay suficientes datos globales para calcular RMSE/MAE para la imputación final.")
                    else:
                        print("\nNo se evaluaron puntos en la imputación final (quizás no había NaNs que coincidieran con valores originales para comparar).")
                else:
                    print("\nNo se pudo realizar la evaluación de la imputación final: 'df_original_eval_actual' no disponible o no se identificaron NaNs para evaluar.")
            else:
                print("\nNo hay columnas con NaNs restantes que requieran imputación en este paso final.")
else:
    print("IterativeImputer ('iterative_imputer_faltantes') no está disponible o no fue definido.")

print(f"\n===============================================================================")
print(f"FIN DE FASE DE IMPUTACIÓN FINAL Y EVALUACIÓN")
print(f"===============================================================================")

### 3. Corrección de "Ruido" Local Basado en Reglas y Vecinos

- Local: Porque solo considera los valores inmediatamente anterior y siguiente al punto ruidoso.
- Basado en Reglas (o Heurístico): Porque la decisión de cómo corregir el valor (promediar o tomar el anterior) se basa en una regla explícita (la comparación de la diferencia de los vecinos con un umbral).
- Vecinos: Los datos de referencia son los puntos adyacentes.

In [ ]:
# --- INICIO DEL PROCESO DE CORRECCIÓN DE RUIDO (MÉTODO PERSONALIZADO) ---

# Definir el nombre de la clase para "Ruido" según el modelo M2
nombre_clase_ruido_config = "Ruido" 
factor_umbral_diferencia = 2.0 # Factor para el umbral de diferencia entre anterior y siguiente

# Verificar que las variables necesarias existen
if 'df_stuck_hybrid_corrected' in locals() or 'df_stuck_hybrid_corrected' in globals():

    df_temp_ruido = locals().get('df_stuck_hybrid_corrected', globals().get('df_stuck_hybrid_corrected'))
    if 'pred_tipo_anomalia' in df_temp_ruido.columns:
        serie_predicciones_hybrid = df_temp_ruido['pred_tipo_anomalia']
        print("\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como Ruido.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_ruido.index)
                common_idx = df_temp_ruido.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print("\n✅ Se usaron las variables del modelo para generar las predicciones de Ruido.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')

    if 'serie_predicciones_hybrid' in locals() or 'serie_predicciones_hybrid' in globals():
        if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
            if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():

                print("\n===============================================================================")
                print(f"INICIANDO FASE DE CORRECCIÓN DE '{nombre_clase_ruido_config}' CON MÉTODO PERSONALIZADO")
                print(f"Factor umbral para diferencia (x std_dev): {factor_umbral_diferencia}")
                print("===============================================================================")

                df_ruido_corregido_custom = df_stuck_hybrid_corrected.copy()
                
                columnas_para_corregir_ruido_custom = [
                    col for col in columnas_objetivo_existentes if col in df_ruido_corregido_custom.columns
                ]
                
                if not columnas_para_corregir_ruido_custom:
                    print("Advertencia: No hay columnas objetivo válidas para la corrección de ruido personalizada. No se realizarán correcciones de ruido.")
                else:
                    print(f"Columnas objetivo para corrección de ruido: {columnas_para_corregir_ruido_custom}")

                    puntos_corregidos_count = {}
                    indices_ruidosos_para_eval_custom = [] 

                    print(f"\n--- Identificando y Corrigiendo '{nombre_clase_ruido_config}' con lógica personalizada ---")
                    
                    lista_puntos_ruidosos_identificados = []
                    print("Fase 1: Identificando todos los puntos ruidosos...")
                    for col_idx_check in columnas_para_corregir_ruido_custom:
                        if col_idx_check not in df_ruido_corregido_custom.columns:
                            continue
                        
                        temp_serie_pred_alineada_custom = pd.Series(dtype=object)
                        if not isinstance(df_ruido_corregido_custom.index, pd.Index) or not isinstance(serie_predicciones_hybrid.index, pd.Index):
                            # print(f"Advertencia (Alineación Ruido Custom): Índices no válidos en df_ruido_corregido_custom o serie_predicciones_hybrid para '{col_idx_check}'.")
                            continue

                        if isinstance(df_ruido_corregido_custom.index, pd.DatetimeIndex) and \
                            isinstance(serie_predicciones_hybrid.index, pd.DatetimeIndex):
                            if df_ruido_corregido_custom.index.equals(serie_predicciones_hybrid.index):
                                temp_serie_pred_alineada_custom = serie_predicciones_hybrid
                            else:
                                common_idx_pred_custom = df_ruido_corregido_custom.index.intersection(serie_predicciones_hybrid.index)
                                if not common_idx_pred_custom.empty:
                                    temp_serie_pred_alineada_custom = serie_predicciones_hybrid.loc[common_idx_pred_custom].reindex(df_ruido_corregido_custom.index)
                                else:
                                    continue
                        elif df_ruido_corregido_custom.index.equals(serie_predicciones_hybrid.index):
                            temp_serie_pred_alineada_custom = serie_predicciones_hybrid
                        else: 
                            try:
                                common_idx_fallback_custom = df_ruido_corregido_custom.index.intersection(serie_predicciones_hybrid.index)
                                if not common_idx_fallback_custom.empty:
                                    temp_serie_pred_alineada_custom = serie_predicciones_hybrid.loc[common_idx_fallback_custom].reindex(df_ruido_corregido_custom.index)
                                else:
                                    continue
                            except Exception: # Ignorar errores de alineación fallback silenciosamente por ahora
                                continue
                        
                        if temp_serie_pred_alineada_custom.empty or temp_serie_pred_alineada_custom.isnull().all():
                            continue

                        indices_ruido_col = temp_serie_pred_alineada_custom[temp_serie_pred_alineada_custom == nombre_clase_ruido_config].index
                        for idx_ruido_item in indices_ruido_col:
                            if idx_ruido_item in df_ruido_corregido_custom.index: # Asegurar que el índice es válido para el DF que se está corrigiendo
                                lista_puntos_ruidosos_identificados.append((col_idx_check, idx_ruido_item))
                                indices_ruidosos_para_eval_custom.append((col_idx_check, idx_ruido_item)) 
                    
                    print(f"Fase 1: {len(lista_puntos_ruidosos_identificados)} puntos ruidosos identificados en total.")
                    
                    # --- Preparar DataFrame fuente para vecinos con índice único ---
                    df_source_for_neighbors = df_stuck_hybrid_corrected.copy()
                    if df_source_for_neighbors.index.has_duplicates:
                        print("Advertencia (Ruido Custom - Preparación Vecinos): 'df_stuck_hybrid_corrected' (usado para obtener vecinos) tiene índices duplicados. Se tomará la primera aparición.")
                        df_source_for_neighbors = df_source_for_neighbors[~df_source_for_neighbors.index.duplicated(keep='first')]
                    
                    print("Fase 2: Aplicando corrección a los puntos identificados...")
                    for col_actual, idx_ruido in lista_puntos_ruidosos_identificados:
                        puntos_corregidos_count[col_actual] = puntos_corregidos_count.get(col_actual, 0) + 1
                        
                        i_pos = -1 
                        try:
                            # Usar df_source_for_neighbors que tiene índice (esperadamente) único
                            loc_result = df_source_for_neighbors.index.get_loc(idx_ruido)
                            
                            if isinstance(loc_result, int):
                                i_pos = loc_result
                            elif isinstance(loc_result, slice):
                                i_pos = loc_result.start
                                if i_pos is None: i_pos = -1
                            elif isinstance(loc_result, np.ndarray): 
                                if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                else: positions = loc_result
                                if len(positions) > 0: i_pos = positions[0] 
                                else: i_pos = -1
                                    
                            if i_pos == -1: # Si no se pudo determinar una posición única válida
                                df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan # Marcar como NaN
                                continue
                        except KeyError: # Si idx_ruido (de la identificación) no está en df_source_for_neighbors (después de de-duplicar)
                            # print(f"Advertencia (Ruido Custom GetLoc): Índice {idx_ruido} no encontrado en la fuente de vecinos de-duplicada. Saltando.")
                            df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan # Marcar como NaN
                            continue
                        except Exception as e_getloc:
                            print(f"Error inesperado en get_loc para índice {idx_ruido}, columna {col_actual}: {e_getloc}. Saltando.")
                            df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan # Marcar como NaN
                            continue

                        val_prev_original, val_next_original = np.nan, np.nan
                        prev_val_exists, next_val_exists = False, False

                        # Obtener vecinos de df_source_for_neighbors
                        if 0 <= i_pos < len(df_source_for_neighbors.index):
                            if i_pos > 0:
                                idx_prev_ts = df_source_for_neighbors.index[i_pos - 1]
                                # .loc en df_source_for_neighbors (con índice único) debería dar escalar
                                val_prev_original = df_source_for_neighbors.loc[idx_prev_ts, col_actual] 
                                if pd.notna(val_prev_original):
                                    prev_val_exists = True
                            
                            if i_pos < len(df_source_for_neighbors.index) - 1:
                                idx_next_ts = df_source_for_neighbors.index[i_pos + 1]
                                # .loc en df_source_for_neighbors (con índice único) debería dar escalar
                                val_next_original = df_source_for_neighbors.loc[idx_next_ts, col_actual]
                                if pd.notna(val_next_original): 
                                    next_val_exists = True
                        else:
                            df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan
                            continue

                        corrected_value = np.nan 

                        if prev_val_exists and next_val_exists:
                            diff = abs(val_next_original - val_prev_original)
                            std_dev_col = std_devs_originales.get(col_actual, 0) 
                            threshold = factor_umbral_diferencia * std_dev_col if std_dev_col > 0 else float('inf')

                            if diff > threshold :
                                corrected_value = val_prev_original
                            else:
                                corrected_value = (val_prev_original + val_next_original) / 2
                        elif prev_val_exists:
                            corrected_value = val_prev_original
                        elif next_val_exists:
                            corrected_value = val_next_original
                        
                        df_ruido_corregido_custom.loc[idx_ruido, col_actual] = corrected_value

                    for col_rep, count_rep in puntos_corregidos_count.items():
                        if count_rep > 0:
                            print(f"   En columna '{col_rep}': {count_rep} puntos '{nombre_clase_ruido_config}' procesados con método personalizado.")
                    
                    # --- Evaluación de la Corrección de Ruido Personalizada ---
                    df_original_ref_eval_custom = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                    if df_original_ref_eval_custom is None:
                        df_original_ref_eval_custom = locals().get('df_original', globals().get('df_original'))

                    if df_original_ref_eval_custom is not None and indices_ruidosos_para_eval_custom:
                        
                        print("\n--- Evaluación Detallada de Corrección de Ruido (Método Personalizado) ---")
                        
                        all_true_ruido_custom_eval = []
                        all_imputed_ruido_custom_eval = []
                        ruido_custom_metrics_eval = []

                        df_original_ref_eval_custom = df_original_ref_eval_custom.copy()
                        df_ruido_corregido_eval_custom = df_ruido_corregido_custom.copy() 

                        if df_original_ref_eval_custom.index.has_duplicates:
                            df_original_ref_eval_custom = df_original_ref_eval_custom[~df_original_ref_eval_custom.index.duplicated(keep='first')]
                        
                        if df_ruido_corregido_eval_custom.index.has_duplicates:
                            df_ruido_corregido_eval_custom = df_ruido_corregido_eval_custom[~df_ruido_corregido_eval_custom.index.duplicated(keep='first')]

                        eval_data_by_col_custom = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                        for col_eval_c, idx_eval_c in indices_ruidosos_para_eval_custom:
                            if not (idx_eval_c in df_original_ref_eval_custom.index and \
                                    idx_eval_c in df_ruido_corregido_eval_custom.index and \
                                    col_eval_c in df_original_ref_eval_custom.columns and \
                                    col_eval_c in df_ruido_corregido_eval_custom.columns):
                                continue
                            
                            true_val_c = df_original_ref_eval_custom.loc[idx_eval_c, col_eval_c]
                            imputed_val_c = df_ruido_corregido_eval_custom.loc[idx_eval_c, col_eval_c]

                            if pd.notna(true_val_c) and pd.notna(imputed_val_c): 
                                eval_data_by_col_custom[col_eval_c]['true'].append(true_val_c)
                                eval_data_by_col_custom[col_eval_c]['imputed'].append(imputed_val_c)
                                eval_data_by_col_custom[col_eval_c]['indices'].append(idx_eval_c)
                        
                        for col_eval_metric_c, data_c in eval_data_by_col_custom.items():
                            if not data_c['true']: continue
                            true_vals_series_c = pd.Series(data_c['true'])
                            imputed_vals_series_c = pd.Series(data_c['imputed'])
                            
                            all_true_ruido_custom_eval.extend(true_vals_series_c.tolist())
                            all_imputed_ruido_custom_eval.extend(imputed_vals_series_c.tolist())
                            
                            rmse_col_c = np.sqrt(mean_squared_error(true_vals_series_c, imputed_vals_series_c))
                            mae_col_c = mean_absolute_error(true_vals_series_c, imputed_vals_series_c)
                            
                            ruido_custom_metrics_eval.append({
                                'Columna': col_eval_metric_c,
                                'Num_Puntos_Corregidos': len(true_vals_series_c),
                                'RMSE_Correccion_Custom': rmse_col_c,
                                'MAE_Correccion_Custom': mae_col_c,
                                'Media_Real_Original': true_vals_series_c.mean(),
                                'Media_Imputada_Custom': imputed_vals_series_c.mean()
                            })

                        if ruido_custom_metrics_eval:
                            df_ruido_custom_eval_metrics = pd.DataFrame(ruido_custom_metrics_eval)
                            print("\nMétricas de Calidad de Corrección de Ruido (Método Personalizado) - Resumen por Columna:")
                            print(df_ruido_custom_eval_metrics.to_string())

                            if all_true_ruido_custom_eval and all_imputed_ruido_custom_eval:
                                global_rmse_custom = np.sqrt(mean_squared_error(all_true_ruido_custom_eval, all_imputed_ruido_custom_eval))
                                global_mae_custom = mean_absolute_error(all_true_ruido_custom_eval, all_imputed_ruido_custom_eval)
                                print(f"\nRMSE Global (Corrección Ruido Custom): {global_rmse_custom:.3f} (sobre {len(all_true_ruido_custom_eval)} puntos)")
                                print(f"MAE Global (Corrección Ruido Custom): {global_mae_custom:.3f} (sobre {len(all_true_ruido_custom_eval)} puntos)")
                                
                                plt.figure(figsize=(8, 8))
                                sample_size_custom = min(len(all_true_ruido_custom_eval), 2000)
                                if sample_size_custom > 0:
                                    indices_smp_custom = np.random.choice(len(all_true_ruido_custom_eval), size=sample_size_custom, replace=False)
                                    true_smp_c = np.array(all_true_ruido_custom_eval)[indices_smp_custom]
                                    imputed_smp_c = np.array(all_imputed_ruido_custom_eval)[indices_smp_custom]
                                    
                                    plt.scatter(true_smp_c, imputed_smp_c, alpha=0.5, s=10, label='Corrected Points', c='green')
                                    min_val_plot_c = np.nanmin(np.concatenate([true_smp_c, imputed_smp_c]))
                                    max_val_plot_c = np.nanmax(np.concatenate([true_smp_c, imputed_smp_c]))
                                    if pd.notna(min_val_plot_c) and pd.notna(max_val_plot_c) and min_val_plot_c < max_val_plot_c:
                                        plt.plot([min_val_plot_c, max_val_plot_c], [min_val_plot_c, max_val_plot_c], 'r--', lw=2, label='Ideal')
                                    plt.xlabel("True Values")
                                    plt.ylabel(f"Corrected Values")
                                    plt.legend()
                                    plt.grid(True)
                                    plt.tight_layout()
                                    plt.show()
                        else: 
                            print("\nNo se generaron métricas para la corrección de ruido personalizada.")
                    else: 
                        print("\nNo se pudo realizar la evaluación de corrección de ruido personalizada (falta df_original o no hay puntos para evaluar).")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE '{nombre_clase_ruido_config}' (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print("\nEl DataFrame con las correcciones (ruido con método custom) está en: df_ruido_corregido_custom")
            else:
                print("Error: La variable 'columnas_objetivo_existentes' no está definida o está vacía. No se puede proceder con la corrección de ruido.")
        else: 
            print("Error: La variable 'std_devs_originales' no está disponible. No se puede proceder con la corrección de ruido personalizada.")
    else: 
        print("Error: La variable 'serie_predicciones_hybrid' no está disponible. No se puede proceder con la corrección de ruido personalizada.")
else:
    print("Error: El DataFrame 'df_stuck_hybrid_corrected' no está disponible. Asegúrate de que los pasos anteriores se hayan ejecutado correctamente.")

### 4. Corrección de "Valores Fuera de Rango" Local Basado en Reglas y Vecinos

- Local: Porque solo considera los valores inmediatamente anterior y siguiente al punto ruidoso.
- Basado en Reglas (o Heurístico): Porque la decisión de cómo corregir el valor (promediar o tomar el anterior) se basa en una regla explícita (la comparación de la diferencia de los vecinos con un umbral).
- Vecinos: Los datos de referencia son los puntos adyacentes.

In [ ]:
# Factor para el umbral de diferencia entre anterior y siguiente para esta corrección
factor_umbral_diferencia_oor = 2.0 # Se puede usar el mismo que para ruido o uno diferente

nombre_clase_oor_config = "Valores Fuera de Rango"

# Verificar que las variables necesarias existen
if 'df_ruido_corregido_custom' in locals() or 'df_ruido_corregido_custom' in globals():

    df_temp_oor = locals().get('df_ruido_corregido_custom', globals().get('df_ruido_corregido_custom'))
    if 'pred_tipo_anomalia' in df_temp_oor.columns:
        serie_predicciones_hybrid = df_temp_oor['pred_tipo_anomalia']
        print("\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como Valores Fuera de Rango.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_oor.index)
                common_idx = df_temp_oor.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print("\n✅ Se usaron las variables del modelo para generar las predicciones de Fuera de Rango.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')
    else:
        serie_predicciones_hybrid = locals().get('serie_predicciones_hybrid', globals().get('serie_predicciones_hybrid'))

    if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
        if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():
            if 'rangos_validos_sensores' in locals() or 'rangos_validos_sensores' in globals():

                print("\n===============================================================================")
                print(f"INICIANDO FASE DE CORRECCIÓN DE VALORES FUERA DE RANGO CON MÉTODO PERSONALIZADO")
                print(f"Factor umbral para diferencia de vecinos (x std_dev): {factor_umbral_diferencia_oor}")
                print("===============================================================================")

                df_oor_corregido = df_ruido_corregido_custom.copy()
                
                # Columnas donde se aplicará la corrección
                # (aquellas presentes en rangos_validos_sensores, columnas_objetivo_existentes y el df actual)
                columnas_para_corregir_oor = [
                    col for col in rangos_validos_sensores.keys() 
                    if col in df_oor_corregido.columns and col in columnas_objetivo_existentes
                ]

                if not columnas_para_corregir_oor:
                    print("Advertencia: No hay columnas válidas (con rangos definidos y presentes en el DF) para la corrección de valores fuera de rango.")
                else:
                    print(f"Columnas objetivo para corrección de fuera de rango: {columnas_para_corregir_oor}")

                    puntos_oor_corregidos_count = {}
                    indices_fuera_rango_para_eval = [] 

                    print(f"\n--- Identificando y Corrigiendo Valores Fuera de Rango ---")
                    
                    lista_puntos_fuera_rango_identificados = []
                    print("Fase 1: Identificando todos los puntos fuera de rango...")
                    for col_check in columnas_para_corregir_oor:
                        # Obtener límites del diccionario provisto por el usuario
                        if col_check not in rangos_validos_sensores: # Seguridad adicional
                            # print(f"Advertencia: Rangos no definidos para {col_check} en 'rangos_validos_sensores'. Saltando.")
                            continue
                        
                        rango_col = rangos_validos_sensores[col_check]
                        min_limit = rango_col.get('min')
                        max_limit = rango_col.get('max')

                        if min_limit is None or max_limit is None:
                            # print(f"Advertencia: Límites 'min' o 'max' no especificados para {col_check} en 'rangos_validos_sensores'. Saltando.")
                            continue

                        # Alineación de predicciones para cruzar datos
                        temp_serie_pred_alineada_oor = pd.Series(dtype=object)
                        if isinstance(df_oor_corregido.index, pd.Index) and isinstance(serie_predicciones_hybrid.index, pd.Index) and not serie_predicciones_hybrid.empty:
                            if df_oor_corregido.index.equals(serie_predicciones_hybrid.index):
                                temp_serie_pred_alineada_oor = serie_predicciones_hybrid
                            else:
                                common_idx_oor = df_oor_corregido.index.intersection(serie_predicciones_hybrid.index)
                                if not common_idx_oor.empty:
                                    temp_serie_pred_alineada_oor = serie_predicciones_hybrid.loc[common_idx_oor].reindex(df_oor_corregido.index)
                        
                        # Asegurarse de que la columna es numérica
                        if pd.api.types.is_numeric_dtype(df_oor_corregido[col_check]):
                            # Identificar índices donde el valor está fuera de rango
                            # pd.to_numeric para asegurar que la comparación sea numérica y manejar errores de conversión
                            col_data_numeric = pd.to_numeric(df_oor_corregido[col_check], errors='coerce')
                            out_of_range_mask = (col_data_numeric < min_limit) | (col_data_numeric > max_limit)

                            # Intersección de la regla estricta (diccionario) con la predicción (modelo)
                            if not temp_serie_pred_alineada_oor.empty:
                                mask_pred_oor = (temp_serie_pred_alineada_oor == nombre_clase_oor_config)
                                mascara_final = out_of_range_mask & col_data_numeric.notna() & mask_pred_oor
                            else:
                                # Fallback si no hay predicciones: usar solo el diccionario como heurística
                                mascara_final = out_of_range_mask & col_data_numeric.notna()

                            indices_oor_col = df_oor_corregido[mascara_final].index # Solo considerar no-NaNs fuera de rango
                            
                            for idx_oor_item in indices_oor_col:
                                lista_puntos_fuera_rango_identificados.append((col_check, idx_oor_item))
                                indices_fuera_rango_para_eval.append((col_check, idx_oor_item))
                        else:
                            # print(f"Advertencia: Columna {col_check} no es numérica, se omitirá para la detección de fuera de rango.")
                            pass


                    print(f"Fase 1: {len(lista_puntos_fuera_rango_identificados)} puntos fuera de rango identificados en total.")
                    
                    df_source_for_neighbors_oor = df_ruido_corregido_custom.copy() # Vecinos se toman del DF *antes* de esta corrección OOR
                    if df_source_for_neighbors_oor.index.has_duplicates:
                        print("Advertencia (OOR Custom - Preparación Vecinos): DataFrame fuente para vecinos tiene índices duplicados. Se tomará la primera aparición.")
                        df_source_for_neighbors_oor = df_source_for_neighbors_oor[~df_source_for_neighbors_oor.index.duplicated(keep='first')]
                    
                    print("Fase 2: Aplicando corrección a los puntos fuera de rango identificados...")
                    for col_actual, idx_oor in lista_puntos_fuera_rango_identificados:
                        puntos_oor_corregidos_count[col_actual] = puntos_oor_corregidos_count.get(col_actual, 0) + 1
                        
                        i_pos = -1 
                        try:
                            loc_result = df_source_for_neighbors_oor.index.get_loc(idx_oor)
                            
                            if isinstance(loc_result, int): i_pos = loc_result
                            elif isinstance(loc_result, slice):
                                i_pos = loc_result.start
                                if i_pos is None: i_pos = -1
                            elif isinstance(loc_result, np.ndarray): 
                                if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                else: positions = loc_result
                                if len(positions) > 0: i_pos = positions[0] 
                                else: i_pos = -1
                                    
                            if i_pos == -1:
                                df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                                continue
                        except KeyError:
                            df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                            continue
                        except Exception as e_getloc_oor:
                            print(f"Error inesperado en get_loc (OOR) para índice {idx_oor}, columna {col_actual}: {e_getloc_oor}. Saltando.")
                            df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                            continue

                        val_prev_original, val_next_original = np.nan, np.nan
                        prev_val_exists, next_val_exists = False, False

                        if 0 <= i_pos < len(df_source_for_neighbors_oor.index):
                            if i_pos > 0:
                                idx_prev_ts = df_source_for_neighbors_oor.index[i_pos - 1]
                                val_prev_original = df_source_for_neighbors_oor.loc[idx_prev_ts, col_actual]
                                if pd.notna(val_prev_original): prev_val_exists = True
                            
                            if i_pos < len(df_source_for_neighbors_oor.index) - 1:
                                idx_next_ts = df_source_for_neighbors_oor.index[i_pos + 1]
                                val_next_original = df_source_for_neighbors_oor.loc[idx_next_ts, col_actual]
                                if pd.notna(val_next_original): next_val_exists = True
                        else:
                            df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                            continue

                        corrected_value = np.nan 
                        if prev_val_exists and next_val_exists:
                            diff = abs(val_next_original - val_prev_original)
                            std_dev_col = std_devs_originales.get(col_actual, 0) 
                            threshold = factor_umbral_diferencia_oor * std_dev_col if std_dev_col > 0 else float('inf')

                            if diff > threshold : corrected_value = val_prev_original
                            else: corrected_value = (val_prev_original + val_next_original) / 2
                        elif prev_val_exists: corrected_value = val_prev_original
                        elif next_val_exists: corrected_value = val_next_original
                        
                        df_oor_corregido.loc[idx_oor, col_actual] = corrected_value

                    for col_rep, count_rep in puntos_oor_corregidos_count.items():
                        if count_rep > 0:
                            print(f"   En columna '{col_rep}': {count_rep} puntos Fuera de Rango procesados con método personalizado.")
                    
                    # --- Evaluación de la Corrección de Fuera de Rango ---
                    df_original_ref_eval_oor = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                    if df_original_ref_eval_oor is None:
                        df_original_ref_eval_oor = locals().get('df_original', globals().get('df_original'))

                    if df_original_ref_eval_oor is not None and indices_fuera_rango_para_eval:
                        
                        print("\n--- Evaluación Detallada de Corrección de Valores Fuera de Rango (Método Personalizado) ---")
                        all_true_oor_eval = []
                        all_imputed_oor_eval = []
                        oor_custom_metrics_eval = []

                        df_original_ref_eval_oor = df_original_ref_eval_oor.copy()
                        df_oor_corregido_eval = df_oor_corregido.copy()

                        if df_original_ref_eval_oor.index.has_duplicates:
                            df_original_ref_eval_oor = df_original_ref_eval_oor[~df_original_ref_eval_oor.index.duplicated(keep='first')]
                        if df_oor_corregido_eval.index.has_duplicates:
                            df_oor_corregido_eval = df_oor_corregido_eval[~df_oor_corregido_eval.index.duplicated(keep='first')]

                        eval_data_by_col_oor = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                        for col_eval_oor, idx_eval_oor in indices_fuera_rango_para_eval:
                            if not (idx_eval_oor in df_original_ref_eval_oor.index and \
                                    idx_eval_oor in df_oor_corregido_eval.index and \
                                    col_eval_oor in df_original_ref_eval_oor.columns and \
                                    col_eval_oor in df_oor_corregido_eval.columns):
                                continue
                            
                            true_val_oor = df_original_ref_eval_oor.loc[idx_eval_oor, col_eval_oor]
                            imputed_val_oor = df_oor_corregido_eval.loc[idx_eval_oor, col_eval_oor]

                            if pd.notna(true_val_oor) and pd.notna(imputed_val_oor): 
                                eval_data_by_col_oor[col_eval_oor]['true'].append(true_val_oor)
                                eval_data_by_col_oor[col_eval_oor]['imputed'].append(imputed_val_oor)
                                eval_data_by_col_oor[col_eval_oor]['indices'].append(idx_eval_oor)
                        
                        for col_eval_metric_oor, data_oor in eval_data_by_col_oor.items():
                            if not data_oor['true']: continue
                            true_vals_series_oor = pd.Series(data_oor['true'])
                            imputed_vals_series_oor = pd.Series(data_oor['imputed'])
                            
                            all_true_oor_eval.extend(true_vals_series_oor.tolist())
                            all_imputed_oor_eval.extend(imputed_vals_series_oor.tolist())
                            
                            rmse_col_oor = np.sqrt(mean_squared_error(true_vals_series_oor, imputed_vals_series_oor))
                            mae_col_oor = mean_absolute_error(true_vals_series_oor, imputed_vals_series_oor)
                            
                            oor_custom_metrics_eval.append({
                                'Columna': col_eval_metric_oor,
                                'Num_Puntos_Corregidos': len(true_vals_series_oor),
                                'RMSE_Correccion_OOR': rmse_col_oor,
                                'MAE_Correccion_OOR': mae_col_oor,
                                'Media_Real_Original': true_vals_series_oor.mean(),
                                'Media_Imputada_OOR': imputed_vals_series_oor.mean()
                            })

                        if oor_custom_metrics_eval:
                            df_oor_custom_eval_metrics = pd.DataFrame(oor_custom_metrics_eval)
                            print("\nMétricas de Calidad de Corrección Fuera de Rango (Método Personalizado) - Resumen por Columna:")
                            print(df_oor_custom_eval_metrics.to_string())

                            if all_true_oor_eval and all_imputed_oor_eval:
                                global_rmse_oor = np.sqrt(mean_squared_error(all_true_oor_eval, all_imputed_oor_eval))
                                global_mae_oor = mean_absolute_error(all_true_oor_eval, all_imputed_oor_eval)
                                print(f"\nRMSE Global (Corrección Fuera de Rango): {global_rmse_oor:.3f} (sobre {len(all_true_oor_eval)} puntos)")
                                print(f"MAE Global (Corrección Fuera de Rango): {global_mae_oor:.3f} (sobre {len(all_true_oor_eval)} puntos)")
                                
                                plt.figure(figsize=(8,8))
                                sample_size_oor = min(len(all_true_oor_eval), 2000)
                                if sample_size_oor > 0:
                                    indices_smp_oor = np.random.choice(len(all_true_oor_eval), size=sample_size_oor, replace=False)
                                    true_smp_oor = np.array(all_true_oor_eval)[indices_smp_oor]
                                    imputed_smp_oor = np.array(all_imputed_oor_eval)[indices_smp_oor]
                                    
                                    plt.scatter(true_smp_oor, imputed_smp_oor, alpha=0.5, s=10, label='Corrected Points', c='purple')
                                    min_val_plot_oor = np.nanmin(np.concatenate([true_smp_oor, imputed_smp_oor]))
                                    max_val_plot_oor = np.nanmax(np.concatenate([true_smp_oor, imputed_smp_oor]))
                                    if pd.notna(min_val_plot_oor) and pd.notna(max_val_plot_oor) and min_val_plot_oor < max_val_plot_oor:
                                        plt.plot([min_val_plot_oor, max_val_plot_oor], [min_val_plot_oor, max_val_plot_oor], 'r--', lw=2, label='Ideal')
                                    plt.xlabel("True Values")
                                    plt.ylabel("Corrected Values")
                                    plt.legend()
                                    plt.grid(True)
                                    plt.tight_layout()
                                    plt.show()
                        else: 
                            print("\nNo se generaron métricas para la corrección de valores fuera de rango.")
                    else: 
                        print("\nNo se pudo realizar la evaluación de corrección de valores fuera de rango.")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE VALORES FUERA DE RANGO (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print("\nEl DataFrame con las correcciones (fuera de rango con método custom) está en: df_oor_corregido")
                
            else:
                print("Error: El diccionario 'rangos_validos_sensores' no está definido. No se puede proceder.")
        else: 
            print("Error: 'columnas_objetivo_existentes' no está disponible. No se puede proceder.")
    else: 
        print("Error: 'std_devs_originales' no está disponible. No se puede proceder.")
else: 
    print("Error: El DataFrame resultante de la corrección de ruido ('df_ruido_corregido_custom') no está disponible.")

### 5. Corrección de "Desviación de Correlación" Local Basado en Reglas y Vecinos

- Local: Porque solo considera los valores inmediatamente anterior y siguiente al punto ruidoso.
- Basado en Reglas (o Heurístico): Porque la decisión de cómo corregir el valor (promediar o tomar el anterior) se basa en una regla explícita (la comparación de la diferencia de los vecinos con un umbral).
- Vecinos: Los datos de referencia son los puntos adyacentes.

In [ ]:
nombre_clase_corr_dev_config = "Desviación de Correlación"

# Factor para el umbral de diferencia entre anterior y siguiente para esta corrección
factor_umbral_diferencia_corr_dev = 2.0 

# Verificar que las variables necesarias existen
# El DataFrame de entrada es df_oor_corregido (resultado de la corrección anterior)
if 'df_oor_corregido' in locals() or 'df_oor_corregido' in globals():

    df_temp_corr = locals().get('df_oor_corregido', globals().get('df_oor_corregido'))
    if 'pred_tipo_anomalia' in df_temp_corr.columns:
        serie_predicciones_hybrid = df_temp_corr['pred_tipo_anomalia']
        print(f"\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como '{nombre_clase_corr_dev_config}'.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_corr.index)
                common_idx = df_temp_corr.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print(f"\n✅ Se usaron las variables del modelo para generar las predicciones de '{nombre_clase_corr_dev_config}'.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')
    else:
        serie_predicciones_hybrid = locals().get('serie_predicciones_hybrid', globals().get('serie_predicciones_hybrid'))
    
        if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
            if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():
                if 'df_original_eval_actual' in locals() or 'df_original_eval_actual' in globals():

                    print("\n===============================================================================")
                    print(f"INICIANDO FASE DE CORRECCIÓN DE '{nombre_clase_corr_dev_config}' CON MÉTODO PERSONALIZADO")
                    print(f"Factor umbral para diferencia (x std_dev): {factor_umbral_diferencia_corr_dev}")
                    print("===============================================================================")

                    df_corr_dev_corregido_custom = df_oor_corregido.copy()
                    
                    columnas_para_corregir_corr_dev_custom = [
                        col for col in columnas_objetivo_existentes if col in df_corr_dev_corregido_custom.columns
                    ]
                    
                    if not columnas_para_corregir_corr_dev_custom:
                        print("Advertencia: No hay columnas objetivo válidas para la corrección de Desviación de Correlación personalizada. No se realizarán correcciones.")
                    else:
                        print(f"Columnas objetivo para corrección de Desviación de Correlación: {columnas_para_corregir_corr_dev_custom}")

                        puntos_corr_dev_corregidos_count = {}
                        indices_corr_dev_para_eval_custom = [] 

                        print(f"\n--- Identificando y Corrigiendo '{nombre_clase_corr_dev_config}' con lógica personalizada ---")
                        
                        lista_puntos_corr_dev_identificados = []
                        print("Fase 1: Identificando todos los puntos con Desviación de Correlación...")
                        for col_idx_check in columnas_para_corregir_corr_dev_custom:
                            if col_idx_check not in df_corr_dev_corregido_custom.columns:
                                continue
                            
                            # Alineación de la serie de predicciones con el DataFrame actual
                            temp_serie_pred_alineada_corr_dev = pd.Series(dtype=object)
                            current_df_index = df_corr_dev_corregido_custom.index
                            predictions_index = serie_predicciones_hybrid.index

                            if not isinstance(current_df_index, pd.Index) or not isinstance(predictions_index, pd.Index):
                                print(f"Advertencia (Alineación CorrDev Custom): Índices no válidos en df_corr_dev_corregido_custom o serie_predicciones_hybrid para '{col_idx_check}'.")
                                continue

                            if current_df_index.equals(predictions_index):
                                temp_serie_pred_alineada_corr_dev = serie_predicciones_hybrid
                            else:
                                common_idx_pred_corr_dev = current_df_index.intersection(predictions_index)
                                if not common_idx_pred_corr_dev.empty:
                                    temp_serie_pred_alineada_corr_dev = serie_predicciones_hybrid.loc[common_idx_pred_corr_dev].reindex(current_df_index)
                                else:
                                    # print(f"Advertencia (Alineación CorrDev Custom): No hay índices comunes para '{col_idx_check}'.")
                                    continue
                            
                            if temp_serie_pred_alineada_corr_dev.empty or temp_serie_pred_alineada_corr_dev.isnull().all():
                                # print(f"Advertencia (Alineación CorrDev Custom): Serie de predicciones alineada vacía o toda NaN para '{col_idx_check}'.")
                                continue

                            indices_corr_dev_col = temp_serie_pred_alineada_corr_dev[temp_serie_pred_alineada_corr_dev == nombre_clase_corr_dev_config].index
                            for idx_corr_dev_item in indices_corr_dev_col:
                                if idx_corr_dev_item in df_corr_dev_corregido_custom.index: 
                                    lista_puntos_corr_dev_identificados.append((col_idx_check, idx_corr_dev_item))
                                    indices_corr_dev_para_eval_custom.append((col_idx_check, idx_corr_dev_item)) 
                        
                        print(f"Fase 1: {len(lista_puntos_corr_dev_identificados)} puntos con '{nombre_clase_corr_dev_config}' identificados en total.")
                        
                        # DataFrame fuente para vecinos (estado antes de esta corrección específica)
                        df_source_for_neighbors_corr_dev = df_oor_corregido.copy()
                        if df_source_for_neighbors_corr_dev.index.has_duplicates:
                            print("Advertencia (CorrDev Custom - Preparación Vecinos): DataFrame fuente ('df_oor_corregido') para vecinos tiene duplicados. Se tomará la primera aparición.")
                            df_source_for_neighbors_corr_dev = df_source_for_neighbors_corr_dev[~df_source_for_neighbors_corr_dev.index.duplicated(keep='first')]
                        
                        print("Fase 2: Aplicando corrección a los puntos identificados...")
                        for col_actual, idx_corr_dev in lista_puntos_corr_dev_identificados:
                            puntos_corr_dev_corregidos_count[col_actual] = puntos_corr_dev_corregidos_count.get(col_actual, 0) + 1
                            
                            i_pos = -1 
                            try:
                                loc_result = df_source_for_neighbors_corr_dev.index.get_loc(idx_corr_dev)
                                
                                if isinstance(loc_result, int): i_pos = loc_result
                                elif isinstance(loc_result, slice):
                                    i_pos = loc_result.start
                                    if i_pos is None: i_pos = -1
                                elif isinstance(loc_result, np.ndarray): 
                                    if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                    else: positions = loc_result
                                    if len(positions) > 0: i_pos = positions[0] 
                                    else: i_pos = -1
                                        
                                if i_pos == -1:
                                    df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                    continue
                            except KeyError:
                                df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                continue
                            except Exception as e_getloc:
                                print(f"Error inesperado en get_loc (CorrDev) para índice {idx_corr_dev}, columna {col_actual}: {e_getloc}. Saltando.")
                                df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                continue

                            val_prev_original, val_next_original = np.nan, np.nan
                            prev_val_exists, next_val_exists = False, False

                            if 0 <= i_pos < len(df_source_for_neighbors_corr_dev.index):
                                if i_pos > 0:
                                    idx_prev_ts = df_source_for_neighbors_corr_dev.index[i_pos - 1]
                                    val_prev_original = df_source_for_neighbors_corr_dev.loc[idx_prev_ts, col_actual] 
                                    if pd.notna(val_prev_original): prev_val_exists = True
                                
                                if i_pos < len(df_source_for_neighbors_corr_dev.index) - 1:
                                    idx_next_ts = df_source_for_neighbors_corr_dev.index[i_pos + 1]
                                    val_next_original = df_source_for_neighbors_corr_dev.loc[idx_next_ts, col_actual]
                                    if pd.notna(val_next_original): next_val_exists = True
                            else:
                                df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                continue

                            corrected_value = np.nan 
                            if prev_val_exists and next_val_exists:
                                diff = abs(val_next_original - val_prev_original)
                                std_dev_col = std_devs_originales.get(col_actual, 0) 
                                threshold = factor_umbral_diferencia_corr_dev * std_dev_col if std_dev_col > 0 else float('inf')

                                if diff > threshold : 
                                    corrected_value = val_prev_original 
                                else: 
                                    corrected_value = (val_prev_original + val_next_original) / 2
                            elif prev_val_exists: 
                                corrected_value = val_prev_original
                            elif next_val_exists: 
                                corrected_value = val_next_original
                            
                            df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = corrected_value

                        for col_rep, count_rep in puntos_corr_dev_corregidos_count.items():
                            if count_rep > 0:
                                print(f"   En columna '{col_rep}': {count_rep} puntos '{nombre_clase_corr_dev_config}' procesados con método personalizado.")
                        
                        # --- Evaluación de la Corrección de Desviación de Correlación Personalizada ---
                        df_original_eval_actual_ref = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                        if df_original_eval_actual_ref is None:
                            df_original_eval_actual_ref = locals().get('df_original', globals().get('df_original'))


                        if df_original_eval_actual_ref is not None and indices_corr_dev_para_eval_custom:
                            
                            print(f"\n--- Evaluación Detallada de Corrección de '{nombre_clase_corr_dev_config}' (Método Personalizado) ---")
                            
                            all_true_corr_dev_custom_eval = []
                            all_imputed_corr_dev_custom_eval = []
                            corr_dev_custom_metrics_eval = []

                            # Prepare evaluation DataFrames
                            df_original_ref_eval_corr_dev = df_original_eval_actual_ref.copy()
                            if df_original_ref_eval_corr_dev.index.has_duplicates:
                                df_original_ref_eval_corr_dev = df_original_ref_eval_corr_dev[~df_original_ref_eval_corr_dev.index.duplicated(keep='first')]
                            
                            df_corr_dev_corregido_eval_custom = df_corr_dev_corregido_custom.copy() 
                            if df_corr_dev_corregido_eval_custom.index.has_duplicates:
                                df_corr_dev_corregido_eval_custom = df_corr_dev_corregido_eval_custom[~df_corr_dev_corregido_eval_custom.index.duplicated(keep='first')]

                            eval_data_by_col_corr_dev = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                            for col_eval_cd, idx_eval_cd in indices_corr_dev_para_eval_custom:
                                if not (idx_eval_cd in df_original_ref_eval_corr_dev.index and \
                                        idx_eval_cd in df_corr_dev_corregido_eval_custom.index and \
                                        col_eval_cd in df_original_ref_eval_corr_dev.columns and \
                                        col_eval_cd in df_corr_dev_corregido_eval_custom.columns):
                                    continue
                                
                                true_val_cd = df_original_ref_eval_corr_dev.loc[idx_eval_cd, col_eval_cd]
                                imputed_val_cd = df_corr_dev_corregido_eval_custom.loc[idx_eval_cd, col_eval_cd]

                                if pd.notna(true_val_cd) and pd.notna(imputed_val_cd): 
                                    eval_data_by_col_corr_dev[col_eval_cd]['true'].append(true_val_cd)
                                    eval_data_by_col_corr_dev[col_eval_cd]['imputed'].append(imputed_val_cd)
                                    eval_data_by_col_corr_dev[col_eval_cd]['indices'].append(idx_eval_cd)
                            
                            for col_eval_metric_cd, data_cd in eval_data_by_col_corr_dev.items():
                                if not data_cd['true']: continue
                                true_vals_series_cd = pd.Series(data_cd['true'])
                                imputed_vals_series_cd = pd.Series(data_cd['imputed'])
                                
                                all_true_corr_dev_custom_eval.extend(true_vals_series_cd.tolist())
                                all_imputed_corr_dev_custom_eval.extend(imputed_vals_series_cd.tolist())
                                
                                rmse_col_cd = np.sqrt(mean_squared_error(true_vals_series_cd, imputed_vals_series_cd))
                                mae_col_cd = mean_absolute_error(true_vals_series_cd, imputed_vals_series_cd)
                                
                                corr_dev_custom_metrics_eval.append({
                                    'Columna': col_eval_metric_cd,
                                    'Num_Puntos_Corregidos': len(true_vals_series_cd),
                                    'RMSE_Correccion_CorrDev': rmse_col_cd,
                                    'MAE_Correccion_CorrDev': mae_col_cd,
                                    'Media_Real_Original': true_vals_series_cd.mean(),
                                    'Media_Imputada_CorrDev': imputed_vals_series_cd.mean()
                                })

                            if corr_dev_custom_metrics_eval:
                                df_corr_dev_custom_eval_metrics = pd.DataFrame(corr_dev_custom_metrics_eval)
                                print(f"\nMétricas de Calidad de Corrección de '{nombre_clase_corr_dev_config}' (Método Personalizado) - Resumen por Columna:")
                                print(df_corr_dev_custom_eval_metrics.to_string())

                                if all_true_corr_dev_custom_eval and all_imputed_corr_dev_custom_eval:
                                    global_rmse_corr_dev_custom = np.sqrt(mean_squared_error(all_true_corr_dev_custom_eval, all_imputed_corr_dev_custom_eval))
                                    global_mae_corr_dev_custom = mean_absolute_error(all_true_corr_dev_custom_eval, all_imputed_corr_dev_custom_eval)
                                    print(f"\nRMSE Global (Corrección {nombre_clase_corr_dev_config} Custom): {global_rmse_corr_dev_custom:.3f} (sobre {len(all_true_corr_dev_custom_eval)} puntos)")
                                    print(f"MAE Global (Corrección {nombre_clase_corr_dev_config} Custom): {global_mae_corr_dev_custom:.3f} (sobre {len(all_true_corr_dev_custom_eval)} puntos)")
                                    
                                    plt.figure(figsize=(8, 8))
                                    sample_size_corr_dev = min(len(all_true_corr_dev_custom_eval), 2000)
                                    if sample_size_corr_dev > 0:
                                        indices_smp_corr_dev = np.random.choice(len(all_true_corr_dev_custom_eval), size=sample_size_corr_dev, replace=False)
                                        true_smp_cd = np.array(all_true_corr_dev_custom_eval)[indices_smp_corr_dev]
                                        imputed_smp_cd = np.array(all_imputed_corr_dev_custom_eval)[indices_smp_corr_dev]
                                        
                                        plt.scatter(true_smp_cd, imputed_smp_cd, alpha=0.5, s=10, label=f"Corrected Points", c='orange')
                                        min_val_plot_cd = np.nanmin(np.concatenate([true_smp_cd, imputed_smp_cd]))
                                        max_val_plot_cd = np.nanmax(np.concatenate([true_smp_cd, imputed_smp_cd]))
                                        if pd.notna(min_val_plot_cd) and pd.notna(max_val_plot_cd) and min_val_plot_cd < max_val_plot_cd:
                                            plt.plot([min_val_plot_cd, max_val_plot_cd], [min_val_plot_cd, max_val_plot_cd], 'r--', lw=2, label='Ideal')
                                        plt.xlabel("True Values")
                                        plt.ylabel(f"Corrected Values")
                                        plt.legend()
                                        plt.grid(True)
                                        plt.tight_layout()
                                        plt.show()
                                else:
                                    print("\nNo hay suficientes datos globales para calcular RMSE/MAE para la corrección de Desviación de Correlación.")
                            else:
                                print(f"\nNo se generaron métricas para la corrección de '{nombre_clase_corr_dev_config}'.")
                        else:
                            print(f"\nNo se pudo realizar la evaluación de corrección de '{nombre_clase_corr_dev_config}' (df_original_eval_actual no disponible o no se identificaron puntos para evaluar).")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE '{nombre_clase_corr_dev_config}' (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print(f"\nEl DataFrame con las correcciones ('{nombre_clase_corr_dev_config}' con método custom) está en: df_corr_dev_corregido_custom")
                
                else: 
                    print(f"Error: 'columnas_objetivo_existentes' no está disponible. No se puede proceder con la corrección de '{nombre_clase_corr_dev_config}'.")
            else: 
                print(f"Error: 'std_devs_originales' no está disponible. No se puede proceder con la corrección de '{nombre_clase_corr_dev_config}'.")
        else:
            print(f"Error: El DataFrame 'df_oor_corregido' no está disponible. Asegúrate de que los pasos anteriores se hayan ejecutado correctamente para poder iniciar la corrección de '{nombre_clase_corr_dev_config}'.")

### 5. Corrección de "Anomalia Contextual" Local Basado en Reglas y Vecinos

- Local: Porque solo considera los valores inmediatamente anterior y siguiente al punto ruidoso.
- Basado en Reglas (o Heurístico): Porque la decisión de cómo corregir el valor (promediar o tomar el anterior) se basa en una regla explícita (la comparación de la diferencia de los vecinos con un umbral).
- Vecinos: Los datos de referencia son los puntos adyacentes.

In [ ]:
# --- INICIO DEL PROCESO DE CORRECCIÓN DE "ANOMALÍAS CONTEXTUALES" (MÉTODO PERSONALIZADO) ---

nombre_clase_contextual_config = "Contextual"

# Factor para el umbral de diferencia entre anterior y siguiente para esta corrección
factor_umbral_diferencia_contextual = 2.0 

# Verificar que las variables necesarias existen
# El DataFrame de entrada es df_corr_dev_corregido_custom (resultado de la corrección anterior)
if 'df_corr_dev_corregido_custom' in locals() or 'df_corr_dev_corregido_custom' in globals():
    
    df_temp_ctx = locals().get('df_corr_dev_corregido_custom', globals().get('df_corr_dev_corregido_custom'))
    if 'pred_tipo_anomalia' in df_temp_ctx.columns:
        serie_predicciones_hybrid = df_temp_ctx['pred_tipo_anomalia']
        print(f"\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como '{nombre_clase_contextual_config}'.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_ctx.index)
                common_idx = df_temp_ctx.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print(f"\n✅ Se usaron las variables del modelo para generar las predicciones de '{nombre_clase_contextual_config}'.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')
    else:
        serie_predicciones_hybrid = locals().get('serie_predicciones_hybrid', globals().get('serie_predicciones_hybrid'))

        if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
            if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():

                    print("\n===============================================================================")
                    print(f"INICIANDO FASE DE CORRECCIÓN DE '{nombre_clase_contextual_config}' CON MÉTODO PERSONALIZADO")
                    print(f"Factor umbral para diferencia (x std_dev): {factor_umbral_diferencia_contextual}")
                    print("===============================================================================")

                    df_contextual_corregido_custom = df_corr_dev_corregido_custom.copy()
                    
                    columnas_para_corregir_contextual_custom = [
                        col for col in columnas_objetivo_existentes if col in df_contextual_corregido_custom.columns
                    ]
                    
                    if not columnas_para_corregir_contextual_custom:
                        print(f"Advertencia: No hay columnas objetivo válidas para la corrección de '{nombre_clase_contextual_config}' personalizada.")
                    else:
                        print(f"Columnas objetivo para corrección de '{nombre_clase_contextual_config}': {columnas_para_corregir_contextual_custom}")

                        puntos_contextual_corregidos_count = {}
                        indices_contextual_para_eval_custom = [] 

                        print(f"\n--- Identificando y Corrigiendo '{nombre_clase_contextual_config}' con lógica personalizada ---")
                        
                        lista_puntos_contextual_identificados = []
                        print(f"Fase 1: Identificando todos los puntos con '{nombre_clase_contextual_config}'...")
                        for col_idx_check in columnas_para_corregir_contextual_custom:
                            if col_idx_check not in df_contextual_corregido_custom.columns:
                                continue
                            
                            temp_serie_pred_alineada_contextual = pd.Series(dtype=object)
                            current_df_index_contextual = df_contextual_corregido_custom.index
                            predictions_index_contextual = serie_predicciones_hybrid.index

                            if not isinstance(current_df_index_contextual, pd.Index) or not isinstance(predictions_index_contextual, pd.Index):
                                print(f"Advertencia (Alineación Contextual Custom): Índices no válidos para '{col_idx_check}'.")
                                continue

                            if current_df_index_contextual.equals(predictions_index_contextual):
                                temp_serie_pred_alineada_contextual = serie_predicciones_hybrid
                            else:
                                common_idx_pred_contextual = current_df_index_contextual.intersection(predictions_index_contextual)
                                if not common_idx_pred_contextual.empty:
                                    temp_serie_pred_alineada_contextual = serie_predicciones_hybrid.loc[common_idx_pred_contextual].reindex(current_df_index_contextual)
                                else:
                                    continue
                            
                            if temp_serie_pred_alineada_contextual.empty or temp_serie_pred_alineada_contextual.isnull().all():
                                continue

                            indices_contextual_col = temp_serie_pred_alineada_contextual[temp_serie_pred_alineada_contextual == nombre_clase_contextual_config].index
                            for idx_contextual_item in indices_contextual_col:
                                if idx_contextual_item in df_contextual_corregido_custom.index: 
                                    lista_puntos_contextual_identificados.append((col_idx_check, idx_contextual_item))
                                    indices_contextual_para_eval_custom.append((col_idx_check, idx_contextual_item)) 
                        
                        print(f"Fase 1: {len(lista_puntos_contextual_identificados)} puntos con '{nombre_clase_contextual_config}' identificados en total.")
                        
                        df_source_for_neighbors_contextual = df_corr_dev_corregido_custom.copy()
                        if df_source_for_neighbors_contextual.index.has_duplicates:
                            print(f"Advertencia (Contextual Custom - Preparación Vecinos): DataFrame fuente ('df_corr_dev_corregido_custom') para vecinos tiene duplicados. Se tomará la primera aparición.")
                            df_source_for_neighbors_contextual = df_source_for_neighbors_contextual[~df_source_for_neighbors_contextual.index.duplicated(keep='first')]
                        
                        print("Fase 2: Aplicando corrección a los puntos identificados...")
                        for col_actual, idx_contextual in lista_puntos_contextual_identificados:
                            puntos_contextual_corregidos_count[col_actual] = puntos_contextual_corregidos_count.get(col_actual, 0) + 1
                            
                            i_pos = -1 
                            try:
                                loc_result = df_source_for_neighbors_contextual.index.get_loc(idx_contextual)
                                if isinstance(loc_result, int): i_pos = loc_result
                                elif isinstance(loc_result, slice):
                                    i_pos = loc_result.start
                                    if i_pos is None: i_pos = -1
                                elif isinstance(loc_result, np.ndarray): 
                                    if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                    else: positions = loc_result
                                    if len(positions) > 0: i_pos = positions[0] 
                                    else: i_pos = -1
                                if i_pos == -1:
                                    df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                    continue
                            except KeyError:
                                df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                continue
                            except Exception as e_getloc:
                                print(f"Error inesperado en get_loc (Contextual) para índice {idx_contextual}, columna {col_actual}: {e_getloc}. Saltando.")
                                df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                continue

                            val_prev_original, val_next_original = np.nan, np.nan
                            prev_val_exists, next_val_exists = False, False

                            if 0 <= i_pos < len(df_source_for_neighbors_contextual.index):
                                if i_pos > 0:
                                    idx_prev_ts = df_source_for_neighbors_contextual.index[i_pos - 1]
                                    val_prev_original = df_source_for_neighbors_contextual.loc[idx_prev_ts, col_actual] 
                                    if pd.notna(val_prev_original): prev_val_exists = True
                                if i_pos < len(df_source_for_neighbors_contextual.index) - 1:
                                    idx_next_ts = df_source_for_neighbors_contextual.index[i_pos + 1]
                                    val_next_original = df_source_for_neighbors_contextual.loc[idx_next_ts, col_actual]
                                    if pd.notna(val_next_original): next_val_exists = True
                            else:
                                df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                continue

                            corrected_value = np.nan 
                            if prev_val_exists and next_val_exists:
                                diff = abs(val_next_original - val_prev_original)
                                std_dev_col = std_devs_originales.get(col_actual, 0) 
                                threshold = factor_umbral_diferencia_contextual * std_dev_col if std_dev_col > 0 else float('inf')
                                if diff > threshold : corrected_value = val_prev_original 
                                else: corrected_value = (val_prev_original + val_next_original) / 2
                            elif prev_val_exists: corrected_value = val_prev_original
                            elif next_val_exists: corrected_value = val_next_original
                            
                            df_contextual_corregido_custom.loc[idx_contextual, col_actual] = corrected_value

                        for col_rep, count_rep in puntos_contextual_corregidos_count.items():
                            if count_rep > 0:
                                print(f"   En columna '{col_rep}': {count_rep} puntos '{nombre_clase_contextual_config}' procesados con método personalizado.")
                        
                        # --- Evaluación de la Corrección de Anomalías Contextuales Personalizada ---
                        df_original_eval_actual_ref = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                        if df_original_eval_actual_ref is None:
                            df_original_eval_actual_ref = locals().get('df_original', globals().get('df_original'))

                        if df_original_eval_actual_ref is not None and indices_contextual_para_eval_custom:
                            
                            print(f"\n--- Evaluación Detallada de Corrección de '{nombre_clase_contextual_config}' (Método Personalizado) ---")
                            
                            all_true_contextual_custom_eval = []
                            all_imputed_contextual_custom_eval = []
                            contextual_custom_metrics_eval = []

                            df_original_ref_eval_contextual = df_original_eval_actual_ref.copy()
                            if df_original_ref_eval_contextual.index.has_duplicates:
                                df_original_ref_eval_contextual = df_original_ref_eval_contextual[~df_original_ref_eval_contextual.index.duplicated(keep='first')]
                            
                            df_contextual_corregido_eval_custom = df_contextual_corregido_custom.copy() 
                            if df_contextual_corregido_eval_custom.index.has_duplicates:
                                df_contextual_corregido_eval_custom = df_contextual_corregido_eval_custom[~df_contextual_corregido_eval_custom.index.duplicated(keep='first')]

                            eval_data_by_col_contextual = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                            for col_eval_ctx, idx_eval_ctx in indices_contextual_para_eval_custom:
                                if not (idx_eval_ctx in df_original_ref_eval_contextual.index and \
                                        idx_eval_ctx in df_contextual_corregido_eval_custom.index and \
                                        col_eval_ctx in df_original_ref_eval_contextual.columns and \
                                        col_eval_ctx in df_contextual_corregido_eval_custom.columns):
                                    continue
                                true_val_ctx = df_original_ref_eval_contextual.loc[idx_eval_ctx, col_eval_ctx]
                                imputed_val_ctx = df_contextual_corregido_eval_custom.loc[idx_eval_ctx, col_eval_ctx]
                                if pd.notna(true_val_ctx) and pd.notna(imputed_val_ctx): 
                                    eval_data_by_col_contextual[col_eval_ctx]['true'].append(true_val_ctx)
                                    eval_data_by_col_contextual[col_eval_ctx]['imputed'].append(imputed_val_ctx)
                                    eval_data_by_col_contextual[col_eval_ctx]['indices'].append(idx_eval_ctx)
                            
                            for col_eval_metric_ctx, data_ctx in eval_data_by_col_contextual.items():
                                if not data_ctx['true']: continue
                                true_vals_series_ctx = pd.Series(data_ctx['true'])
                                imputed_vals_series_ctx = pd.Series(data_ctx['imputed'])
                                all_true_contextual_custom_eval.extend(true_vals_series_ctx.tolist())
                                all_imputed_contextual_custom_eval.extend(imputed_vals_series_ctx.tolist())
                                rmse_col_ctx = np.sqrt(mean_squared_error(true_vals_series_ctx, imputed_vals_series_ctx))
                                mae_col_ctx = mean_absolute_error(true_vals_series_ctx, imputed_vals_series_ctx)
                                contextual_custom_metrics_eval.append({
                                    'Columna': col_eval_metric_ctx,
                                    'Num_Puntos_Corregidos': len(true_vals_series_ctx),
                                    'RMSE_Correccion_Contextual': rmse_col_ctx,
                                    'MAE_Correccion_Contextual': mae_col_ctx,
                                    'Media_Real_Original': true_vals_series_ctx.mean(),
                                    'Media_Imputada_Contextual': imputed_vals_series_ctx.mean()
                                })

                            if contextual_custom_metrics_eval:
                                df_contextual_custom_eval_metrics = pd.DataFrame(contextual_custom_metrics_eval)
                                print(f"\nMétricas de Calidad de Corrección de '{nombre_clase_contextual_config}' (Método Personalizado) - Resumen por Columna:")
                                print(df_contextual_custom_eval_metrics.to_string())

                                if all_true_contextual_custom_eval and all_imputed_contextual_custom_eval:
                                    global_rmse_contextual_custom = np.sqrt(mean_squared_error(all_true_contextual_custom_eval, all_imputed_contextual_custom_eval))
                                    global_mae_contextual_custom = mean_absolute_error(all_true_contextual_custom_eval, all_imputed_contextual_custom_eval)
                                    print(f"\nRMSE Global (Corrección {nombre_clase_contextual_config} Custom): {global_rmse_contextual_custom:.3f} (sobre {len(all_true_contextual_custom_eval)} puntos)")
                                    print(f"MAE Global (Corrección {nombre_clase_contextual_config} Custom): {global_mae_contextual_custom:.3f} (sobre {len(all_true_contextual_custom_eval)} puntos)")
                                    
                                    plt.figure(figsize=(8, 8))
                                    sample_size_contextual = min(len(all_true_contextual_custom_eval), 2000)
                                    if sample_size_contextual > 0:
                                        indices_smp_contextual = np.random.choice(len(all_true_contextual_custom_eval), size=sample_size_contextual, replace=False)
                                        true_smp_ctx = np.array(all_true_contextual_custom_eval)[indices_smp_contextual]
                                        imputed_smp_ctx = np.array(all_imputed_contextual_custom_eval)[indices_smp_contextual]
                                        plt.scatter(true_smp_ctx, imputed_smp_ctx, alpha=0.5, s=10, label=f"Corrected Points", c='purple') # Changed color for distinction
                                        min_val_plot_ctx = np.nanmin(np.concatenate([true_smp_ctx, imputed_smp_ctx]))
                                        max_val_plot_ctx = np.nanmax(np.concatenate([true_smp_ctx, imputed_smp_ctx]))
                                        if pd.notna(min_val_plot_ctx) and pd.notna(max_val_plot_ctx) and min_val_plot_ctx < max_val_plot_ctx:
                                            plt.plot([min_val_plot_ctx, max_val_plot_ctx], [min_val_plot_ctx, max_val_plot_ctx], 'r--', lw=2, label='Ideal')
                                        plt.xlabel("True Values")
                                        plt.ylabel("Corrected Values")
                                        plt.legend()
                                        plt.grid(True)
                                        plt.tight_layout()
                                        plt.show()
                                else:
                                    print("\nNo hay suficientes datos globales para calcular RMSE/MAE para la corrección de Anomalías Contextuales.")
                            else:
                                print(f"\nNo se generaron métricas para la corrección de '{nombre_clase_contextual_config}'.")
                        else:
                            print(f"\nNo se pudo realizar la evaluación de corrección de '{nombre_clase_contextual_config}' (df_original_eval_actual no disponible o no se identificaron puntos para evaluar).")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE '{nombre_clase_contextual_config}' (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print(f"\nEl DataFrame con las correcciones ('{nombre_clase_contextual_config}' con método custom) está en: df_contextual_corregido_custom")
            else: 
                print(f"Error: 'columnas_objetivo_existentes' no está disponible. No se puede proceder con la corrección de '{nombre_clase_contextual_config}'.")
        else: 
            print(f"Error: 'std_devs_originales' no está disponible. No se puede proceder con la corrección de '{nombre_clase_contextual_config}'.")
else:
    print(f"Error: El DataFrame 'df_corr_dev_corregido_custom' no está disponible. Asegúrate de que los pasos anteriores se hayan ejecutado correctamente para poder iniciar la corrección de '{nombre_clase_contextual_config}'.")

In [ ]:
def mostrar_dataframes_en_memoria(scope=globals()):
    print("📦 Exploración del entorno: DataFrames en memoria\n" + "="*60)
    for nombre, objeto in scope.items():
        if isinstance(objeto, pd.DataFrame):
            print(f"📄 DataFrame '{nombre}' con shape {objeto.shape}:")
            print(f"🧾 Columnas: {list(objeto.columns)}\n")


mostrar_dataframes_en_memoria()

In [ ]:
# --- Lista completa de variables a guardar ---
# Actualizada para incluir todos los DataFrames de la exploración del entorno
# --- Lista completa de variables a guardar ---
# Actualizada para incluir todos los DataFrames de la exploración del entorno
variable_names_to_save = [
    # --- Datos Originales y Modelos ---
    'df_original', 'df_trabajo', 'df_stats', 'correlation_matrix', 'var_value',
    'df_original_modelo', 'df_trabajo_modelo', 'X', 'X_train', 'X_test',
    'X_train_imputed', 'X_test_imputed', 'feature_importance_df',
    'df_eval_detector', 'df_anomalias_reales_en_test', 'detection_summary',
    'X_model2_train', 'X_model2_test', 
    
    # --- Correcciones: Datos Faltantes ---
    'df_para_corregir', 'df_corregido_faltantes', 'df_subset_imputado_faltantes', 
    'df_imputation_metrics_faltantes_eval', 'df_imputation_metrics', 
    
    # --- Evaluación Actual ---
    'df_entrada_correccion_actual_orig', 'X_model2_test_actual_orig', 
    'df_original_eval_actual_orig', 'df_original_eval_actual', 
    'df_entrada_correccion_actual', 'X_model2_test_actual', 'temp_df_original', 
    'temp_df_input', 'df_para_mapeo_fechas', 
    
    # --- Correcciones: Sensor Atascado ---
    'df_stuck_hybrid_corrected', 'df_stuck_eval_metrics_seasonal',
    'resumen_por_columna_seasonal_stuck', 'df_stuck_eval_metrics_stable',
    'resumen_por_columna_stable_stuck', 'current_df_to_clean', 'nan_check_df',
    
    # --- Correcciones: Ruido ---
    'df_temp_ruido', 'df_ruido_corregido_custom', 'df_source_for_neighbors', 
    'df_original_ref_eval_custom', 'df_ruido_corregido_eval_custom', 
    'df_ruido_custom_eval_metrics', 
    
    # --- Correcciones: Fuera de Rango (OOR) ---
    'df_temp_oor', 'df_oor_corregido', 'df_source_for_neighbors_oor', 
    'df_original_ref_eval_oor', 'df_oor_corregido_eval', 'df_oor_custom_eval_metrics', 
    
    # --- Correcciones: Desviación de Correlación ---
    'df_temp_corr', 'df_corr_dev_corregido_custom', 'df_source_for_neighbors_corr_dev', 
    'df_original_eval_actual_ref', 'df_original_ref_eval_corr_dev', 
    'df_corr_dev_corregido_eval_custom', 'df_corr_dev_custom_eval_metrics', 
    
    # --- Correcciones: Anomalía Contextual ---
    'df_temp_ctx', 'df_contextual_corregido_custom', 'df_source_for_neighbors_contextual', 
    'df_original_ref_eval_contextual', 'df_contextual_corregido_eval_custom', 
    'df_contextual_custom_eval_metrics'
]
found_critical_vars = {
    'X': False,
    'X_train': False,
    'X_test': False
}

data_to_save = {}
all_vars_found_or_critical_met = True
non_critical_missing = []

print("📦 Intentando recopilar variables para guardar...")

for var_name in variable_names_to_save:
    if var_name in locals() or var_name in globals(): # Revisar en locals y globals
        # Priorizar locals, luego globals si no está en locals
        data_to_save[var_name] = locals().get(var_name, globals().get(var_name))
        if var_name in found_critical_vars:
            found_critical_vars[var_name] = True
        print(f"   ✅ '{var_name}' encontrada y añadida.")
    else:
        if var_name in found_critical_vars:
            all_vars_found_or_critical_met = False
            print(f"   ❌ ERROR CRÍTICO: '{var_name}' no fue encontrada.")
        else:
            non_critical_missing.append(var_name)
            print(f"   ⚠️ ADVERTENCIA: '{var_name}' no fue encontrada.")

# Verificación de variables críticas
if not all(found_critical_vars.values()):
    all_vars_found_or_critical_met = False # Reafirmar por si alguna crítica no se encontró
    print("⛔ No se guardará el estado: faltan variables críticas.")
    for var_name, found in found_critical_vars.items():
        if not found:
            print(f"   ❌ Variable crítica FALTANTE: '{var_name}'")
elif not data_to_save:
    print("❗ No se encontró ninguna variable para guardar.")
else:
    try:
        file_path = "estado_completo_proceso.pkl"
        with open(file_path, "wb") as f:
            pickle.dump(data_to_save, f)
        print(f"✅ Estado guardado exitosamente en '{file_path}'.")
        print(f"📝 Variables guardadas: {list(data_to_save.keys())}")
        if non_critical_missing:
            print(f"🔍 Variables no críticas no encontradas: {non_critical_missing}")
    except Exception as e:
        print(f"❌ Error al guardar el estado: {e}")

### 6. Evaluación

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import joblib
# import pickle
# from sklearn.model_selection import train_test_split
# from sklearn.impute import SimpleImputer
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import LabelEncoder
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer, KNNImputer
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from pandas import Timedelta
# from collections import defaultdict
# # Configuraciones adicionales para mejorar la visualización (opcional)
# plt.style.use('seaborn-v0_8-whitegrid') # Estilo de gráficos
# pd.set_option('display.max_columns', None) # Para mostrar todas las columnas de un DataFrame
# pd.set_option('display.float_format', lambda x: '%.3f' % x) # Formato para números flotantes

In [ ]:
# --- Cargar el estado completo del proceso guardado ---
file_path = "estado_completo_proceso.pkl"

try:
    with open(file_path, "rb") as f:
        loaded_data = pickle.load(f)

    # Cargar cada variable guardada en el entorno global actual
    for var_name, var_value in loaded_data.items():
        globals()[var_name] = var_value

    print(f"✅ Estado cargado exitosamente desde '{file_path}'.")
    print(f"📦 Variables cargadas: {list(loaded_data.keys())}")

    # Lista de variables clave que deberíamos tener cargadas
    # Actualizada para incluir todos los DataFrames guardados previamente
    expected_vars = [
        # --- Datos Originales y Modelos ---
        'df_original', 'df_trabajo', 'df_stats', 'correlation_matrix', 'var_value',
        'df_original_modelo', 'df_trabajo_modelo', 'X', 'X_train', 'X_test',
        'X_train_imputed', 'X_test_imputed', 'feature_importance_df',
        'df_eval_detector', 'df_anomalias_reales_en_test', 'detection_summary',
        'X_model2_train', 'X_model2_test', 
        
        # --- Correcciones: Datos Faltantes ---
        'df_para_corregir', 'df_corregido_faltantes', 'df_subset_imputado_faltantes', 
        'df_imputation_metrics_faltantes_eval', 'df_imputation_metrics', 
        
        # --- Evaluación Actual ---
        'df_entrada_correccion_actual_orig', 'X_model2_test_actual_orig', 
        'df_original_eval_actual_orig', 'df_original_eval_actual', 
        'df_entrada_correccion_actual', 'X_model2_test_actual', 'temp_df_original', 
        'temp_df_input', 'df_para_mapeo_fechas', 
        
        # --- Correcciones: Sensor Atascado ---
        'df_stuck_hybrid_corrected', 'df_stuck_eval_metrics_seasonal',
        'resumen_por_columna_seasonal_stuck', 'df_stuck_eval_metrics_stable',
        'resumen_por_columna_stable_stuck', 'current_df_to_clean', 'nan_check_df',
        
        # --- Correcciones: Ruido ---
        'df_ruido_corregido_custom', 'df_source_for_neighbors', 'df_original_ref_eval_custom',
        'df_ruido_corregido_eval_custom', 'df_ruido_custom_eval_metrics', 
        
        # --- Correcciones: Fuera de Rango (OOR) ---
        'df_oor_corregido', 'df_source_for_neighbors_oor', 'df_original_ref_eval_oor', 
        'df_oor_corregido_eval', 'df_oor_custom_eval_metrics', 
        
        # --- Correcciones: Desviación de Correlación ---
        'df_corr_dev_corregido_custom', 'df_source_for_neighbors_corr_dev', 
        'df_original_eval_actual_ref', 'df_original_ref_eval_corr_dev', 
        'df_corr_dev_corregido_eval_custom', 'df_corr_dev_custom_eval_metrics', 
        
        # --- Correcciones: Anomalía Contextual ---
        'df_contextual_corregido_custom', 'df_source_for_neighbors_contextual', 
        'df_original_ref_eval_contextual', 'df_contextual_corregido_eval_custom', 
        'df_contextual_custom_eval_metrics',
        
        # --- Secuencia de Correcciones ---
        'df_temp_ruido', 'df_temp_oor', 'df_temp_corr', 'df_temp_ctx'
    ]

    # Verificación de variables importantes
    print("\n🔍 Verificando variables importantes (según expected_vars):")
    all_expected_found = True
    for var in expected_vars:
        if var in globals():
            # Para DataFrames, podríamos imprimir también el shape si quisieramos más detalle
            # if isinstance(globals()[var], pd.DataFrame):
            # print(f"   ✅ '{var}' está disponible. Shape: {globals()[var].shape}")
            # else:
            # print(f"   ✅ '{var}' está disponible.")
            print(f"   ✅ '{var}' está disponible.")
        else:
            print(f"   ⚠️ '{var}' NO se encontró en el entorno.")
            all_expected_found = False
    
    if all_expected_found:
        print("\n✨ Todas las variables esperadas fueron encontradas en el entorno global.")
    else:
        print("\n❗ Algunas variables esperadas NO fueron encontradas. Revisa la lista anterior.")


except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{file_path}'. Asegúrate de que existe y la ruta es correcta.")
except Exception as e:
    print(f"❌ Error al cargar el estado del proceso: {e}")

In [ ]:
print("--- Iniciando Preparación de Índices de DataFrames para Visualización ---")

dataframe_names = ["df_original", "df_trabajo", "df_contextual_corregido_custom"]

for df_name in dataframe_names:
    if df_name not in globals() or globals()[df_name] is None:
        print(f"DataFrame '{df_name}' no encontrado en memoria. Se omitirá su preparación.")
        continue

    # Obtener la referencia al DataFrame global para modificarlo directamente si es necesario
    df_instance = globals()[df_name] 
    original_shape = df_instance.shape # Guardar shape original para referencia
    print(f"Procesando DataFrame '{df_name}' (Shape inicial: {original_shape})...")
    
    try:
        # 1. Asegurar que 'Fecha' sea el índice y de tipo DatetimeIndex
        if isinstance(df_instance.index, pd.DatetimeIndex) and df_instance.index.name == 'Fecha':
            print(f"  '{df_name}' ya tiene 'Fecha' como DatetimeIndex.")
        elif 'Fecha' in df_instance.columns:
            print(f"  Configurando 'Fecha' como DatetimeIndex para '{df_name}'.")
            
            # Trabajar con una copia para manipulaciones de índice complejas si es necesario, pero el objetivo es actualizar el DataFrame global.
            current_df_object = df_instance.copy() # Usar copia para evitar SettingWithCopy en la variable iterada directamente
            
            # Si el índice se llama 'Fecha' pero no es Datetime, resetearlo antes de usar la columna 'Fecha'.
            if current_df_object.index.name == 'Fecha' and not isinstance(current_df_object.index, pd.DatetimeIndex):
                print(f"    Advertencia: '{df_name}' tiene un índice llamado 'Fecha' pero no es Datetime. Se reseteará el índice.")
                current_df_object = current_df_object.reset_index(drop=False) 
            
            # Asegurarse de que la columna 'Fecha' (si existe y se va a usar) sea datetime
            if 'Fecha' in current_df_object.columns:
                current_df_object['Fecha'] = pd.to_datetime(current_df_object['Fecha'])
                current_df_object.set_index('Fecha', inplace=True)
                globals()[df_name] = current_df_object # Actualizar la variable global explícitamente
                df_instance = current_df_object # Actualizar la referencia local para los siguientes pasos
            elif not isinstance(current_df_object.index, pd.DatetimeIndex) : # Si después de todo, no hay índice Fecha
                print(f"  ADVERTENCIA CRÍTICA: '{df_name}' no pudo ser configurado con DatetimeIndex 'Fecha' (columna 'Fecha' no encontrada o no se pudo usar).")
                continue # Saltar más procesamientos para este DF si el índice no es el esperado
            
        elif not isinstance(df_instance.index, pd.DatetimeIndex): # El DF no tiene columna 'Fecha' ni es DatetimeIndex
            print(f"  ADVERTENCIA: '{df_name}' no tiene columna 'Fecha' para convertir ni es DatetimeIndex. La visualización podría fallar.")
            continue

        # Utilizar la referencia global actualizada para los pasos de de-duplicación y ordenamiento
        df_to_process = globals()[df_name] 
        if isinstance(df_to_process.index, pd.DatetimeIndex): # Verificar de nuevo que sea DatetimeIndex
            if not df_to_process.index.is_unique:
                num_dupes_before = df_to_process.index.duplicated().sum()
                print(f"  ADVERTENCIA: El índice de '{df_name}' tiene {num_dupes_before} duplicados. Manteniendo la primera ocurrencia.")
                
                # SOLUCIÓN AL WARNING: Agregamos .copy() para desvincular el slice y evitar el SettingWithCopyWarning
                globals()[df_name] = df_to_process[~df_to_process.index.duplicated(keep='first')].copy()
                print(f"  '{df_name}' después de de-duplicar índice. Nuevo shape: {globals()[df_name].shape}")

            # Re-obtener la referencia por si cambió en el paso de de-duplicación
            df_to_sort = globals()[df_name] 
            if not df_to_sort.index.is_monotonic_increasing:
                print(f"  ADVERTENCIA: El índice de '{df_name}' no es monotónicamente creciente. Ordenando.")
                
                # SOLUCIÓN AL WARNING: Reasignamos el dataframe ordenado en lugar de usar inplace=True
                globals()[df_name] = df_to_sort.sort_index() 
    except Exception as e:
        print(f"  Error al procesar el índice para '{df_name}': {e}")

print("--- Preparación de Índices de DataFrames Completada ---")

# Re-obtener referencias finales a los DataFrames globales después de toda la preparación
df_original = globals().get('df_original')
df_trabajo = globals().get('df_trabajo')
df_contextual_corregido_custom = globals().get('df_contextual_corregido_custom')

# --- DEFINICIÓN SEGURA DE LAS COLUMNAS A VISUALIZAR ---
columnas_objetivo_existentes_global = globals().get('columnas_objetivo_existentes')
if not columnas_objetivo_existentes_global:
    print("ℹ️ 'columnas_objetivo_existentes' no encontrada en memoria. Usando lista por defecto.")
    columnas_objetivo_existentes_global = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV', 'UVENT_cen', 'UVENT_lN', 'XCO2I', 'XHINV', 'XTINV', 'XTS']


# Verificar la preparación final de los DataFrames
df_original_ready = df_original is not None and isinstance(df_original.index, pd.DatetimeIndex)
df_trabajo_ready = df_trabajo is not None and isinstance(df_trabajo.index, pd.DatetimeIndex)
df_contextual_ready = df_contextual_corregido_custom is not None and isinstance(df_contextual_corregido_custom.index, pd.DatetimeIndex)

if not (df_original_ready and df_trabajo_ready and df_contextual_ready and \
        columnas_objetivo_existentes_global and \
        isinstance(columnas_objetivo_existentes_global, list) and \
        len(columnas_objetivo_existentes_global) > 0):
    print("\nError: DataFrames base ('df_original', 'df_trabajo', 'df_contextual_corregido_custom') o 'columnas_objetivo_existentes' no están disponibles, no tienen DatetimeIndex configurado correctamente, o la lista de columnas está vacía/no es válida. No se pueden generar las visualizaciones.")
else:
    import matplotlib.pyplot as plt
    for columna in columnas_objetivo_existentes_global:
        print(f"\n--- Visualizando columna: {columna} ---")
        
        original_col_exists = columna in df_original.columns
        trabajo_col_exists = columna in df_trabajo.columns
        contextual_col_exists = columna in df_contextual_corregido_custom.columns

        # --- Comparación 1: df_original vs df_trabajo ---
        if original_col_exists and trabajo_col_exists:
            # print(f"Comparación 1: df_original vs df_trabajo para {columna}") # Opcional: silenciado para menor ruido
            fig1, axes1 = plt.subplots(2, 1, figsize=(22, 10), sharex=True)
            fig1.suptitle(f"Comparación para: {columna}\n1. Original vs. Con Anomalías Inyectadas", fontsize=16)
            axes1[0].plot(df_original.index, df_original[columna], label="Original (df_original)", alpha=0.7, color="blue")
            axes1[0].set_title(f"{columna} - Datos Originales")
            axes1[0].set_ylabel(columna)
            axes1[0].legend(loc='upper left')
            axes1[0].grid(True, linestyle='--', alpha=0.5)
            axes1[1].plot(df_trabajo.index, df_trabajo[columna], label="Con Anomalías Inyectadas (df_trabajo)", linestyle="dashed", alpha=0.7, color="orange")
            if 'etiqueta_tipo_anomalia' in df_trabajo.columns:
                indices_con_anomalias_inyectadas = df_trabajo[df_trabajo['etiqueta_tipo_anomalia'].notna()].index
                common_indices_trabajo = df_trabajo.index.intersection(indices_con_anomalias_inyectadas)
                if not common_indices_trabajo.empty:
                    axes1[1].scatter(
                        df_trabajo.loc[common_indices_trabajo].index,
                        df_trabajo.loc[common_indices_trabajo, columna],
                        color='red', label='Anomalías Inyectadas', s=50, edgecolors='black', zorder=3 )
            axes1[1].set_title(f"{columna} - Datos con Anomalías Inyectadas")
            axes1[1].set_xlabel("Fecha")
            axes1[1].set_ylabel(columna)
            axes1[1].legend(loc='upper left')
            axes1[1].grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
            plt.show()
        else:
            print(f"  Skipping Comparación 1 para {columna}: Columna no encontrada.")

        # --- Comparación 2: df_original vs df_contextual_corregido_custom ---
        if original_col_exists and contextual_col_exists:
            # print(f"Comparación 2: df_original vs df_contextual_corregido_custom para {columna}")
            fig2, axes2 = plt.subplots(2, 1, figsize=(22, 10), sharex=True)
            fig2.suptitle(f"Comparación para: {columna}\n2. Original vs. Totalmente Corregido", fontsize=16)
            axes2[0].plot(df_original.index, df_original[columna], label="Original (df_original)", alpha=0.7, color="blue")
            axes2[0].set_title(f"{columna} - Datos Originales")
            axes2[0].set_ylabel(columna)
            axes2[0].legend(loc='upper left')
            axes2[0].grid(True, linestyle='--', alpha=0.5)
            axes2[1].plot(df_contextual_corregido_custom.index, df_contextual_corregido_custom[columna], label="Totalmente Corregido", alpha=0.7, color="green")
            axes2[1].set_title(f"{columna} - Datos Corregidos (Todas las etapas)")
            axes2[1].set_xlabel("Fecha")
            axes2[1].set_ylabel(columna)
            axes2[1].legend(loc='upper left')
            axes2[1].grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            print(f"  Skipping Comparación 2 para {columna}: Columna no encontrada.")

        # --- Comparación 3: df_trabajo vs df_contextual_corregido_custom ---
        if trabajo_col_exists and contextual_col_exists:
            # print(f"Comparación 3: df_trabajo vs df_contextual_corregido_custom para {columna}")
            fig3, axes3 = plt.subplots(2, 1, figsize=(22, 10), sharex=True)
            fig3.suptitle(f"Comparación para: {columna}\n3. Con Anomalías Inyectadas vs. Totalmente Corregido", fontsize=16)
            axes3[0].plot(df_trabajo.index, df_trabajo[columna], label="Con Anomalías Inyectadas (df_trabajo)", linestyle="dashed", alpha=0.7, color="orange")
            if 'etiqueta_tipo_anomalia' in df_trabajo.columns:
                indices_con_anomalias_inyectadas_trabajo = df_trabajo[df_trabajo['etiqueta_tipo_anomalia'].notna()].index
                common_indices_trabajo_comp3 = df_trabajo.index.intersection(indices_con_anomalias_inyectadas_trabajo)
                if not common_indices_trabajo_comp3.empty:
                    axes3[0].scatter(
                        df_trabajo.loc[common_indices_trabajo_comp3].index,
                        df_trabajo.loc[common_indices_trabajo_comp3, columna],
                        color='red', label='Anomalías Inyectadas', s=50, edgecolors='black', zorder=3 )
            axes3[0].set_title(f"{columna} - Datos con Anomalías Inyectadas")
            axes3[0].set_ylabel(columna)
            axes3[0].legend(loc='upper left')
            axes3[0].grid(True, linestyle='--', alpha=0.5)
            axes3[1].plot(df_contextual_corregido_custom.index, df_contextual_corregido_custom[columna], label="Totalmente Corregido", alpha=0.7, color="green")
            axes3[1].set_title(f"{columna} - Datos Corregidos (Todas las etapas)")
            axes3[1].set_xlabel("Fecha")
            axes3[1].set_ylabel(columna)
            axes3[1].legend(loc='upper left')
            axes3[1].grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            print(f"  Skipping Comparación 3 para {columna}: Columna no encontrada.")

print("\n--- Fin de la Visualización ---")

In [ ]:
print("--- Iniciando Análisis Numérico de Diferencias ---")

# Re-obtener referencias finales a los DataFrames globales
df_original = globals().get('df_original')
df_trabajo = globals().get('df_trabajo')
df_contextual_corregido_custom = globals().get('df_contextual_corregido_custom')

# --- MODIFICACIÓN IMPORTANTE: Validar y crear variables si no existen ---
columnas_objetivo_existentes_global = globals().get('columnas_objetivo_existentes')
if not columnas_objetivo_existentes_global:
    print("ℹ️ 'columnas_objetivo_existentes' no encontrada en memoria. Usando lista por defecto.")
    columnas_objetivo_existentes_global = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV', 'UVENT_cen', 'UVENT_lN', 'XCO2I', 'XHINV', 'XTINV', 'XTS']

std_devs_originales = globals().get('std_devs_originales')
if std_devs_originales is None and df_original is not None:
    print("ℹ️ 'std_devs_originales' no encontrada en memoria. Calculándola automáticamente desde df_original.")
    # Calculamos la desviación estándar al vuelo para poder hacer la evaluación
    std_devs_originales = df_original[columnas_objetivo_existentes_global].std().to_dict()
# ------------------------------------------------------------------------

# Verificar la disponibilidad y preparación de los DataFrames
df_original_ready = df_original is not None and isinstance(df_original.index, pd.DatetimeIndex) and df_original.index.is_unique and df_original.index.is_monotonic_increasing
df_trabajo_ready = df_trabajo is not None and isinstance(df_trabajo.index, pd.DatetimeIndex) and df_trabajo.index.is_unique and df_trabajo.index.is_monotonic_increasing
df_contextual_ready = df_contextual_corregido_custom is not None and isinstance(df_contextual_corregido_custom.index, pd.DatetimeIndex) and df_contextual_corregido_custom.index.is_unique and df_contextual_corregido_custom.index.is_monotonic_increasing

if not (df_original_ready and df_trabajo_ready and df_contextual_ready and \
        columnas_objetivo_existentes_global and isinstance(columnas_objetivo_existentes_global, list) and len(columnas_objetivo_existentes_global) > 0 and \
        std_devs_originales is not None):
    print("\nError: DataFrames base no están listos (índice no es Datetime, único y ordenado), 'columnas_objetivo_existentes' o 'std_devs_originales' no disponibles. No se puede realizar el análisis numérico.")
else:
    print("\nDataFrames listos para el análisis numérico.")
    
    resumen_diferencias = []

    for columna in columnas_objetivo_existentes_global:
        print(f"\n--- Analizando columna: {columna} ---")

        # Asegurarse de que la columna exista en todos los DFs relevantes para las comparaciones
        if not (columna in df_original.columns and columna in df_trabajo.columns and columna in df_contextual_corregido_custom.columns):
            print(f"  Advertencia: La columna '{columna}' no está presente en todos los DataFrames necesarios. Saltando esta columna.")
            continue

        # --- 1. Conteo de Diferencias ---
        
        # Alineación y conteo: df_original vs df_trabajo
        common_idx_ot = df_original.index.intersection(df_trabajo.index)
        s_orig_ot = df_original.loc[common_idx_ot, columna]
        s_trab_ot = df_trabajo.loc[common_idx_ot, columna]
        
        # Contar diferencias: (val1 != val2) OR (uno es NaN y el otro no)
        # Llenar NaNs con un placeholder único para comparación directa si pd.NA no funciona como se espera con !=
        diff_mask_ot = (s_orig_ot != s_trab_ot) | (s_orig_ot.isnull() != s_trab_ot.isnull())
        count_diff_ot = diff_mask_ot.sum()
        print(f"  Número de valores diferentes entre df_original y df_trabajo (anomalías inyectadas): {count_diff_ot} / {len(common_idx_ot)}")

        # Alineación y conteo: df_original vs df_contextual_corregido_custom
        common_idx_oc = df_original.index.intersection(df_contextual_corregido_custom.index)
        s_orig_oc = df_original.loc[common_idx_oc, columna]
        s_corr_oc = df_contextual_corregido_custom.loc[common_idx_oc, columna]
        
        diff_mask_oc = (s_orig_oc != s_corr_oc) | (s_orig_oc.isnull() != s_corr_oc.isnull())
        count_diff_oc = diff_mask_oc.sum()
        print(f"  Número de valores diferentes entre df_original y df_contextual_corregido_custom: {count_diff_oc} / {len(common_idx_oc)}")

        # --- 2. Métricas de Diferencia (df_original vs df_contextual_corregido_custom) ---
        
        # Preparar datos para métricas (eliminar NaNs en pares)
        df_compare_oc = pd.DataFrame({'original': s_orig_oc, 'corregido': s_corr_oc}).dropna()
        
        if df_compare_oc.empty:
            print(f"  No hay datos comparables (después de dropna) entre df_original y df_contextual_corregido_custom para la columna {columna}.")
            rmse_overall = np.nan
            mae_overall = np.nan
            num_points_overall = 0
        else:
            rmse_overall = np.sqrt(mean_squared_error(df_compare_oc['original'], df_compare_oc['corregido']))
            mae_overall = mean_absolute_error(df_compare_oc['original'], df_compare_oc['corregido'])
            num_points_overall = len(df_compare_oc)
            print(f"  Métricas Generales (Original vs Corregido Total) para {num_points_overall} puntos:")
            print(f"    RMSE General: {rmse_overall:.4f}")
            print(f"    MAE General: {mae_overall:.4f}")

        # Métricas para puntos "muy diferentes"
        # Usar std_devs_originales[columna]
        std_col = std_devs_originales.get(columna)
        if std_col is None or std_col == 0: # Si no hay std_dev o es 0, no se puede usar para umbral
            print(f"  Advertencia: Desviación estándar para '{columna}' no disponible o es cero. No se calcularán métricas de diferencias significativas.")
            rmse_sig_2std = np.nan; mae_sig_2std = np.nan; count_sig_2std = 0
            rmse_sig_3std = np.nan; mae_sig_3std = np.nan; count_sig_3std = 0
        else:
            abs_diff = (df_compare_oc['original'] - df_compare_oc['corregido']).abs()
            
            # Umbral de 2 desviaciones estándar
            threshold_2std = 2 * std_col
            significant_diff_2std_mask = abs_diff > threshold_2std
            df_sig_diff_2std = df_compare_oc[significant_diff_2std_mask]
            count_sig_2std = len(df_sig_diff_2std)

            if count_sig_2std > 0:
                rmse_sig_2std = np.sqrt(mean_squared_error(df_sig_diff_2std['original'], df_sig_diff_2std['corregido']))
                mae_sig_2std = mean_absolute_error(df_sig_diff_2std['original'], df_sig_diff_2std['corregido'])
                print(f"  Métricas para {count_sig_2std} puntos 'muy diferentes' (> 2 std dev = {threshold_2std:.4f}):")
                print(f"    RMSE Significativo (2std): {rmse_sig_2std:.4f}")
                print(f"    MAE Significativo (2std): {mae_sig_2std:.4f}")
            else:
                print(f"  No se encontraron puntos con diferencias > 2 std dev para {columna}.")
                rmse_sig_2std = np.nan; mae_sig_2std = np.nan
            
            # Umbral de 3 desviaciones estándar
            threshold_3std = 3 * std_col
            significant_diff_3std_mask = abs_diff > threshold_3std
            df_sig_diff_3std = df_compare_oc[significant_diff_3std_mask]
            count_sig_3std = len(df_sig_diff_3std)

            if count_sig_3std > 0:
                rmse_sig_3std = np.sqrt(mean_squared_error(df_sig_diff_3std['original'], df_sig_diff_3std['corregido']))
                mae_sig_3std = mean_absolute_error(df_sig_diff_3std['original'], df_sig_diff_3std['corregido'])
                print(f"  Métricas para {count_sig_3std} puntos 'muy diferentes' (> 3 std dev = {threshold_3std:.4f}):")
                print(f"    RMSE Significativo (3std): {rmse_sig_3std:.4f}")
                print(f"    MAE Significativo (3std): {mae_sig_3std:.4f}")
            else:
                print(f"  No se encontraron puntos con diferencias > 3 std dev para {columna}.")
                rmse_sig_3std = np.nan; mae_sig_3std = np.nan

        resumen_diferencias.append({
            'Columna': columna,
            'Diff_Orig_vs_Trabajo': count_diff_ot,
            'Puntos_Comp_OT': len(common_idx_ot),
            'Diff_Orig_vs_CorregidoTotal': count_diff_oc,
            'Puntos_Comp_OC': len(common_idx_oc),
            'Puntos_Eval_Overall_OC': num_points_overall,
            'RMSE_Overall_OC': rmse_overall,
            'MAE_Overall_OC': mae_overall,
            'StdDev_Original_Col': std_col if std_col is not None else np.nan,
            'Puntos_SigDiff_2Std': count_sig_2std,
            'RMSE_SigDiff_2Std': rmse_sig_2std,
            'MAE_SigDiff_2Std': mae_sig_2std,
            'Puntos_SigDiff_3Std': count_sig_3std,
            'RMSE_SigDiff_3Std': rmse_sig_3std,
            'MAE_SigDiff_3Std': mae_sig_3std,
        })

    if resumen_diferencias:
        df_resumen_diferencias = pd.DataFrame(resumen_diferencias)
        print("\n\n--- Resumen del Análisis de Diferencias ---")
        # Configurar pandas para mostrar todas las columnas
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000) # Ajustar ancho para mejor visualización en consola
        print(df_resumen_diferencias.to_string())
        # Guardarlo como global si se necesita en el siguiente bloque
        globals()['df_resumen_diferencias'] = df_resumen_diferencias
    else:
        print("\nNo se generó resumen de diferencias (posiblemente no había columnas para analizar).")

print("\n--- Fin del Análisis Numérico de Diferencias ---")

In [ ]:
# --- Inicio de la Visualización del Análisis Numérico ---

if 'df_resumen_diferencias' in globals() and isinstance(globals()['df_resumen_diferencias'], pd.DataFrame):
    df_resumen = globals()['df_resumen_diferencias'].copy()
    print("DataFrame 'df_resumen_diferencias' encontrado. Iniciando visualizaciones.")

    # --- 1. Número de Valores Diferentes por Sensor ---
    df_plot_diff_counts = df_resumen.set_index('Columna')[['Diff_Orig_vs_Trabajo', 'Diff_Orig_vs_CorregidoTotal']]
    # Ordenar para mejor visualización
    df_plot_diff_counts_sorted = df_plot_diff_counts.sort_values(by='Diff_Orig_vs_CorregidoTotal', ascending=False)
    
    ax1 = df_plot_diff_counts_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax1.set_title('Número de Valores Diferentes por Sensor', fontsize=16)
    ax1.set_xlabel('Sensor', fontsize=12)
    ax1.set_ylabel('Cantidad de Diferencias', fontsize=12)
    ax1.legend(['Original vs. Trabajo (Anomalías Inyectadas)', 'Original vs. Corregido Total'])
    ax1.tick_params(axis='x', rotation=45) # Removido ha='right'
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ','))) # Formatear eje Y
    plt.tight_layout()
    plt.show()

    # --- 2. Métricas Generales (RMSE y MAE) - Original vs. Corregido Total ---
    df_plot_overall_metrics = df_resumen.set_index('Columna')[['RMSE_Overall_OC', 'MAE_Overall_OC']]
    
    # RMSE General
    df_plot_rmse_overall_sorted = df_plot_overall_metrics[['RMSE_Overall_OC']].sort_values(by='RMSE_Overall_OC', ascending=False)
    ax2_rmse = df_plot_rmse_overall_sorted.plot(kind='bar', figsize=(15, 7), color='skyblue', legend=None)
    ax2_rmse.set_title('RMSE General (Original vs. Corregido Total)', fontsize=16)
    ax2_rmse.set_xlabel('Sensor', fontsize=12)
    ax2_rmse.set_ylabel('RMSE', fontsize=12)
    ax2_rmse.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # MAE General
    df_plot_mae_overall_sorted = df_plot_overall_metrics[['MAE_Overall_OC']].sort_values(by='MAE_Overall_OC', ascending=False)
    ax2_mae = df_plot_mae_overall_sorted.plot(kind='bar', figsize=(15, 7), color='lightcoral', legend=None)
    ax2_mae.set_title('MAE General (Original vs. Corregido Total)', fontsize=16)
    ax2_mae.set_xlabel('Sensor', fontsize=12)
    ax2_mae.set_ylabel('MAE', fontsize=12)
    ax2_mae.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # --- 3. Número de Puntos Significativamente Diferentes ---
    df_plot_sig_counts = df_resumen.set_index('Columna')[['Puntos_SigDiff_2Std', 'Puntos_SigDiff_3Std']]
    df_plot_sig_counts_sorted = df_plot_sig_counts.sort_values(by='Puntos_SigDiff_2Std', ascending=False)

    ax3 = df_plot_sig_counts_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax3.set_title('Número de Puntos Significativamente Diferentes (Original vs. Corregido Total)', fontsize=16)
    ax3.set_xlabel('Sensor', fontsize=12)
    ax3.set_ylabel('Cantidad de Puntos', fontsize=12)
    ax3.legend(['Diferencia > 2 Std Dev', 'Diferencia > 3 Std Dev'])
    ax3.tick_params(axis='x', rotation=45)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    plt.tight_layout()
    plt.show()

    # --- 4. RMSE para Puntos Significativamente Diferentes ---
    df_plot_rmse_sig = df_resumen.set_index('Columna')[['RMSE_SigDiff_2Std', 'RMSE_SigDiff_3Std']]
    df_plot_rmse_sig_sorted = df_plot_rmse_sig.sort_values(by='RMSE_SigDiff_2Std', ascending=False)
    
    ax4 = df_plot_rmse_sig_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax4.set_title('RMSE para Puntos Significativamente Diferentes (Original vs. Corregido Total)', fontsize=16)
    ax4.set_xlabel('Sensor', fontsize=12)
    ax4.set_ylabel('RMSE', fontsize=12)
    ax4.legend(['RMSE para Puntos con Diff > 2 Std Dev', 'RMSE para Puntos con Diff > 3 Std Dev'])
    ax4.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    # --- 5. MAE para Puntos Significativamente Diferentes ---
    df_plot_mae_sig = df_resumen.set_index('Columna')[['MAE_SigDiff_2Std', 'MAE_SigDiff_3Std']]
    df_plot_mae_sig_sorted = df_plot_mae_sig.sort_values(by='MAE_SigDiff_2Std', ascending=False)

    ax5 = df_plot_mae_sig_sorted.plot(kind='bar', figsize=(15, 7), width=0.8)
    ax5.set_title('MAE para Puntos Significativamente Diferentes (Original vs. Corregido Total)', fontsize=16)
    ax5.set_xlabel('Sensor', fontsize=12)
    ax5.set_ylabel('MAE', fontsize=12)
    ax5.legend(['MAE para Puntos con Diff > 2 Std Dev', 'MAE para Puntos con Diff > 3 Std Dev'])
    ax5.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

    print("\n--- Fin de la Visualización del Análisis Numérico ---")

else:
    print("Error: No se encontró el DataFrame 'df_resumen_diferencias' en el entorno global o no es un DataFrame. Por favor, asegúrate de que se haya generado correctamente en el paso anterior.")

### 7. Observaciones y conclusiones

Basándonos en la información resultante, podemos redactar las observaciones y conclusiones finales sobre el proceso de clasificación y corrección de anomalías.

El proceso desarrollado abarcó la **generación sintética de anomalías**, la **detección** y **clasificación** de estas, y finalmente, la **corrección** específica para cada tipo identificado, con el objetivo de mejorar la calidad de los datos.

1.  **Inyección de Anomalías Sintéticas:**
    *   Se inyectaron varios tipos de anomalías para crear un conjunto de datos con etiquetas de "anomalia" y su tipo específico.
    *   Se crearon **150 secuencias de "Sensor Atascado"** con duraciones entre 5 y 20 periodos. Al final de la inyección, se etiquetaron **1958 filas** como 'Sensor Atascado'.
    *   Se inyectó **"Ruido / Picos Aleatorios"** en el 3% de las filas 'normales', afectando a 1 sensor por fila, con una magnitud basada en la desviación estándar (entre 3 y 6 veces la std). Esto resultó en la modificación de **6802 puntos de datos** en **6802 filas**, todas etiquetadas como 'Ruido'.
    *   Se inyectaron **"Valores Fuera de Rango"** en el 2% de las filas 'normales', modificando 1 sensor por fila. La inyección se basó en rangos válidos predefinidos para cada sensor, excediendo los límites por un factor (0.1) del rango. Esto resultó en la modificación de **4399 puntos de datos** en **4399 filas**, etiquetados como 'Valores Fuera de Rango'.
    *   Se inyectó **"Desviación de Correlación"** en el 3% de las filas 'normales' para pares de sensores con alta correlación. La lógica buscó romper la relación esperada basándose en percentiles (P25/P75) y la correlación observada. Se identificaron **5568 puntos** con 'Desviación de Correlación', de los cuales se modificaron **1617 puntos de datos**, etiquetados como 'Desviación de Correlación'.
    *   Se generaron **anomalías "Contextuales"** basadas en dos reglas: 'Iluminación Interior Anómala Nocturna' (PRGINT no cero de noche cuando PRAD es cero) y 'Falta de Respuesta del CO2 Interior a la Ventilación' (XCO2I no baja con alta ventilación y gradiente favorable). Se intentaron crear 100 secuencias de CO2 contextual, pero solo se crearon 0. Se inyectó 'Luz Nocturna Anómala' en 1% de las filas candidatas nocturnas, modificando 538 filas. En total, se etiquetaron **538 filas** como 'Contextual'.
    *   El total de filas en el DataFrame con anomalías inyectadas (`df_trabajo`) fue de **233381**, con **19981 filas marcadas como 'anomalia'** (aproximadamente **8.56%** del total).

2.  **Modelos de Detección y Clasificación:**
    *   **Modelo 1 (Detector Normal vs. Anomalía):** Se entrenó un clasificador **Random Forest** para distinguir entre datos 'normales' y 'anomalia'.
        *   El modelo mostró una **alta precisión general (Accuracy: 0.9966)**.
        *   La matriz de confusión y el reporte de clasificación indicaron un rendimiento casi perfecto para la clase 'Normal' (Precision 1.00, Recall 1.00) y un rendimiento muy bueno para la clase 'Anomalía' (Precision 1.00, Recall 0.96, F1-score 0.98).
        *   Se detectaron correctamente **5771 anomalías (Verdaderos Positivos)**, se marcaron incorrectamente **18 normales como anomalías (Falsos Positivos)**, y se dejaron de detectar **223 anomalías (Falsos Negativos)**.
        *   El **ROC AUC Score fue 0.9999** y el **Precision-Recall AUC Score (Average Precision) fue 0.9992**, métricas muy altas que confirman la excelente capacidad del modelo para separar las clases.
        *   El desglose por tipo de anomalía real en el conjunto de prueba mostró tasas de detección variadas: **Contextual (100.0%)**, Datos Faltantes (99.9%), Valores Fuera de Rango (99.8%), Desviación de Correlación (97.6%), Ruido (94.9%), y Sensor Atascado (82.4%).
        *   Las características más importantes para este modelo incluyeron PRAD, XTINV, XHINV, PCO2EXT, PRGINT, PTEXT, XCO2I, XTS, PHEXT, PVV, y las columnas indicadoras de datos faltantes.

    *   **Modelo 2 (Clasificador de Tipos de Anomalía):** Se entrenó otro clasificador **Random Forest** usando solo las instancias que eran verdaderas anomalías, para predecir su tipo específico ('Contextual', 'Datos Faltantes', etc.).
        *   La precisión general de este clasificador fue **0.8869**.
        *   El reporte de clasificación mostró un excelente rendimiento (Precision y Recall de 1.00 o cercanos) para tipos como 'Contextual', 'Datos Faltantes', 'Desviación de Correlación' y 'Sensor Atascado'.
        *   Los tipos 'Ruido' y 'Valores Fuera de Rango' mostraron un rendimiento menor (Ruido: Precision 0.79, Recall 0.92; Valores Fuera de Rango: Precision 0.84, Recall 0.61). Esto sugiere que clasificar `Ruido` y `Valores Fuera de Rango` puede ser más desafiante o que el modelo confunde estos tipos con otros en algunos casos.

3.  **Corrección de Anomalías por Tipo:**
    *   **Datos Faltantes:** Se utilizó **IterativeImputer con un Random Forest Regressor** para rellenar los valores ausentes. Se evaluó la calidad de la imputación comparando los valores imputados con los valores originales donde originalmente había NaNs. Las métricas globales fueron un **RMSE de 5.343** y **MAE de 2.199** sobre 4667 puntos evaluados. Se realizó una "Imputación Final" de NaNs restantes tras otras correcciones, con un **RMSE global de 0.733** y **MAE de 0.254** sobre 1535 puntos.
    *   **Sensor Atascado:** Se implementó un **método híbrido**.
        *   Para **sensores dinámicos** (PRAD, PRGINT, UVENT\_cen, UVENT\_lN), se usó un **método estacional simple** que promedia valores de la misma hora en días previos. La evaluación de los segmentos corregidos mostró un **RMSE global de 14.708** y **MAE de 2.571** sobre 98259 puntos.
        *   Para **sensores estables** (el resto), se usó **interpolación lineal** después de marcar los segmentos atascados como NaN. La evaluación mostró un **RMSE global de 1.806** y **MAE de 0.378** sobre 2956 puntos.
    *   **Ruido:** Se aplicó un **método local personalizado** que corrige los puntos ruidosos identificados por el Modelo 2. La corrección se basa en comparar el valor anómalo con el promedio de sus vecinos inmediatos; si la diferencia es grande (usando un umbral basado en 2 * std dev), se reemplaza por el promedio de los vecinos o el valor anterior. La evaluación de los puntos corregidos mostró un **RMSE global de 22.373** y **MAE de 1.291** sobre 28752 puntos.
    *   **Valores Fuera de Rango:** Se aplicó un **método local similar al de Ruido**, también basado en vecinos y un umbral (2 * std dev), pero operando sobre los puntos identificados como 'Fuera de Rango' por el Modelo 2. La evaluación de los puntos corregidos mostró un **RMSE global de 40.038** y **MAE de 3.101** sobre 7485 puntos.
    *   **Contextual:** Se aplicó un **método local similar a Ruido y Fuera de Rango**, basado en vecinos y un umbral (2 * std dev). La evaluación de los puntos corregidos mostró un **RMSE global de 0.448** y **MAE de 0.058** sobre 1824 puntos.

4.  **Análisis Numérico de Diferencias Globales (Original vs. Corregido Total):**
    *   Se compararon los DataFrames `df_original` (datos limpios), `df_trabajo` (con anomalías inyectadas) y `df_contextual_corregido_custom` (después de todas las correcciones).
    *   El análisis mostró que, aunque el DataFrame con anomalías inyectadas (`df_trabajo`) tenía un número de diferencias respecto al original (`df_original`) que variaba por columna (ej. 1517 para PCO2EXT, 2288 para PRAD), el DataFrame corregido (`df_contextual_corregido_custom`) mostró **un número significativamente mayor de diferencias totales respecto al original (ej. 4353 para PCO2EXT, 3264 para PRAD)**. Esto indica que las correcciones introdujeron variaciones en más puntos de los que originalmente tenían anomalías inyectadas.
    *   Las métricas generales (RMSE y MAE) entre `df_original` y `df_contextual_corregido_custom` proporcionan una medida global de qué tan cerca quedó el conjunto de datos corregido del original. Estos valores varían considerablemente entre columnas (ej. RMSE General de 0.2647 para PVV, 32.8549 para PRAD).
    *   El análisis de diferencias significativas (puntos con diferencia > 2 o 3 * std dev) mostró que, si bien estos puntos son pocos, tienen un RMSE y MAE muy elevados. Esto sugiere que, aunque la mayoría de los puntos corregidos están cerca de los valores originales, hay un pequeño subconjunto de puntos donde la corrección falló significativamente en replicar el valor original, posiblemente por las limitaciones de los métodos de corrección simple utilizados (vecinos, estacional, interpolación) o por la complejidad inherente de algunas anomalías contextuales o de correlación.

**En Resumen:**

Se implementó un pipeline completo para generar, detectar, clasificar y corregir cinco tipos de anomalías en datos de sensores. Los modelos de detección y clasificación mostraron un alto rendimiento general. Los métodos de corrección, que variaron desde la imputación iterativa para datos faltantes hasta métodos locales y estacionales para otros tipos, demostraron capacidad para modificar los valores anómalos. Sin embargo, la evaluación final comparando el dataset corregido con el original sugiere que los métodos de corrección aplicados, aunque efectivos en promedio (RMSE/MAE generales), pueden no ser ideales para todos los puntos o tipos de anomalía, especialmente aquellos que requieren un entendimiento más profundo del contexto o la dinámica del sistema, resultando en algunas diferencias significativas persistentes o introducidas.

# Experimentacion

## Paso 1: Guardar (Exportar) el Modelo 1, el Imputador y las Columnas

In [ ]:
# Guardamos el imputador (que aprendió las medianas de tus datos originales)
joblib.dump(imputer, 'imputador_nans.joblib')

# Guardamos el modelo detector (Modelo 1)
joblib.dump(rf_detector, 'modelo_1_detector.joblib')

# ES MUY IMPORTANTE guardar el orden exacto de las columnas que usó el modelo
joblib.dump(columnas_potenciales_features, 'features_modelo_1.joblib')

print("¡Modelo 1, Imputador y variables guardados con éxito!")

## Paso 2: Cargar y aplicar los modelos a la NUEVA base de datos

### 1. Cargar la nueva base de datos

In [ ]:
# 1. Definir la ruta a la carpeta donde están los CSV
ruta_carpeta = 'Completados_30s/*.csv'
archivos_csv = glob.glob(ruta_carpeta)

if not archivos_csv:
    print("No se encontraron archivos CSV. Verifica que la carpeta 'Completados' exista en el mismo directorio que este notebook.")
else:
    columnas_por_archivo = {}
    
    # 2. Leer los encabezados de cada archivo
    for archivo in archivos_csv:
        nombre_archivo = os.path.basename(archivo)
        try:
            # Usamos nrows=0 para leer solo los encabezados y no saturar la memoria
            df = pd.read_csv(archivo, nrows=0) 
            columnas_por_archivo[nombre_archivo] = set(df.columns)
        except Exception as e:
            print(f"Error al leer {nombre_archivo}: {e}")

    # 3. Mostrar las columnas de cada archivo
    print("=== COLUMNAS POR ARCHIVO ===")
    for archivo, columnas in columnas_por_archivo.items():
        print(f"\n📁 {archivo} ({len(columnas)} columnas):")
        print(list(columnas))

In [ ]:
# Ingeniería de características

# Diccionario donde guardaremos cada base de datos procesada por separado.
# La llave será el nombre del archivo y el valor será el DataFrame listo.
datasets_listos = {}

if not archivos_csv:
    print("No se encontraron archivos CSV.")
else:
    for archivo in archivos_csv:
        nombre_dataset = os.path.basename(archivo).replace('.csv', '')
        
        try:
            print(f"\n=======================================================")
            print(f"Procesando base de datos: {nombre_dataset}")
            print(f"=======================================================")
            
            # 1. Cargar el DataFrame individual
            df_temp = pd.read_csv(archivo)
            
            # 2. Manejo de Fechas
            if df_temp['Fecha'].dtype == 'object':
                # Cambiamos el format para que coincida con lo que imprimimos antes
                df_temp['Fecha'] = pd.to_datetime(df_temp['Fecha'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
                
                fechas_nulas = df_temp['Fecha'].isnull().sum()
                if fechas_nulas > 0:
                    print(f"  ⚠️ Advertencia: {fechas_nulas} fechas no se pudieron leer. Revisa el formato.")
                else:
                    print(f"  📅 Fechas leídas correctamente (Formato Día/Mes/Año)")

            # 3. Extraer características temporales
            df_temp['Hora'] = df_temp['Fecha'].dt.hour
            df_temp['DiaSemana'] = df_temp['Fecha'].dt.dayofweek
            df_temp['Mes'] = df_temp['Fecha'].dt.month
            
            # 4. Preparar las columnas para las predicciones
            df_temp['etiqueta_deteccion'] = 'normal' 
            df_temp['etiqueta_tipo_anomalia'] = pd.NA 
            
            # 5. Guardar en el diccionario
            datasets_listos[nombre_dataset] = df_temp
            
            print(f"  ✅ Procesado exitosamente. Filas: {len(df_temp)}")
            
        except Exception as e:
            print(f"  ❌ Error al procesar {nombre_dataset}: {e}")

print("\n=== RESUMEN DE BASES DE DATOS LISTAS PARA EVALUAR ===")
for nombre in datasets_listos.keys():
    print(f"- {nombre}")

### 2. Cargar el imputador, el modelo y la lista de columnas

In [ ]:
# 1. Cargar el imputador, el modelo y la lista de columnas
try:
    imputer_cargado = joblib.load('imputador_nans.joblib')
    rf_detector_cargado = joblib.load('modelo_1_detector.joblib')
    features_esperadas = joblib.load('features_modelo_1.joblib')
    print("✅ Modelos y variables cargados correctamente desde el disco.\n")
except FileNotFoundError:
    print("❌ Error: No se encontraron los archivos .joblib. Recuerda que primero debes guardar los modelos en tu notebook de entrenamiento usando joblib.dump().")
    # Detenemos la ejecución si no hay modelos
    raise 

print("=== INICIANDO DETECCIÓN DE ANOMALÍAS (MODELO 1) ===")

# Iterar sobre cada base de datos ya procesada
for nombre, df in datasets_listos.items():
    print(f"\nEvaluando: {nombre}")
    
    try:
        # 1. Seleccionar SOLO las columnas con las que el modelo fue entrenado
        X_nuevo = df[features_esperadas].copy()
        
        # 2. Aplicar el imputador para rellenar los NaNs (usamos .transform)
        X_nuevo_imputed_np = imputer_cargado.transform(X_nuevo)
        
        # 3. Hacer las predicciones (0 para normal, 1 para anomalía)
        predicciones = rf_detector_cargado.predict(X_nuevo_imputed_np)
        
        # 4. Guardar las predicciones en el DataFrame actual
        # Convertimos los 1 y 0 a texto para mantener la coherencia con tu código
        df['etiqueta_deteccion'] = np.where(predicciones == 1, 'anomalia', 'normal')
        
        # Mostrar resumen
        anomalias_detectadas = (predicciones == 1).sum()
        total_filas = len(df)
        porcentaje = (anomalias_detectadas / total_filas) * 100
        
        print(f"  -> Anomalías detectadas: {anomalias_detectadas} ({porcentaje:.2f}%)")
        
    except KeyError as e:
        print(f"  ❌ Error: Al dataset le faltan columnas necesarias para el modelo: {e}")

In [ ]:
# 1. Definir la lista de todas las variables que quieres revisar
variables_a_graficar = ['XTS', 'PCO2EXT', 'XHINV', 'PHEXT', 'PRGINT', 'PRAD', 
                        'PTEXT', 'XCO2I', 'XTINV', 'UVENT_cen', 'UVENT_lN', 'PVV']

# 2. Elegir el dataset
nombre_evaluar = 'AgroConnect_OPC_20241001_20241115'
df_visual = datasets_listos[nombre_evaluar].copy()

# 3. Bucle para generar todas las gráficas
for variable in variables_a_graficar:
    # Verificamos si la variable existe en este dataset para evitar errores
    if variable in df_visual.columns:
        plt.figure(figsize=(15, 5))
        
        # Filtramos normales y anomalías
        normales = df_visual[df_visual['etiqueta_deteccion'] == 'normal']
        anomalias = df_visual[df_visual['etiqueta_deteccion'] == 'anomalia']
        
        # Dibujamos puntos normales
        plt.scatter(normales['Fecha'], normales[variable], 
                    color='blue', label='Normal', alpha=0.4, s=10)
        
        # Dibujamos anomalías en rojo
        plt.scatter(anomalias['Fecha'], anomalias[variable], 
                    color='red', label='Anomalía Detectada', alpha=0.8, s=20)
        
        # Configuración de la gráfica
        plt.title(f"Detección de Anomalías - Dataset: {nombre_evaluar}\nVariable: {variable}")
        plt.xlabel("Fecha")
        plt.ylabel(f"Valor de {variable}")
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True, linestyle='--', alpha=0.5) # Añadimos rejilla para mejor lectura
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ La variable {variable} no se encuentra en el dataset.")

In [ ]:
# 1. Definir la variable específica que quieres revisar
variable = 'XHINV'

# 2. Elegir el dataset
nombre_evaluar = 'AgroConnect_OPC_20241001_20241115'
df_visual = datasets_listos[nombre_evaluar].copy()

# 3. Asegurar que la columna 'Fecha' sea un objeto datetime para poder filtrar por rango
df_visual['Fecha'] = pd.to_datetime(df_visual['Fecha'], dayfirst=True, errors='coerce')

# 4. Filtrar por el rango de fechas solicitado 
fecha_inicio = '2024-10-28 18:30:00'
fecha_fin = '2024-10-29 06:00:00'

df_filtrado = df_visual[(df_visual['Fecha'] >= fecha_inicio) & (df_visual['Fecha'] <= fecha_fin)]

# 5. Generar la gráfica
if variable in df_filtrado.columns:
    plt.figure(figsize=(15, 5))
    
    # Filtramos normales y anomalías usando el DataFrame YA FILTRADO por fechas
    normales = df_filtrado[df_filtrado['etiqueta_deteccion'] == 'normal']
    anomalias = df_filtrado[df_filtrado['etiqueta_deteccion'] == 'anomalia']
    
    # Dibujamos puntos normales
    plt.scatter(normales['Fecha'], normales[variable], 
                color='blue', label='Normal', alpha=0.4, s=10)
    
    # Dibujamos anomalías en rojo
    plt.scatter(anomalias['Fecha'], anomalias[variable], 
                color='red', label='Outlier', alpha=0.8, s=20)
    
    # Configuración de la gráfica
    plt.xlabel("Date") # Cambiado para reflejar fecha y hora
    plt.ylabel(variable) # Ajustado a la variable dinámicamente
    plt.legend()
    
    # <-- NUEVA CONFIGURACIÓN PARA FECHA Y HORA -->
    # Forzamos a que el eje X muestre Día/Mes/Año Hora:Minuto
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d/%m/%Y %H:%M'))
    
    plt.xticks
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️ La variable {variable} no se encuentra en el dataset.")

# Paso 3: Conectar con el Modelo 2 (Clasificación)

In [ ]:
joblib.dump(rf_clasificador_tipo, 'modelo_2_clasificador.joblib')
joblib.dump(label_encoder_model2, 'label_encoder_modelo_2.joblib')
print("¡Modelo 2 y Codificador guardados!")

In [ ]:
# 1. Cargar el Modelo 2 y el Label Encoder
try:
    rf_clasificador_cargado = joblib.load('modelo_2_clasificador.joblib')
    label_encoder_cargado = joblib.load('label_encoder_modelo_2.joblib')
    print("✅ Modelo 2 y Label Encoder cargados correctamente.\n")
except FileNotFoundError:
    print("❌ Error: No se encontraron los archivos del Modelo 2. Asegúrate de guardarlos primero con joblib.dump()")
    raise

print("=== INICIANDO CLASIFICACIÓN DE TIPOS DE ANOMALÍAS (MODELO 2) ===")

# Iterar sobre las bases de datos ya evaluadas por el Modelo 1
for nombre, df in datasets_listos.items():
    print(f"\nClasificando anomalías en: {nombre}")
    
    # 1. Crear una máscara para filtrar SOLO las anomalías
    mascara_anomalias = df['etiqueta_deteccion'] == 'anomalia'
    df_solo_anomalias = df[mascara_anomalias]
    
    if df_solo_anomalias.empty:
        print("  -> No se detectaron anomalías en este dataset. Se omite el Modelo 2.")
        continue
        
    try:
        # 2. Preparar X solo para esas filas anómalas
        X_anomalias = df_solo_anomalias[features_esperadas].copy()
        
        # OJO: Usamos el MISMO imputador del Modelo 1 para rellenar los NaNs
        X_anom_imputed_np = imputer_cargado.transform(X_anomalias)
        
        # 3. Predecir el tipo de anomalía (esto devuelve números enteros)
        predicciones_numericas = rf_clasificador_cargado.predict(X_anom_imputed_np)
        
        # 4. Convertir los números de vuelta a texto usando el Label Encoder
        predicciones_texto = label_encoder_cargado.inverse_transform(predicciones_numericas)
        
        # 5. Guardar las etiquetas de tipo EXACTAMENTE en las filas donde hay anomalías
        # Usamos .loc y la máscara para no sobreescribir las filas 'normales'
        df.loc[mascara_anomalias, 'etiqueta_tipo_anomalia'] = predicciones_texto
        
        # 6. Mostrar un resumen de lo que clasificó
        conteo_tipos = df.loc[mascara_anomalias, 'etiqueta_tipo_anomalia'].value_counts()
        print("  -> Tipos de anomalías encontradas:")
        for tipo, cantidad in conteo_tipos.items():
            print(f"     - {tipo}: {cantidad} registros")
            
    except Exception as e:
        print(f"  ❌ Error al clasificar en {nombre}: {e}")

print("\n✅ Proceso de Clasificación (Modelo 2) Finalizado.")

In [ ]:
# 1. Definir las variables a revisar
variables_a_graficar = ['XTS', 'PCO2EXT', 'XHINV', 'PHEXT', 'PRGINT', 'PRAD', 
                        'PTEXT', 'XCO2I', 'XTINV', 'UVENT_cen', 'UVENT_lN', 'PVV']

# 2. Elegir el dataset que ya pasó por el Modelo 2
nombre_evaluar = 'AgroConnect_OPC_20250301_20250430'
df_visual = datasets_listos[nombre_evaluar].copy()

# 3. Bucle para generar las gráficas con leyenda multiclase
for variable in variables_a_graficar:
    if variable in df_visual.columns:
        plt.figure(figsize=(15, 6))
        
        # --- A. GRAFICAR PUNTOS NORMALES ---
        # Usamos un color neutro (gris o azul claro) para que no distraigan
        normales = df_visual[df_visual['etiqueta_deteccion'] == 'normal']
        plt.scatter(normales['Fecha'], normales[variable], 
                    color='lightgrey', label='Normal', alpha=0.3, s=10)
        
        # --- B. GRAFICAR ANOMALÍAS POR TIPO ---
        # Obtenemos solo las filas que son anomalías
        df_anomalias = df_visual[df_visual['etiqueta_deteccion'] == 'anomalia']
        
        # Identificamos qué tipos de anomalías encontró el Modelo 2
        tipos_encontrados = df_anomalias['etiqueta_tipo_anomalia'].unique()
        
        # Usamos un mapa de colores para asignar uno distinto a cada tipo
        cmap = plt.get_cmap('tab10') 
        
        for i, tipo in enumerate(tipos_encontrados):
            # Filtramos el subset de este tipo específico de anomalía
            subset = df_anomalias[df_anomalias['etiqueta_tipo_anomalia'] == tipo]
            
            plt.scatter(subset['Fecha'], subset[variable], 
                        color=cmap(i), 
                        label=f"Anomalía: {tipo}", 
                        edgecolors='black', # Borde negro para que resalten
                        linewidths=0.5,
                        s=30)
        
        # --- CONFIGURACIÓN DE LA GRÁFICA ---
        plt.title(f"Clasificación de Anomalías (Modelo 2) - {nombre_evaluar}\nVariable: {variable}")
        plt.xlabel("Fecha")
        plt.ylabel(f"Valor de {variable}")
        
        # Colocamos la leyenda fuera de la gráfica si hay muchos tipos
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), title="Categorías")
        
        plt.xticks(rotation=45)
        plt.grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ Variable {variable} no encontrada.")

# Conectar con el Modelo 3 (Correción)

In [ ]:
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'DATOS FALTANTES' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Sincronización Crítica) ---
nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'

if nombre_objetivo not in datasets_listos:
    raise KeyError(f"Dataset {nombre_objetivo} no disponible.")

df_trabajo = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

# Función de limpieza que ya probamos que funciona para sincronizar
def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para que df_trabajo y df_original coincidan perfectamente
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_trabajo = preparar_y_redondear(df_trabajo)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. IDENTIFICACIÓN DE COLUMNAS ---
columnas_sensores = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
                     'XCO2I', 'XHINV', 'XTINV', 'XTS', 'UVENT_cen', 'UVENT_lN']
cols_imputar = [col for col in columnas_sensores if col in df_trabajo.columns]

# --- 3. CONFIGURACIÓN DEL IMPUTADOR ---
# Usamos RandomForest para capturar relaciones no lineales entre sensores
iterative_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1, max_depth=10),
    max_iter=10,
    random_state=42,
    initial_strategy='median'
)

# --- 4. EJECUCIÓN DE LA IMPUTACIÓN ---
print(f"Ejecutando IterativeImputer sobre: {cols_imputar}")
# Entrenamos y transformamos
df_imputado_np = iterative_imputer.fit_transform(df_trabajo[cols_imputar])
df_referencia_imputada = pd.DataFrame(df_imputado_np, columns=cols_imputar, index=df_trabajo.index)

# Aplicamos la corrección solo donde: 1. Es NaN  Y  2. El modelo dijo "Datos Faltantes"
mask_pred_faltantes = df_trabajo['etiqueta_tipo_anomalia'] == 'Datos Faltantes'

all_true, all_imputed = [], []

for col in cols_imputar:
    mask_nan = df_trabajo[col].isnull()
    mask_final = mask_pred_faltantes & mask_nan
    
    if mask_final.any():
        idx_a_corregir = df_trabajo.index[mask_final]
        
        # --- EVALUACIÓN (Sincronizada) ---
        idx_comun = idx_a_corregir.intersection(df_original_eval.index)
        if not idx_comun.empty:
            t_vals = df_original_eval.loc[idx_comun, col]
            i_vals = df_referencia_imputada.loc[idx_comun, col]
            m_valid = t_vals.notna() & i_vals.notna()
            all_true.extend(t_vals[m_valid].tolist())
            all_imputed.extend(i_vals[m_valid].tolist())
        
        # Aplicar el valor imputado al dataset real
        df_trabajo.loc[idx_a_corregir, col] = df_referencia_imputada.loc[idx_a_corregir, col]

# --- 5. MÉTRICAS Y GRÁFICA ---
if all_true:
    rmse = np.sqrt(mean_squared_error(all_true, all_imputed))
    print(f"\n✅ Imputación evaluada sobre {len(all_true)} puntos.")
    print(f"RMSE Global de Imputación: {rmse:.3f}")

    plt.figure(figsize=(7, 7))
    plt.scatter(all_true, all_imputed, alpha=0.4, s=10, c='blue', label='Valores Imputados')
    lims = [min(all_true), max(all_true)]
    plt.plot(lims, lims, 'r--', label='Ideal')
    plt.title("Calidad de Imputación: Real vs IterativeImputer")
    plt.xlabel("Valor Real (Original)"); plt.ylabel("Valor Imputado")
    plt.legend(); plt.grid(True); plt.show()
else:
    print("\n⚠️ No se encontraron puntos coincidentes para evaluar (revisa los rangos de fecha).")

# Guardar progreso
datasets_listos[nombre_objetivo] = df_trabajo.reset_index()
print("\n✅ FASE 1 COMPLETADA.")

In [ ]:
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'SENSOR ATASCADO' ===")

# --- 1. FUNCIONES DE CORRECCIÓN (Mantenidas de tu código original) ---

def marcar_segmentos_stuck_como_nan_condicional(df_a_marcar, columna_nombre, predicciones_series, min_duracion_atasco=5):
    valores = df_a_marcar[columna_nombre].values
    indices_df = df_a_marcar.index
    n = len(valores)
    segmentos_nanificados = []
    nan_count = 0
    i = 0
    while i < n:
        if pd.isna(valores[i]):
            i += 1
            continue
        j = i
        while j < n and pd.notna(valores[j]) and valores[j] == valores[i]:
            j += 1
        duracion = j - i
        
        if duracion >= min_duracion_atasco:
            indices_segmento = indices_df[i:j]
            # Verificamos si en este segmento el modelo predijo "Sensor Atascado"
            if (predicciones_series.loc[indices_segmento] == "Sensor Atascado").any():
                df_a_marcar.loc[indices_segmento, columna_nombre] = np.nan
                segmentos_nanificados.append((indices_segmento[0], indices_segmento[-1]))
                nan_count += 1
                i = j
                continue
        i += 1
    return nan_count, segmentos_nanificados

def corregir_dinamicos_con_estacional_simple(df_a_corregir, df_historico, columna_a_corregir, start_idx, end_idx, ventana_dias=7):
    # Extraemos el segmento dañado
    segmento_indices = df_a_corregir.loc[start_idx:end_idx].index
    
    for idx_anomalo in segmento_indices:
        hora_anomala = idx_anomalo.hour
        fecha_anomala = idx_anomalo.date()
        
        # Buscamos en los días anteriores
        min_date = fecha_anomala - pd.Timedelta(days=ventana_dias + 5)
        max_date = fecha_anomala - pd.Timedelta(days=1)
        
        # Filtramos el histórico para esa hora específica
        mask_historico = (df_historico.index.date >= min_date) & (df_historico.index.date <= max_date) & (df_historico.index.hour == hora_anomala)
        valores_historicos = df_historico.loc[mask_historico, columna_a_corregir].dropna()
        
        if len(valores_historicos) >= 1:
            df_a_corregir.loc[idx_anomalo, columna_a_corregir] = valores_historicos.median()

# --- 2. PREPARACIÓN DE DATOS ---
nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'
df_hibrido = datasets_listos[nombre_objetivo].copy()

# Es vital tener el índice como Datetime para la corrección estacional e interpolación
if 'Fecha' in df_hibrido.columns and not isinstance(df_hibrido.index, pd.DatetimeIndex):
    df_hibrido['Fecha'] = pd.to_datetime(df_hibrido['Fecha'], errors='coerce')
    df_hibrido.set_index('Fecha', inplace=True)
    df_hibrido = df_hibrido.sort_index()

df_original_eval = df_original.copy()
if 'Fecha' in df_original_eval.columns and not isinstance(df_original_eval.index, pd.DatetimeIndex):
    df_original_eval['Fecha'] = pd.to_datetime(df_original_eval['Fecha'], errors='coerce')
    df_original_eval.set_index('Fecha', inplace=True)
    df_original_eval = df_original_eval.sort_index()

serie_predicciones = df_hibrido['etiqueta_tipo_anomalia']

# --- 3. SEPARACIÓN DE VARIABLES ---
columnas_dinamicas = [col for col in ['PRAD', 'PRGINT', 'UVENT_cen', 'UVENT_lN', 'PVV'] if col in df_hibrido.columns]
columnas_estables = [col for col in ['XTS', 'PCO2EXT', 'XHINV', 'PHEXT', 'PTEXT', 'XCO2I', 'XTINV'] if col in df_hibrido.columns]

print(f"-> Columnas Dinámicas (Estacional): {columnas_dinamicas}")
print(f"-> Columnas Estables (Interpolación): {columnas_estables}")

segmentos_dinamicos_corregidos = []
segmentos_estables_corregidos = []

# --- 4. CORRECCIÓN VARIABLES DINÁMICAS (Método Estacional) ---
print("\n--- Corrigiendo Atascos en Columnas Dinámicas ---")
for col in columnas_dinamicas:
    num_marcados, lista_segmentos = marcar_segmentos_stuck_como_nan_condicional(df_hibrido, col, serie_predicciones)
    if num_marcados > 0:
        print(f"  -> {col}: {num_marcados} segmentos atascados detectados.")
        for start_idx, end_idx in lista_segmentos:
            corregir_dinamicos_con_estacional_simple(df_hibrido, df_original_eval, col, start_idx, end_idx)
            segmentos_dinamicos_corregidos.append((col, start_idx, end_idx))

# --- 5. CORRECCIÓN VARIABLES ESTABLES (Interpolación) ---
print("\n--- Corrigiendo Atascos en Columnas Estables ---")
for col in columnas_estables:
    num_marcados, lista_segmentos = marcar_segmentos_stuck_como_nan_condicional(df_hibrido, col, serie_predicciones)
    if num_marcados > 0:
        print(f"  -> {col}: {num_marcados} segmentos atascados detectados e interpolados.")
        df_hibrido[col] = df_hibrido[col].interpolate(method='linear', limit_direction='both')
        for start_idx, end_idx in lista_segmentos:
            segmentos_estables_corregidos.append((col, start_idx, end_idx))

# --- 6. IMPUTACIÓN FINAL (Limpieza de NaNs residuales) ---
print("\n--- Ejecutando IterativeImputer Final para NaNs residuales ---")

columnas_objetivo = columnas_dinamicas + columnas_estables

# Seleccionamos las columnas numéricas que se van a imputar
cols_imputer = [col for col in columnas_objetivo if col in df_hibrido.columns]
nans_antes_final = df_hibrido[cols_imputer].isnull().sum().sum()

if nans_antes_final > 0:
    print(f"  -> Detectados {nans_antes_final} NaNs residuales (no resueltos por interpolación o estacionalidad).")
    print("  -> Entrenando y aplicando Imputador Iterativo localmente...")
    
    # Creamos un imputador fresco y lo entrenamos (fit_transform) directamente con estos datos
    imputador_final = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=30, random_state=42, max_depth=10, min_samples_leaf=5),
        max_iter=10,
        random_state=42,
        initial_strategy='median'
    )
    
    imputed_array = imputador_final.fit_transform(df_hibrido[cols_imputer])
    df_imputed_temp = pd.DataFrame(imputed_array, index=df_hibrido.index, columns=cols_imputer)
    
    for col in cols_imputer:
        df_hibrido[col] = df_hibrido[col].fillna(df_imputed_temp[col])
        
    print(f"  -> Imputación final completada. NaNs restantes: {df_hibrido[cols_imputer].isnull().sum().sum()}")
else:
    print("  -> No hay NaNs residuales. Omitiendo paso.")

# --- 7. EVALUACIÓN DE CORRECCIONES (Versión Corregida) ---
def evaluar_correcciones(segmentos, nombre_metodo):
    if not segmentos: 
        print(f" No se detectaron segmentos para {nombre_metodo}. No hay nada que graficar.")
        return
    
    all_true = []
    all_imputed = []
    
    # Usamos una copia para no alterar el original durante la comparación
    df_ref = df_original_eval
    
    for col, start_idx, end_idx in segmentos:
        try:
            # Extraemos los rangos
            true_vals = df_ref.loc[start_idx:end_idx, col]
            imputed_vals = df_hibrido.loc[start_idx:end_idx, col]
            
            # Alineación crítica por índice
            common_idx = true_vals.index.intersection(imputed_vals.index)
            if common_idx.empty: continue

            t_v = true_vals.loc[common_idx]
            i_v = imputed_vals.loc[common_idx]
            
            # Solo comparamos donde ambos tengan valores numéricos (evitar NaNs en el original)
            mask = t_v.notna() & i_v.notna()
            
            all_true.extend(t_v[mask].values)
            all_imputed.extend(i_v[mask].values)
        except Exception as e:
            print(f" Error evaluando segmento {col} ({start_idx}): {e}")

    # --- AQUÍ ESTÁ EL CAMBIO DE LÓGICA ---
    if len(all_true) > 0:
        # 1. Calculamos métricas primero
        rmse = np.sqrt(mean_squared_error(all_true, all_imputed))
        mae = mean_absolute_error(all_true, all_imputed)
        
        print(f"\nResultados Globales - {nombre_metodo}:")
        print(f"  -> RMSE: {rmse:.3f} | MAE: {mae:.3f} (sobre {len(all_true)} puntos)")

        # 2. Generamos la gráfica (DENTRO del bloque if len > 0)
        plt.figure(figsize=(6, 6))
        plt.scatter(all_true, all_imputed, alpha=0.5, s=15, 
                    c='green' if 'Interpolación' in nombre_metodo else 'orange')
        
        min_val = min(min(all_true), min(all_imputed))
        max_val = max(max(all_true), max(all_imputed))
        
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal')
        plt.title(f"Reales vs Corregidos ({nombre_metodo})")
        plt.xlabel("Valores Reales")
        plt.ylabel("Valores Corregidos")
        plt.legend()
        plt.grid(True)
        plt.show() # Ahora sí mostrará la gráfica con datos
    else:
        print(f"⚠️ No se encontraron puntos coincidentes o válidos para comparar en {nombre_metodo}.")

# Llamada a las funciones
evaluar_correcciones(segmentos_dinamicos_corregidos, "Método Estacional (Dinámicos)")
evaluar_correcciones(segmentos_estables_corregidos, "Interpolación (Estables)")

# Guardar y resetear índice
df_hibrido = df_hibrido.reset_index() 
datasets_listos[nombre_objetivo] = df_hibrido

In [ ]:
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'RUIDO' (ANOMALÍAS PUNTUALES) ===")

# --- 1. PREPARACIÓN CON REDONDEO (Indispensable para que haya gráficas) ---
nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'
df_ruido = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeamos al minuto para asegurar que df_ruido y df_original coincidan exactamente
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_ruido = preparar_y_redondear(df_ruido)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. LÓGICA DE CORRECCIÓN ---
mask_ruido = df_ruido['etiqueta_tipo_anomalia'] == "Ruido"
all_true_ruido, all_imputed_ruido = [], []
puntos_corregidos_resumen = {}

if not mask_ruido.any():
    print(f"\nNo se detectaron anomalías de tipo 'Ruido'.")
else:
    # Iteramos solo por las columnas que existen en el dataset
    cols_a_procesar = [c for c in columnas_objetivo if c in df_ruido.columns]
    
    for col in cols_a_procesar:
        # Calculamos el promedio de los vecinos (anterior y posterior)
        val_prev = df_ruido[col].shift(1)
        val_next = df_ruido[col].shift(-1)
        promedio_vecinos = (val_prev + val_next) / 2
        
        # Identificamos los puntos a corregir en esta columna
        mask_a_corregir = mask_ruido & df_ruido[col].notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_ruido.index[mask_a_corregir]
            
            # --- EVALUACIÓN CONTRA ORIGINAL ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            if not idx_comun.empty:
                t_vals = df_original_eval.loc[idx_comun, col]
                i_vals = promedio_vecinos.loc[idx_comun]
                # Solo agregamos si ninguno es NaN para no romper el RMSE
                mask_valid = t_vals.notna() & i_vals.notna()
                all_true_ruido.extend(t_vals[mask_valid].tolist())
                all_imputed_ruido.extend(i_vals[mask_valid].tolist())
            
            # --- APLICAR LA CORRECCIÓN AL DATASET ---
            df_ruido.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_corregidos_resumen[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y GRÁFICA FINAL ---
    print("\nResumen de Correcciones:")
    for col, count in puntos_corregidos_resumen.items():
        print(f"   - {col}: {count} ruidos corregidos.")

    if all_true_ruido:
        rmse_ruido = np.sqrt(mean_squared_error(all_true_ruido, all_imputed_ruido))
        mae_ruido = mean_absolute_error(all_true_ruido, all_imputed_ruido)
        print(f"\n✅ Evaluación exitosa sobre {len(all_true_ruido)} puntos.")
        print(f"RMSE Global Ruido: {rmse_ruido:.3f} | MAE: {mae_ruido:.3f}")

        plt.figure(figsize=(6, 6))
        plt.scatter(all_true_ruido, all_imputed_ruido, alpha=0.6, color='red', s=30, label='Valores Corregidos')
        
        # Línea de identidad (Ideal)
        mn = min(min(all_true_ruido), min(all_imputed_ruido))
        mx = max(max(all_true_ruido), max(all_imputed_ruido))
        plt.plot([mn, mx], [mn, mx], 'k--', lw=2, label="Ideal")
        
        plt.title("Evaluación Modelo 3: Corrección de Ruido")
        plt.xlabel("Valor Real (Dataset Original)")
        plt.ylabel("Valor Corregido (Promedio Vecinos)")
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        plt.show()

# Guardar resultados y resetear el índice para mantener el formato original
df_ruido = df_ruido.reset_index()
datasets_listos[nombre_objetivo] = df_ruido

print("\n✅ FASE 3 COMPLETADA. El dataset ha sido purificado de Ruido.")

In [ ]:
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'VALORES FUERA DE RANGO' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Sincronización de índices) ---
nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'
df_oor = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para asegurar coincidencia entre datasets
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_oor = preparar_y_redondear(df_oor)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. CONFIGURACIÓN ---
factor_umbral_diferencia_oor = 2.0
nombre_clase_oor = "Valores Fuera de Rango"
mask_oor_global = df_oor['etiqueta_tipo_anomalia'] == nombre_clase_oor

puntos_oor_corregidos_por_columna = {}
all_true_oor, all_imputed_oor = [], []

if not mask_oor_global.any():
    print(f"\nNo se detectaron anomalías de tipo '{nombre_clase_oor}'.")
elif 'rangos_validos_sensores' not in locals() and 'rangos_validos_sensores' not in globals():
    print("\n❌ Error: El diccionario 'rangos_validos_sensores' no está definido.")
else:
    # Columnas que están en el dataset y en el diccionario de rangos
    columnas_validas = [col for col in rangos_validos_sensores.keys() if col in df_oor.columns]
    
    for col in columnas_validas:
        rango = rangos_validos_sensores[col]
        min_limit, max_limit = rango.get('min'), rango.get('max')
        
        if min_limit is None or max_limit is None: continue
            
        # 1. Valores y vecinos (Series para mantener el índice Datetime)
        val_actual = pd.to_numeric(df_oor[col], errors='coerce')
        val_prev = val_actual.shift(1)
        val_next = val_actual.shift(-1)
        
        # 2. Doble validación: ¿El modelo dijo OOR Y el valor rompe los límites físicos?
        out_of_range_mask = (val_actual < min_limit) | (val_actual > max_limit)
        mask_a_corregir = mask_oor_global & out_of_range_mask & val_actual.notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_oor.index[mask_a_corregir]
            promedio_vecinos = (val_prev + val_next) / 2
            
            # --- EVITAR EL ASSERTION ERROR: Intersección explícita ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            
            if not idx_comun.empty:
                true_vals = df_original_eval.loc[idx_comun, col]
                imputed_vals = promedio_vecinos.loc[idx_comun]
                
                # Solo comparamos si ambos tienen datos
                v_mask = true_vals.notna() & imputed_vals.notna()
                all_true_oor.extend(true_vals[v_mask].tolist())
                all_imputed_oor.extend(imputed_vals[v_mask].tolist())
            
            # --- APLICAR CORRECCIÓN ---
            df_oor.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_oor_corregidos_por_columna[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y GRÁFICA ---
    if puntos_oor_corregidos_por_columna:
        print("\nResumen de Correcciones (OOR):")
        for col, count in puntos_oor_corregidos_por_columna.items():
            print(f"   - {col}: {count} valores corregidos.")
        
        if all_true_oor:
            rmse_oor = np.sqrt(mean_squared_error(all_true_oor, all_imputed_oor))
            print(f"\n✅ Evaluación exitosa sobre {len(all_true_oor)} puntos. RMSE: {rmse_oor:.3f}")
            
            plt.figure(figsize=(6, 6))
            plt.scatter(all_true_oor, all_imputed_oor, alpha=0.5, s=20, c='purple', label='Puntos corregidos')
            mn, mx = min(all_true_oor), max(all_true_oor)
            plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Ideal')
            plt.title("OOR: Real vs Corregido"); plt.xlabel("Real"); plt.ylabel("Corregido")
            plt.legend(); plt.grid(True); plt.show()
    else:
        print("\nNo se realizaron correcciones: los valores estaban dentro de los rangos físicos.")

# Guardar y resetear índice
df_oor = df_oor.reset_index()
datasets_listos[nombre_objetivo] = df_oor
print("\n✅ FASE 4 COMPLETADA. El dataset ha sido purificado de Valores Fuera de Rango.")

In [ ]:
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'DESVIACIÓN DE CORRELACIÓN' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Indispensable para sincronizar) ---
nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'
df_corr_dev = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para asegurar que df_corr_dev y df_original coincidan
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_corr_dev = preparar_y_redondear(df_corr_dev)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. LÓGICA DE CORRECCIÓN ---
factor_umbral_diferencia_corr_dev = 2.0
nombre_clase_corr_dev = "Desviación de Correlación"

# Máscara global según la predicción del modelo
mask_corr_dev = df_corr_dev['etiqueta_tipo_anomalia'] == nombre_clase_corr_dev

puntos_corregidos_por_columna = {}
all_true_corr_dev = []
all_imputed_corr_dev = []

if not mask_corr_dev.any():
    print(f"\nNo se detectaron anomalías de tipo '{nombre_clase_corr_dev}'.")
else:
    print(f"-> Filas marcadas como Desviación: {mask_corr_dev.sum()}")
    
    for col in [c for c in columnas_objetivo if c in df_corr_dev.columns]:
        std_dev_col = df_original_eval[col].std()
        umbral = factor_umbral_diferencia_corr_dev * std_dev_col
        
        # Trabajamos con Series para mantener la alineación de fechas
        val_actual = df_corr_dev[col]
        val_prev = val_actual.shift(1)
        val_next = val_actual.shift(-1)
        
        # La corrección propuesta: promedio de vecinos (suavizado lineal)
        promedio_vecinos = (val_prev + val_next) / 2
        
        # Filtramos donde corregir
        mask_a_corregir = mask_corr_dev & val_actual.notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_corr_dev.index[mask_a_corregir]
            
            # --- EVALUACIÓN VS ORIGINAL (Evita el AssertionError) ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            
            if not idx_comun.empty:
                true_vals = df_original_eval.loc[idx_comun, col]
                imputed_vals = promedio_vecinos.loc[idx_comun]
                
                v_mask = true_vals.notna() & imputed_vals.notna()
                if v_mask.any():
                    all_true_corr_dev.extend(true_vals[v_mask].tolist())
                    all_imputed_corr_dev.extend(imputed_vals[v_mask].tolist())
            
            # --- APLICACIÓN DE LA CORRECCIÓN ---
            df_corr_dev.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_corregidos_por_columna[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y GRÁFICA ---
    if puntos_corregidos_por_columna:
        print("\nResumen de Correcciones:")
        for col, count in puntos_corregidos_por_columna.items():
            print(f"   - {col}: {count} valores corregidos.")
        
        if all_true_corr_dev:
            rmse_corr = np.sqrt(mean_squared_error(all_true_corr_dev, all_imputed_corr_dev))
            print(f"\n--- Evaluación de Calidad ---")
            print(f"RMSE Global: {rmse_corr:.3f} (sobre {len(all_true_corr_dev)} puntos)")
            
            plt.figure(figsize=(6, 6))
            plt.scatter(all_true_corr_dev, all_imputed_corr_dev, alpha=0.5, s=15, c='orange')
            mn, mx = min(all_true_corr_dev), max(all_true_corr_dev)
            plt.plot([mn, mx], [mn, mx], 'k--', lw=2, label='Ideal')
            plt.title("Desviación de Correlación: Real vs Corregido")
            plt.xlabel("Real"); plt.ylabel("Corregido"); plt.grid(True); plt.show()
    else:
        print("\nNo se realizaron correcciones efectivas.")

# Finalizar
df_corr_dev = df_corr_dev.reset_index()
datasets_listos[nombre_objetivo] = df_corr_dev

print("\n✅ FASE 5 COMPLETADA.")

In [ ]:
print("=== INICIANDO MODELO 3: CORRECCIÓN DE 'ANOMALÍAS CONTEXTUALES' ===")

# --- 1. PREPARACIÓN CON REDONDEO (Sincronización Crítica) ---
nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'
df_contextual = datasets_listos[nombre_objetivo].copy()
df_original_eval = df_original.copy()

def preparar_y_redondear(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # Redondeo al minuto para asegurar que df_contextual y df_original coincidan exactamente
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    return df[~df.index.duplicated(keep='first')].sort_index()

df_contextual = preparar_y_redondear(df_contextual)
df_original_eval = preparar_y_redondear(df_original_eval)

# --- 2. LÓGICA DE CORRECCIÓN ---
factor_umbral_diferencia_ctx = 2.0
nombre_clase_ctx = "Contextual"

# Máscara global según la predicción del modelo
mask_ctx_global = df_contextual['etiqueta_tipo_anomalia'] == nombre_clase_ctx

puntos_corregidos_por_columna = {}
all_true_ctx = []
all_imputed_ctx = []

if not mask_ctx_global.any():
    print(f"\nNo se detectaron anomalías de tipo '{nombre_clase_ctx}'.")
else:
    print(f"-> Filas marcadas como Contextuales: {mask_ctx_global.sum()}")
    
    for col in [c for c in columnas_objetivo if c in df_contextual.columns]:
        std_dev_col = df_original_eval[col].std()
        umbral = factor_umbral_diferencia_ctx * std_dev_col
        
        # Trabajamos con Series para mantener la alineación de fechas
        val_actual = df_contextual[col]
        val_prev = val_actual.shift(1)
        val_next = val_actual.shift(-1)
        
        # Corrección propuesta: promedio de vecinos
        promedio_vecinos = (val_prev + val_next) / 2
        
        # Filtramos donde corregir
        mask_a_corregir = mask_ctx_global & val_actual.notna()
        
        if mask_a_corregir.any():
            idx_a_corregir = df_contextual.index[mask_a_corregir]
            
            # --- EVALUACIÓN VS ORIGINAL (Evita el AssertionError) ---
            idx_comun = idx_a_corregir.intersection(df_original_eval.index)
            
            if not idx_comun.empty:
                true_vals = df_original_eval.loc[idx_comun, col]
                imputed_vals = promedio_vecinos.loc[idx_comun]
                
                v_mask = true_vals.notna() & imputed_vals.notna()
                if v_mask.any():
                    all_true_ctx.extend(true_vals[v_mask].tolist())
                    all_imputed_ctx.extend(imputed_vals[v_mask].tolist())
            
            # --- APLICACIÓN DE LA CORRECCIÓN ---
            df_contextual.loc[idx_a_corregir, col] = promedio_vecinos.loc[idx_a_corregir]
            puntos_corregidos_por_columna[col] = len(idx_a_corregir)

    # --- 3. RESUMEN Y EVALUACIÓN ---
    if puntos_corregidos_por_columna:
        print("\nResumen de Correcciones por Columna:")
        for col, count in puntos_corregidos_por_columna.items():
            print(f"   - {col}: {count} valores corregidos.")
        
        if all_true_ctx:
            rmse_ctx = np.sqrt(mean_squared_error(all_true_ctx, all_imputed_ctx))
            mae_ctx = mean_absolute_error(all_true_ctx, all_imputed_ctx)
            
            print(f"\n--- Evaluación de Calidad (Anomalía Contextual) ---")
            print(f"RMSE Global: {rmse_ctx:.3f} | MAE Global: {mae_ctx:.3f} (sobre {len(all_true_ctx)} puntos)")
            
            plt.figure(figsize=(6, 6))
            plt.scatter(all_true_ctx, all_imputed_ctx, alpha=0.5, s=15, c='darkblue')
            mn, mx = min(all_true_ctx), max(all_true_ctx)
            plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Ideal')
            plt.title("Anomalías Contextuales: Real vs Corregido")
            plt.xlabel("Valores Verdaderos (Original)")
            plt.ylabel("Valores Corregidos")
            plt.grid(True, linestyle='--', alpha=0.5); plt.legend(); plt.show()
    else:
        print("\nNo se realizaron correcciones efectivas en esta fase.")

# Devolvemos el DataFrame al diccionario
df_contextual = df_contextual.reset_index()
datasets_listos[nombre_objetivo] = df_contextual

print("\n✅ FASE 6 COMPLETADA. El dataset ha sido purificado de Anomalías Contextuales.")
print("🎉 EL MODELO 3 HA FINALIZADO TODAS SUS FASES EXITOSAMENTE 🎉")

In [ ]:
print("=== INICIANDO PREPARACIÓN DE VISUALIZACIONES ===")

nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'

# --- 1. CARGA SEGURA DE LOS 3 DATAFRAMES ---
# A. Original (Sin daños)
df_plot_orig = df_original.copy()
# B. Inyectado (El que evaluó el Modelo 2, antes de corregir. Se guardó como 'df_trabajo' en la Fase 1)
df_plot_inyectado = df_trabajo.copy() 
# C. Totalmente Corregido (El resultado final de nuestra Fase 6)
df_plot_corregido = datasets_listos[nombre_objetivo].copy()

# Función auxiliar para garantizar que el índice sea de Tiempo (Datetime) y esté limpio
def preparar_indice_tiempo(df, nombre):
    if 'Fecha' in df.columns and not isinstance(df.index, pd.DatetimeIndex):
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
        df.set_index('Fecha', inplace=True)
    if isinstance(df.index, pd.DatetimeIndex):
        df = df[~df.index.duplicated(keep='first')].sort_index()
    else:
        print(f"⚠️ Advertencia: {nombre} no pudo convertirse a DatetimeIndex.")
    return df

df_plot_orig = preparar_indice_tiempo(df_plot_orig, "df_original")
df_plot_inyectado = preparar_indice_tiempo(df_plot_inyectado, "df_trabajo (Inyectado)")
df_plot_corregido = preparar_indice_tiempo(df_plot_corregido, "df_corregido")

# --- 2. DEFINICIÓN DE COLUMNAS A VISUALIZAR ---
columnas_objetivo = ['PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV', 
                     'XCO2I', 'XHINV', 'XTINV', 'XTS', 'UVENT_cen', 'UVENT_lN']

columnas_a_graficar = [col for col in columnas_objetivo if col in df_plot_orig.columns and col in df_plot_corregido.columns]

# --- 3. GENERACIÓN DE GRÁFICAS ---
for columna in columnas_a_graficar:
    print(f"\n--- Generando gráficas para: {columna} ---")
    
    # Preparamos los puntos rojos (Anomalías detectadas por el Modelo 2)
    puntos_anomalos_x = []
    puntos_anomalos_y = []
    if 'etiqueta_tipo_anomalia' in df_plot_inyectado.columns:
        # Filtramos solo las filas que NO son nulas en la etiqueta de anomalía
        mask_anomalias = df_plot_inyectado['etiqueta_tipo_anomalia'].notna()
        puntos_anomalos_x = df_plot_inyectado[mask_anomalias].index
        puntos_anomalos_y = df_plot_inyectado.loc[mask_anomalias, columna]

    # --- FIGURA 1: Original vs Inyectado ---
    fig1, axes1 = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    fig1.suptitle(f"1. IMPACTO DE ANOMALÍAS - Variable: {columna}", fontsize=16, fontweight='bold')
    
    axes1[0].plot(df_plot_orig.index, df_plot_orig[columna], label="Datos Originales (Sanos)", color="blue", alpha=0.8)
    axes1[0].set_title("Comportamiento Original")
    axes1[0].set_ylabel(columna)
    axes1[0].legend(loc='upper left')
    axes1[0].grid(True, linestyle='--', alpha=0.5)
    
    axes1[1].plot(df_plot_inyectado.index, df_plot_inyectado[columna], label="Datos Dañados", color="orange", alpha=0.8)
    if len(puntos_anomalos_x) > 0:
        axes1[1].scatter(puntos_anomalos_x, puntos_anomalos_y, color='red', label='Anomalía Detectada (Modelo 2)', s=40, edgecolor='black', zorder=3)
    axes1[1].set_title("Comportamiento con Anomalías Inyectadas")
    axes1[1].set_ylabel(columna)
    axes1[1].legend(loc='upper left')
    axes1[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

    # --- FIGURA 2: Original vs Totalmente Corregido ---
    fig2, axes2 = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    fig2.suptitle(f"2. EFECTIVIDAD DE CORRECCIÓN - Variable: {columna}", fontsize=16, fontweight='bold')
    
    axes2[0].plot(df_plot_orig.index, df_plot_orig[columna], label="Datos Originales (Sanos)", color="blue", alpha=0.8)
    axes2[0].set_title("Comportamiento Original")
    axes2[0].set_ylabel(columna)
    axes2[0].legend(loc='upper left')
    axes2[0].grid(True, linestyle='--', alpha=0.5)
    
    axes2[1].plot(df_plot_corregido.index, df_plot_corregido[columna], label="Datos Corregidos (Modelo 3)", color="green", alpha=0.8)
    axes2[1].set_title("Comportamiento Reconstruido")
    axes2[1].set_ylabel(columna)
    axes2[1].legend(loc='upper left')
    axes2[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

    # --- FIGURA 3: Inyectado vs Totalmente Corregido ---
    fig3, axes3 = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    fig3.suptitle(f"3. PROCESO DE LIMPIEZA - Variable: {columna}", fontsize=16, fontweight='bold')
    
    axes3[0].plot(df_plot_inyectado.index, df_plot_inyectado[columna], label="Datos Dañados", color="orange", alpha=0.8)
    if len(puntos_anomalos_x) > 0:
        axes3[0].scatter(puntos_anomalos_x, puntos_anomalos_y, color='red', label='Anomalía Detectada', s=40, edgecolor='black', zorder=3)
    axes3[0].set_title("Antes de la Corrección")
    axes3[0].set_ylabel(columna)
    axes3[0].legend(loc='upper left')
    axes3[0].grid(True, linestyle='--', alpha=0.5)
    
    axes3[1].plot(df_plot_corregido.index, df_plot_corregido[columna], label="Datos Corregidos (Modelo 3)", color="green", alpha=0.8)
    axes3[1].set_title("Después de la Corrección")
    axes3[1].set_xlabel("Fecha")
    axes3[1].set_ylabel(columna)
    axes3[1].legend(loc='upper left')
    axes3[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

print("\n=== FIN DE LA VISUALIZACIÓN ===")

In [ ]:
print("=== INICIANDO ANÁLISIS NUMÉRICO DE DIFERENCIAS ===")

nombre_objetivo = 'AgroConnect_OPC_20250301_20250430'

# --- 1. CARGA Y PREPARACIÓN CON REDONDEO (Sincronización Crítica) ---
df_orig = df_original.copy()
df_inyectado = df_trabajo.copy()
df_corregido = datasets_listos[nombre_objetivo].copy()

def preparar_y_sincronizar(df):
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True, errors='coerce')
        df = df.set_index('Fecha')
    # REDONDEO AL MINUTO: Vital para que la intersección no devuelva 0 filas
    df.index = pd.to_datetime(df.index, errors='coerce').round('1min')
    # Limpieza de duplicados y ordenamiento
    return df[~df.index.duplicated(keep='first')].sort_index()

df_orig = preparar_y_sincronizar(df_orig)
df_inyectado = preparar_y_sincronizar(df_inyectado)
df_corregido = preparar_y_sincronizar(df_corregido)

# Columnas a evaluar
columnas_validas = [col for col in columnas_objetivo if col in df_orig.columns and col in df_corregido.columns]

resumen_diferencias = []

# --- 2. ANÁLISIS COLUMNA POR COLUMNA ---
for columna in columnas_validas:
    # Intersección de los 3 índices para comparación triple exacta
    idx_comun = df_orig.index.intersection(df_inyectado.index).intersection(df_corregido.index)
    
    if idx_comun.empty:
        print(f"⚠️ Advertencia: No hay coincidencia de fechas para {columna}.")
        continue

    s_orig = df_orig.loc[idx_comun, columna]
    s_inyect = df_inyectado.loc[idx_comun, columna]
    s_corr = df_corregido.loc[idx_comun, columna]
    
    # A. Conteo de celdas modificadas (ignora si ambos son NaN)
    # Usamos np.isclose o una tolerancia pequeña si son floats, o != si son precisos
    mask_daño = (s_orig != s_inyect) & ~(s_orig.isna() & s_inyect.isna())
    mask_residuo = (s_orig != s_corr) & ~(s_orig.isna() & s_corr.isna())
    
    diff_inyectadas = mask_daño.sum()
    diff_restantes = mask_residuo.sum()
    
    # B. Métricas Globales (Solo sobre puntos con datos en ambos)
    mask_valid = s_orig.notna() & s_corr.notna()
    if mask_valid.any():
        s_o_v = s_orig[mask_valid]
        s_c_v = s_corr[mask_valid]
        
        std_col = s_o_v.std()
        rmse_overall = np.sqrt(mean_squared_error(s_o_v, s_c_v))
        mae_overall = mean_absolute_error(s_o_v, s_c_v)
        
        # C. Errores Significativos (> 2 y 3 Desviaciones Estándar)
        abs_diff = (s_o_v - s_c_v).abs()
        
        # Umbral 2 STD
        mask_2std = abs_diff > (2 * std_col)
        c_2std = mask_2std.sum()
        r_2std = np.sqrt(mean_squared_error(s_o_v[mask_2std], s_c_v[mask_2std])) if c_2std > 0 else 0
        
        # Umbral 3 STD
        mask_3std = abs_diff > (3 * std_col)
        c_3std = mask_3std.sum()
        r_3std = np.sqrt(mean_squared_error(s_o_v[mask_3std], s_c_v[mask_3std])) if c_3std > 0 else 0
        
        resumen_diferencias.append({
            'Columna': columna,
            'Daños_Inyectados': diff_inyectadas,
            'Diferencias_Restantes': diff_restantes,
            'Puntos_Eval': len(s_o_v),
            'RMSE_General': round(rmse_overall, 4),
            'MAE_General': round(mae_overall, 4),
            'Err_>2Std_(Cant)': c_2std,
            'RMSE_>2Std': round(r_2std, 4),
            'Err_>3Std_(Cant)': c_3std,
            'RMSE_>3Std': round(r_3std, 4)
        })

# --- 3. TABLA DE RESUMEN FINAL ---
if resumen_diferencias:
    df_resumen = pd.DataFrame(resumen_diferencias)
    print("\n" + "="*80)
    print("                 RESUMEN FINAL DE CALIDAD DE CORRECCIÓN")
    print("="*80)
    print(df_resumen.to_string(index=False))
    df_metricas_finales = df_resumen 
else:
    print("\n❌ Error: No se pudieron sincronizar los datos para el análisis.")

print("\n=== FIN DEL ANÁLISIS NUMÉRICO ===")

In [ ]:
print("=== INICIANDO VISUALIZACIÓN DEL ANÁLISIS NUMÉRICO ===")

# Verificamos que el DataFrame del paso anterior exista
if 'df_metricas_finales' in locals() or 'df_metricas_finales' in globals():
    df_resumen = df_metricas_finales.copy()
    print("-> Tabla de métricas encontrada. Generando gráficas...\n")

    # Configuración visual global
    plt.style.use('default')

    # --- 1. Progreso de Limpieza: Daños vs Restantes ---
    # Ajustamos a los nombres de columna del bloque anterior: 'Daños_Inyectados' y 'Diferencias_Restantes'
    df_plot_diff = df_resumen.set_index('Columna')[['Daños_Inyectados', 'Diferencias_Restantes']]
    df_plot_diff = df_plot_diff.sort_values(by='Diferencias_Restantes', ascending=False)
    
    ax1 = df_plot_diff.plot(kind='bar', figsize=(15, 6), width=0.8, color=['#FFA07A', '#20B2AA'])
    ax1.set_title('1. Progreso de Limpieza: Daños Inyectados vs Diferencias Restantes', fontsize=16, fontweight='bold')
    ax1.set_xlabel('Sensor', fontsize=12)
    ax1.set_ylabel('Cantidad de Celdas', fontsize=12)
    ax1.legend(['Daños Iniciales (Inyectados)', 'Errores Residuales (Post-Corrección)'])
    ax1.tick_params(axis='x', rotation=45)
    # Formateador de miles para legibilidad
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # --- 2. Métricas Generales (RMSE y MAE) ---
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # RMSE General
    df_rmse = df_resumen.set_index('Columna')[['RMSE_General']].sort_values(by='RMSE_General', ascending=False)
    df_rmse.plot(kind='bar', ax=axes[0], color='skyblue', legend=False)
    axes[0].set_title('2A. Error Cuadrático Medio (RMSE) General', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('RMSE', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    
    # MAE General
    df_mae = df_resumen.set_index('Columna')[['MAE_General']].sort_values(by='MAE_General', ascending=False)
    df_mae.plot(kind='bar', ax=axes[1], color='lightcoral', legend=False)
    axes[1].set_title('2B. Error Absoluto Medio (MAE) General', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('MAE', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

    # --- 3. Cantidad de Errores Graves ---
    # Ajustado a: 'Err_>2Std_(Cant)' y 'Err_>3Std_(Cant)'
    df_plot_sig = df_resumen.set_index('Columna')[['Err_>2Std_(Cant)', 'Err_>3Std_(Cant)']]
    df_plot_sig = df_plot_sig.sort_values(by='Err_>2Std_(Cant)', ascending=False)

    ax3 = df_plot_sig.plot(kind='bar', figsize=(15, 6), width=0.8, color=['#FFD700', '#DC143C'])
    ax3.set_title('3. Frecuencia de Errores Graves (>2 y >3 Desviaciones Estándar)', fontsize=16, fontweight='bold')
    ax3.set_xlabel('Sensor', fontsize=12)
    ax3.set_ylabel('Cantidad de Puntos', fontsize=12)
    ax3.legend(['Error > 2 Std Dev', 'Error > 3 Std Dev'])
    ax3.tick_params(axis='x', rotation=45)
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # --- 4. RMSE de Errores Graves ---
    # Ajustado a: 'RMSE_>2Std' y 'RMSE_>3Std'
    df_plot_rmse_sig = df_resumen.set_index('Columna')[['RMSE_>2Std', 'RMSE_>3Std']]
    df_plot_rmse_sig = df_plot_rmse_sig.sort_values(by='RMSE_>2Std', ascending=False)
    
    ax4 = df_plot_rmse_sig.plot(kind='bar', figsize=(15, 6), width=0.8, color=['#DAA520', '#8B0000'])
    ax4.set_title('4. Magnitud del Error (RMSE) en Puntos Críticos', fontsize=16, fontweight='bold')
    ax4.set_xlabel('Sensor', fontsize=12)
    ax4.set_ylabel('RMSE', fontsize=12)
    ax4.legend(['RMSE en Errores > 2 Std Dev', 'RMSE en Errores > 3 Std Dev'])
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    print("\n=== FIN DE LA VISUALIZACIÓN DEL ANÁLISIS NUMÉRICO ===")

else:
    print("❌ Error: No se encontró la tabla 'df_metricas_finales'.")